In [ ]:
import pandas as pd
import os
import numpy as np
import seaborn as sns
import re
import matplotlib.pyplot as plt
import requests
import time
import json
import anndata as ad
from pathlib import Path
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests

proteomics_folder = '../../data/review_screen/proteomics'
output_folder = '../../outputs/followup_screen/output_tables/'
os.makedirs(output_folder, exist_ok=True)

# local imports
from utils.unicode_utils import apply_greek_labels_heatmap, introduce_greek_letters, replace_greek_letters, apply_greek_labels
from utils.helper_functions import get_common_index, leiden_clustering, calculate_cell_fractions
from utils.plotting_utils import cm_to_inches, setup_plotting_environment, plot_overview_labels_along_y_axis, create_sankey_diagram_strength
from utils.plotting_utils import annotate_pvalues
setup_plotting_environment()

# ------- CRITICAL -------
alt_flag = False # use an alternative cell type set (more fine-grained)
write_SI_tables_flag = True # whether to write tables with SI values (if False, only plots will be generated)
# ------- CRITICAL -------
if not alt_flag:
    from utils.color_constants import CELL_TYPE_COLORS, ALL_CELL_TYPES
else:
    from utils.color_constants import CELL_TYPE_COLORS_ALT, ALL_CELL_TYPES_ALT
    ALL_CELL_TYPES = ALL_CELL_TYPES_ALT
    CELL_TYPE_COLORS = CELL_TYPE_COLORS_ALT

plot_folder = '../../outputs/followup_screen/plots'
os.makedirs(plot_folder, exist_ok=True)

In [ ]:
def split_and_strip(s, val=';'):
    s = s.split(val)
    s = [sub.strip() for sub in s]
    return s

def format_treatment(treat, connection="&"):
    treat = treat.replace("_", connection)
    treat = treat.replace("AB", 'alpha')
    treat = introduce_greek_letters(treat)
    return treat

def robust_z_score(df):
    median = df.median(axis=1)
    mad = (df.sub(median, axis=0)).abs().median(axis=1)
    robust_z = df.sub(median, axis=0).div(1.4826 * mad.replace(0, np.nan), axis=0)
    return robust_z

# 0 - Load data

In [ ]:
# info on cytokines
cytokine_info = pd.read_excel("../../annotations/cytokine_info.xlsx", index_col=0, sheet_name="all_cytokines")
gene_to_cytokine = cytokine_info.set_index("gene")['gene_to_cytokine'].dropna().to_dict()
cytokine_to_gene = {v: k for k, v in gene_to_cytokine.items()}
ALL_CYTOKINES_ORDER = cytokine_info['gene_to_cytokine'].dropna().tolist()

cytokine_name_to_level_1_family = cytokine_info.reset_index().set_index("name")["family_level_1"].to_dict()
cytokine_name_to_level_2_family = cytokine_info.reset_index().set_index("name")["family_level_2"].to_dict()
cytokine_name_to_functional = cytokine_info.reset_index().set_index("name")["functional_grouping"].to_dict()

# gene-level names (e.g. IL-32, C3, C5, LT-alpha-beta) are aliases of cytokine_info names
gene_level_alias = {alias: name for name, alias in cytokine_info["gene_to_cytokine"].dropna().items() if alias != name}
for name_map in [cytokine_name_to_level_1_family, cytokine_name_to_level_2_family, cytokine_name_to_functional]:
    name_map.update({alias: name_map[name] for alias, name in gene_level_alias.items()})

cytokine_level_1_order = list(dict.fromkeys(cytokine_name_to_level_1_family.values()))
cytokine_level_2_order = list(dict.fromkeys(cytokine_name_to_level_2_family.values()))
level2_to_level1 = cytokine_info.set_index("family_level_2")["family_level_1"].to_dict()

# tested subset
cytokine_info_parse = cytokine_info[cytokine_info["parse_cytokine"]]

# ordering
cytokine_info_order = pd.read_excel("../../annotations/cytokine_info.xlsx", index_col=0, sheet_name="suggested_order")
CYTOKINE_ORDER = list(cytokine_info_order["name"])
CYTOKINE_ORDER_WITH_PBS = ["PBS"] + CYTOKINE_ORDER
SINGLE_FAMILY_ORDER = cytokine_info_order["DEG_plot_family"].unique()

assert cytokine_info['uniprot'].str.match(r'^[A-Z0-9]+(;[A-Z0-9]+)*$').all()
cytokine_info['uniprot'] = cytokine_info['uniprot'].apply(lambda x:';'.join(sorted([s.strip() for s in x.split(';')])))
uniprot_id_to_cytokine = cytokine_info.reset_index().set_index('uniprot')['name'].to_dict()

In [ ]:
donor_metadata = pd.read_excel("../../annotations/donor_metadata.xlsx")
donor_metadata["Donor number"] = donor_metadata["Donor number"].str.replace(" ", "")
donor_metadata.index = donor_metadata["Donor number"]
donor_metadata["age"] = 2024 - donor_metadata["Birth year"]
donor_metadata = donor_metadata.sort_values(by="Sex")
DONOR_NAMES = donor_metadata.index.tolist()
DONOR_NAMES = [d.replace("Donor", "D") for d in DONOR_NAMES]

DONOR_GROUPS = {
    'interferon': ["D1", "D3", "D4", "D10"],
    'non-interferon': ["D5", "D7", "D8", "D9", "D11", "D12"],
    'other': ['D6', 'D2'] # intermediate/variable, IL-32-like condition
}
DONOR_TO_GROUP = {d: group for group, donors in DONOR_GROUPS.items() for d in donors}
DONOR_COLOR_SCHEME = {
    "interferon": "royalblue",
    "non-interferon": "forestgreen",
    "other": "tab:gray"
}
DONOR_GROUPS_DEMOGRAPHIC = {
    "D1": "older\nfemale",
    "D3": "older\nfemale", 
    "D4": "older\nfemale",
    "D7": "younger\nfemale",
    "D8": "younger\nfemale",
    "D12": "younger\nfemale", 
    "D2": "younger\nmale",
    "D5": "younger\nmale",
    "D6": "younger\nmale", 
    "D9": "younger\nmale",
    "D10": "younger\nmale",
    "D11": "younger\nmale"
}
DONOR_ORDER = list(DONOR_GROUPS_DEMOGRAPHIC.keys())
DONOR_ORDER_IFN_GROUPED = ['D1', 'D3', 'D4', 'D10', 'D5', 'D7', 'D8', 'D9', 'D11', 'D12', 'D6', 'D2']
DONOR_COLOR_DIRECT = {
    donor: DONOR_COLOR_SCHEME['interferon'] if donor in DONOR_GROUPS['interferon'] else \
        DONOR_COLOR_SCHEME['non-interferon'] if donor in DONOR_GROUPS['non-interferon'] else \
        DONOR_COLOR_SCHEME['other'] for donor in DONOR_ORDER_IFN_GROUPED
}
DONOR_COLOR_SCHEME_D2 = {d: 'black' for d in DONOR_NAMES}
DONOR_COLOR_SCHEME_D2["D2"] = 'tab:red'
BAR_COLOR_STANDARD = [DONOR_COLOR_DIRECT[donor] for donor in DONOR_ORDER_IFN_GROUPED]

AB_CONDITIONS = ['AB-IFN-gamma', 'AB-TNF-alpha', 'AB-IL-10', 'AB-IL-6', 'IL-1Ra']

In [ ]:
from utils.cell_type_mapping import get_relevant_cell_types
relevant_cell_types, relevant_cell_type_conversion, relevant_cell_to_pseudobulk = get_relevant_cell_types(alt_flag)

In [ ]:
pseudobulk_review = ad.read_h5ad("../../data/review_screen/scrna/pseudobulk_review_screen.h5ad")
pseudobulk_review = pseudobulk_review[pseudobulk_review.obs['cell_type'].isin(ALL_CELL_TYPES)]

In [ ]:
count_df_raw = pseudobulk_review.layers['sum'].T
index = pseudobulk_review.var.index
cols = pd.MultiIndex.from_tuples((row['cell_type'], row['treatment'], row['donor']) for _, row in pseudobulk_review.obs.iterrows())
count_df_raw = pd.DataFrame(count_df_raw, columns=cols, index=index)
count_df_raw.columns.names = ['cell_type', 'cytokine', 'donor']

gene_map = {
    "TDGF1": "CRIPTO", # fix one gene mapping
}
count_df_raw.rename(index=gene_map, inplace=True)

In [ ]:
ALL_TREATMENTS = list(count_df_raw.columns.get_level_values(1).unique())

In [ ]:
from utils.processing_utils import get_protein_coding_genes, get_sex_gene_list
protein_coding_genes = get_protein_coding_genes("../../annotations/", exclude_ribosomal=True)
sex_gene_list = get_sex_gene_list("../../data/main_screen/")

In [ ]:
cell_count_review = pseudobulk_review.obs["cell_count"].values
index = pd.MultiIndex.from_tuples((row['cell_type'], row['treatment'], row['donor']) for _, row in pseudobulk_review.obs.iterrows())
cell_count_review = pd.Series(cell_count_review, index=index)
cell_count_review.index.names = ['cell_type', 'cytokine', 'donor']

In [ ]:
main_pbs_counts = {}
for cell_type in relevant_cell_types:
    path = f"../../data/main_screen/pseudobulks/pbulk_{relevant_cell_to_pseudobulk[cell_type]}_PBS.h5ad"
    pb = ad.read_h5ad(path, backed="r")
    counts = pb.obs["psbulk_n_cells"].copy()
    counts.index = counts.index.str.split("_").str[0].str.replace("Donor", "D", regex=False)
    main_pbs_counts[cell_type] = counts
    pb.file.close()
main_pbs_counts = pd.DataFrame(main_pbs_counts)  # donors x cell types

In [ ]:
review_pbs_counts = cell_count_review.xs("PBS", level="cytokine").unstack("cell_type")
review_t0_counts = cell_count_review.xs("TimePoint-0", level="cytokine").unstack("cell_type")
cells = main_pbs_counts.columns.intersection(review_pbs_counts.columns)

main = main_pbs_counts[cells]
review = review_pbs_counts[cells]
main = main.div(main.sum(axis=1), axis=0)
review = review.div(review.sum(axis=1), axis=0)

fig, axes = plt.subplots(int(np.ceil(len(cells) / 4)), 4, figsize=cm_to_inches((10, 7)))
axes = axes.flatten()
for cell_i, cell in enumerate(cells):
    ax = axes[cell_i]
    paired = pd.concat([main[cell], review[cell]], axis=1).dropna()
    x, y = paired.iloc[:, 0] * 100, paired.iloc[:, 1] * 100
    r = x.corr(y)
    ax.scatter(x, y, s=5)
    limit = max(x.max(), y.max()) * 1.1
    ax.plot([0, limit], [0, limit], "--", color="grey", lw=0.7)
    ax.set_xlim(0, limit)
    ax.set_ylim(0, limit)
    ax.set_title(f"{cell}: r={r:.2f}", fontsize=6)
    if cell_i % 4 == 0:
        ax.set_ylabel("Follow-up PBS (%)")
    if cell_i > 7:
        ax.set_xlabel("Main PBS (%)")
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
review_pbs_counts = cell_count_review.xs("PBS", level="cytokine").unstack("cell_type")
review_t0_counts = cell_count_review.xs("TimePoint-0", level="cytokine").unstack("cell_type")
cells = review_t0_counts.columns.intersection(review_pbs_counts.columns)

pbs = review_pbs_counts[cells]
t0 = review_t0_counts[cells]
pbs = pbs.div(pbs.sum(axis=1), axis=0)
t0 = t0.div(t0.sum(axis=1), axis=0)

fig, axes = plt.subplots(int(np.ceil(len(cells) / 4)), 4, figsize=cm_to_inches((10, 7)))
axes = axes.flatten()
for cell_i, cell in enumerate(cells):
    ax = axes[cell_i]
    paired = pd.concat([pbs[cell], t0[cell]], axis=1).dropna()
    x, y = paired.iloc[:, 0] * 100, paired.iloc[:, 1] * 100
    r = x.corr(y)
    ax.scatter(x, y, s=5)
    limit = max(x.max(), y.max()) * 1.1
    ax.plot([0, limit], [0, limit], "--", color="grey", lw=0.7)
    ax.set_xlim(0, limit)
    ax.set_ylim(0, limit)
    ax.set_title(f"{cell}: r={r:.2f}", fontsize=6)
    if cell_i % 4 == 0:
        ax.set_ylabel("Follow-up t0 (%)")
    if cell_i > 7:
        ax.set_xlabel("Follow-up PBS (%)")
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

### aggregation

In [ ]:
exclude_cell_types = ['HSPC']
aggregate_attribution = {
    "B cell": "B cell",
    "CD4 T cell": "T cell",
    "CD8 T cell": "T cell",
    "Treg": "T cell",
    "NKT": "T cell",
    "MAIT": "T cell",
    "NK CD56low": "NK cell",
    "NK CD56hi": "NK cell",
    "CD14 Mono": "Mono/cDC",
    "CD16 Mono": "Mono/cDC",
    "cDC": "Mono/cDC",
}

# create aggregate cell counts
cell_count_agg = cell_count_review[~cell_count_review.index.get_level_values(0).isin(exclude_cell_types)].copy()
cell_count_agg = cell_count_agg.rename(index=aggregate_attribution, level=0)
cell_count_agg = cell_count_agg.groupby(level=[0, 1, 2]).sum()

In [ ]:
from utils.processing_utils import get_filter_by_counts, normalize_df_to_rpm

count_df_raw_agg = count_df_raw.loc[:, ~count_df_raw.columns.get_level_values("cell_type").isin(exclude_cell_types)].copy()
count_df_raw_agg.columns = pd.MultiIndex.from_tuples([(aggregate_attribution[i[0]], i[1], i[2]) for i in count_df_raw_agg.columns])
count_df_raw_agg = count_df_raw_agg.fillna(0).T.groupby(level=[0, 1, 2]).sum().T
count_df_raw_agg.columns.names = ['cell_type', 'cytokine', 'donor']

# normalize to rpm
mask_average_calc = get_filter_by_counts(cell_count_agg, plot=False)
mask_average_calc = mask_average_calc[mask_average_calc.index.isin(count_df_raw_agg.columns)]
count_df_raw_agg = count_df_raw_agg.astype(float)
count_df_raw_agg.loc[:, mask_average_calc] = np.nan
count_df_norm_agg = normalize_df_to_rpm(count_df_raw_agg, along_axis=0, pseudocount_initial=0, pseudocount_after=0.5)

In [ ]:
SHARED_DONORS = cell_count_agg['B cell']['AB-IL-10'].index
counts_per_cell = count_df_raw_agg.sum(axis=0) / cell_count_agg
counts_per_cell = counts_per_cell.loc[(slice(None), slice(None), SHARED_DONORS)]
counts_per_cell = counts_per_cell.reset_index()
counts_per_cell = counts_per_cell.rename({0: "counts"}, axis=1)
counts_per_cell['counts'] = np.log10(counts_per_cell['counts'])

mean_UMIs_per_cell = 10**counts_per_cell.groupby("cell_type")['counts'].mean()
mean_UMIs_per_cell = mean_UMIs_per_cell/mean_UMIs_per_cell.median()

In [ ]:
from utils.plotting_utils import test_pairs, annotate_pair_results

plt.figure(figsize=cm_to_inches((7, 4)))
order = ['B cell', 'Mono/cDC', 'NK cell', 'T cell']

sns.swarmplot(data=counts_per_cell, x="cell_type", y="counts", s=2, color='black', order=order, rasterized=True)
plt.xlabel('')
plt.ylabel('log10(mean UMIs per cell)')

pairs = [
    ("B cell", "Mono/cDC"),
    ("B cell", "NK cell"),
    ("Mono/cDC", "NK cell"),
    ("Mono/cDC", "T cell"),
    ("NK cell", "T cell")
]

test_results = test_pairs(counts_per_cell, x='cell_type', y='counts', pairs=pairs, alternative='two-sided')
annotate_pair_results(plt.gca(), results=test_results, order=order, loc='outside', margin=0.1, gap=0.1)

plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "1_UMIs_per_cell.svg"), dpi=400)
plt.show()

### /aggregation

In [ ]:
# get filter and plot
mask_average_calc = get_filter_by_counts(cell_count_review, cell_type_colors=CELL_TYPE_COLORS)

# subset to only conditions present in the data (e.g. some cytokines are missing for some cell types)
mask_average_calc = mask_average_calc[mask_average_calc.index.isin(count_df_raw.columns)]

# filter these out by setting to nan (these will be ignored in the average calculation)
count_df_raw = count_df_raw.astype(float)
count_df_raw.loc[:, mask_average_calc] = np.nan

# normalize to rpm
count_df_norm = normalize_df_to_rpm(count_df_raw, along_axis=0)

In [ ]:
# get mean values
count_df_raw_mean = count_df_raw.T.groupby(level=[0, 1]).mean().T.loc[:, ALL_CELL_TYPES]
count_df_norm_mean = count_df_norm.T.groupby(level=[0, 1]).mean().T.loc[:, ALL_CELL_TYPES]

In [ ]:
from utils.processing_utils import get_raw_log2fc_values

raw_log2fc_values_by_donor, raw_log2fc_values, raw_log2fc_values_std = get_raw_log2fc_values(
    normalize_df_to_rpm(count_df_raw, along_axis=0),
    count_df_raw,
    require_both_counts=False
)
raw_log2fc_values_by_donor = raw_log2fc_values_by_donor.loc[:, raw_log2fc_values_by_donor.columns.get_level_values('cytokine') != 'TimePoint-0']
raw_log2fc_values = raw_log2fc_values.loc[:, raw_log2fc_values.columns.get_level_values('cytokine') != 'TimePoint-0']
raw_log2fc_values_std = raw_log2fc_values_std.loc[:, raw_log2fc_values_std.columns.get_level_values('cytokine') != 'TimePoint-0']

In [ ]:
edger_results = pd.read_parquet('../../data/review_screen/scrna/edgeR_results_treatment_review_screen.parquet')
edger_results = edger_results.rename(columns={'treatment': "cytokine"})
edger_results = edger_results[edger_results["cell_type"].isin(ALL_CELL_TYPES)].copy()

In [ ]:
from utils.processing_utils import create_edger_val_df

edger_df_padj = create_edger_val_df(edger_results, "adj_p_value")
edger_df_log2CPM = create_edger_val_df(edger_results, "logCPM")
edger_df_log2FC = create_edger_val_df(edger_results, "log_fc")

In [ ]:
significant_results = edger_results[
    (edger_results['adj_p_value'] < 0.05) &
    (edger_results['log_fc'].abs() > 1)
    ]

In [ ]:
edger_df_log2FC_main = pd.read_parquet("../../outputs/main_screen/output_tables/EdgeR_Log2FC.parquet")
edger_df_padj_main = pd.read_parquet("../../outputs/main_screen/output_tables/EdgeR_Padj.parquet")
raw_log2fc_values_by_donor_main = pd.read_parquet("../../outputs/main_screen/output_tables/raw_log2fc_values_by_donor.parquet")

# 1 - T=0h analysis and reproduction of the origin of donor variability

In [ ]:
current_plot_folder = os.path.join(plot_folder, "1_Time_0_and_baseline")
os.makedirs(current_plot_folder, exist_ok=True)

### Cell count

In [ ]:
sums = cell_count_review.groupby(level=(0, 1)).sum()
perturbation_order = cell_count_review.index.get_level_values(1).unique()
fractions = sums.div(sums.groupby(level=1).transform("sum"))

In [ ]:
cell_count_review_t0 = cell_count_review.xs("TimePoint-0", level='cytokine')
fractions_by_donor = cell_count_review_t0.div(cell_count_review_t0.groupby(level='donor').transform("sum"))

In [ ]:
s = cell_count_agg.xs('TimePoint-0', level=1).reset_index()
s.columns = ['cell_type', 'donor', 'val']
s = s.pivot_table(index='donor', columns='cell_type', values='val')
s = s.div(s.sum(axis=1), axis=0)
s

In [ ]:
# Prepare data for stacked barplot
plot_data = fractions_by_donor.unstack(level=0, fill_value=0).T
plot_data = plot_data.reindex(DONOR_ORDER_IFN_GROUPED, axis=1)  # Order donors
plot_data = plot_data.reindex(ALL_CELL_TYPES, axis=0)  # Order cell types

# Create stacked barplot of cell type counts by donor
fig, ax = plt.subplots(figsize=cm_to_inches((8, 4.3)))
# Create stacked barplot
bottom = np.zeros(len(DONOR_ORDER_IFN_GROUPED))
for cell_type in ALL_CELL_TYPES:
    if cell_type in plot_data.index:
        values = plot_data.loc[cell_type, DONOR_ORDER_IFN_GROUPED[::-1]].values
        ax.barh(DONOR_ORDER_IFN_GROUPED[::-1], values, left=bottom, 
               color=CELL_TYPE_COLORS[cell_type], label=cell_type, height=0.8)
        bottom += values


plt.tick_params(axis='y', length=0)
ax.set_xlabel('fraction of cells')
plt.tight_layout()
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=6, ncol=1)
plt.show()

In [ ]:
fig, axes = plt.subplots(int(len(ALL_CELL_TYPES)/2), 2, figsize=cm_to_inches((16, 18)), sharex=True, sharey=False)
axes = axes.flatten()
for cell_i, cell_type in enumerate(ALL_CELL_TYPES):
    ax = axes[cell_i]
    data = fractions[cell_type][perturbation_order]
    ax.tick_params(axis='x', rotation=90)
    ax.bar(data.index, data.values, color=CELL_TYPE_COLORS[cell_type])
    ax.set_title(cell_type) 
    
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, '1_cell_type_fractions.png'), dpi=400)

In [ ]:
setup_plotting_environment()
cell_fractions = calculate_cell_fractions(cell_count_agg).unstack(["treatment", "donor"])
results = []
for cell_type in cell_fractions.index:
    paired = pd.concat({"TimePoint-0": cell_fractions.loc[cell_type, "TimePoint-0"], "PBS": cell_fractions.loc[cell_type, "PBS"]}, axis=1).dropna()

    stat, p = stats.wilcoxon(paired["TimePoint-0"], paired["PBS"], alternative="two-sided")
    results.append({ "cell_type": cell_type, "stat": stat, "p": p})
results = pd.DataFrame(results).set_index('cell_type')
results["q"] = multipletests(results["p"], method="fdr_bh")[1]

cell_fractions_logfc = np.log2(cell_fractions['PBS'] / cell_fractions['TimePoint-0'])
results['log2fc'] = cell_fractions_logfc.mean(axis=1)

cell_fractions_logfc_melt = cell_fractions_logfc.reset_index().melt(id_vars='cell_type')
cell_fractions_logfc_melt['donor_type'] = cell_fractions_logfc_melt['donor'].map(DONOR_TO_GROUP)
cell_fractions_logfc_melt['donor_type'] = cell_fractions_logfc_melt['donor_type']
plt.figure(figsize=cm_to_inches((4, 2.35)))
sns.swarmplot(cell_fractions_logfc_melt, x='cell_type', y='value', s=3, hue='donor_type', palette=DONOR_COLOR_SCHEME)
for cell_i, cell in enumerate(results.index):
    val = results.loc[cell, 'log2fc']
    p = results.loc[cell, 'q']
    plt.plot([cell_i-0.25, cell_i+0.25], [val, val], color='black')
    plt.text(cell_i, 1.5, f'{p:.2g}', ha='center')
plt.text(-0.5, 1.5, f'p=', ha='center')
plt.xlabel("")
plt.tick_params(axis='x', length=0)
plt.xticks(rotation=25)
plt.ylabel("log2(fraction 24h / 0h)")
ax = plt.gca()
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, [label.replace("interferon", "IFN") for label in labels], loc=[0.7, 0.2], title=None)
plt.savefig(os.path.join(current_plot_folder, "1_cell_abundances_to_t0.svg"))
plt.show()

In [ ]:
2**cell_fractions_logfc_melt.groupby('cell_type')['value'].median()

### Change relative to time 0

In [ ]:
edger_t0 = pd.read_parquet('../../data/review_screen/scrna/edgeR_results_time_review_screen.parquet')
edger_t0 = edger_t0.rename(columns={'treatment': "cytokine"})
edger_t0 = edger_t0[edger_t0["cell_type"].isin(relevant_cell_types)].copy()

In [ ]:
edger_df_t0_padj = create_edger_val_df(edger_t0, "adj_p_value")
edger_df_t0_log2CPM = create_edger_val_df(edger_t0, "logCPM")
edger_df_t0_log2FC = create_edger_val_df(edger_t0, "log_fc")

In [ ]:
raw_log2fc_by_donor_t0, raw_log2fc_values_mean_t0, raw_log2fc_values_std_t0 = get_raw_log2fc_values(
    count_df = normalize_df_to_rpm(count_df_raw, along_axis=0),
    count_df_raw = count_df_raw,
    baseline_label = 'TimePoint-0',
    require_both_counts=False,
    )

In [ ]:
raw_log2fc_by_donor_t0 = raw_log2fc_by_donor_t0[
    raw_log2fc_by_donor_t0.index.isin(protein_coding_genes) & ~raw_log2fc_by_donor_t0.index.isin(sex_gene_list)
]

In [ ]:
cell_types = ['CD4 T cell', 'CD14 Mono', 'NK CD56hi', 'HSPC']
fig, axes = plt.subplots(1, 4, figsize=cm_to_inches((18, 4)), sharex = True, sharey = True)
axes = axes.flatten()

i = 0
for cell_type in cell_types:
        ax = axes[i]
        i+=1

        x = edger_df_t0_log2FC[cell_type]['PBS'].dropna()
        y = raw_log2fc_values_mean_t0[cell_type]['PBS'].dropna()
        x, y = get_common_index(x ,y)

        ax.scatter(x, y, s=3, alpha=0.5, edgecolor="none")
        ax.plot([-6, 6], [-6, 6], ls='--', lw=0.5, color='black')
        r = np.corrcoef(x,y)[0, 1]
        ax.set_title(f"{cell_type}, r={r:.2f}")
        ax.set_xlabel('log2fc_edger')
        ax.set_ylabel('log2fc_raw')

plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, '1_raw_edger_comp.png'), dpi=300)
plt.show()

In [ ]:
cell_type = 'CD14 Mono'
genes = ['IFIT1', 'CXCL9', 'CXCL10', # antiviral
         'CXCL5', 'CXCL8', 'IL1B', # IL-1 / neutrophil-recruiting
         'TNF', 'PTGS2', 'NFKBIA'] # generic inflammation
fig, axes = plt.subplots(3, 3, figsize=cm_to_inches((12, 8)), sharex=True)
axes = axes.flatten()
for gene_i, gene in enumerate(genes):
    ax = axes[gene_i]
    vals_pbs = count_df_norm[cell_type]['PBS'][DONOR_ORDER_IFN_GROUPED]
    #vals_t0 = count_df_norm[cell_type]['TimePoint-0'][DONOR_ORDER_IFN_GROUPED]
    vals_t0 = count_df_norm[cell_type]['AB-IFN-gamma'][DONOR_ORDER_IFN_GROUPED]
    x_vals = np.arange(len(DONOR_ORDER_IFN_GROUPED))
    ax.bar(x_vals - 0.2,vals_t0.loc[gene],width=0.4,color="#A3ABB4",label="0h",)
    ax.bar(x_vals + 0.2,vals_pbs.loc[gene],width=0.4,color="#3F6B8A",label="24h (PBS)",)
    n_ifn = 4  # D1, D3, D4, D10

    ax.axvspan(-1, n_ifn - 0.5,color=DONOR_COLOR_SCHEME['interferon'],alpha=0.1,zorder=0,)
    ax.axvspan(n_ifn - 0.5, 13,color=DONOR_COLOR_SCHEME['non-interferon'],alpha=0.1,zorder=0,)
    ax.set_xlim(-1, 12)
    ax.set_xticks(x_vals, labels=DONOR_ORDER_IFN_GROUPED)
    ax.tick_params(axis='y', rotation=90)
    for label in ax.get_yticklabels():
        label.set_horizontalalignment('center')
        label.set_verticalalignment('center')
    ax.tick_params(axis='x', rotation=90, length=0)
    ax.set_title(gene)

    if gene_i % 3 == 0:
        ax.set_ylabel("cpm")

    if gene_i == 0:
        ax.legend(fontsize=7)

plt.subplots_adjust(wspace=0.2, hspace=0.4)
plt.suptitle(cell_type, y=0.99, fontsize=8)
plt.savefig(os.path.join(current_plot_folder, "1_Monocyte_gene_expression.svg"))
plt.show()

In [ ]:
import gseapy as gp
msigdb = gp.Msigdb()
hallmark = msigdb.get_gmt(category="h.all", dbver="2026.1.Hs")
ISGs = (set(hallmark["HALLMARK_INTERFERON_ALPHA_RESPONSE"]) | set(hallmark["HALLMARK_INTERFERON_GAMMA_RESPONSE"]))

In [ ]:
cell_type = "CD14 Mono"
donors = DONOR_ORDER_IFN_GROUPED
donor_colors = dict(zip(donors, sns.color_palette("tab20", len(donors))))
genes = ['IFIT1', 'IFIT2', 'IFIT3', 'ISG15', 'LAMP3', 'CXCL9', 'CXCL10', 'CXCL11']

vals_t0 = np.log10(count_df_norm[cell_type]["TimePoint-0"].loc[genes, donors])
vals_pbs = np.log10(count_df_norm[cell_type]["PBS"].loc[genes, donors])

fig, ax = plt.subplots(figsize=cm_to_inches((8, 4)))
gene_x = np.arange(len(genes))
offsets = np.linspace(-0.32, 0.32, len(donors))
offset = 0.15

rng = np.random.default_rng(0)
for donor in donors:
    a, b = vals_t0[donor].to_numpy(), vals_pbs[donor].to_numpy()
    color = DONOR_COLOR_DIRECT[donor]
    jitter = rng.uniform(-0.05, 0.05, len(gene_x))

    ax.scatter(gene_x - offset + jitter, a, s=15, facecolors="none", edgecolors=color, linewidths=0.7, alpha=1, zorder=3)
    ax.scatter(gene_x + offset + jitter, b, s=15, color=color, linewidths=0, alpha=0.7, zorder=4)

# Legend: donor groups and time points.
for group, label in [
    ("interferon", "interferon"),
    ("non-interferon", "non-interferon"),
    ("other", "Other"),
]:
    ax.scatter([], [], s=12, color=DONOR_COLOR_SCHEME[group], label=label)

ax.scatter([], [], s=12, facecolors="none", edgecolors="black", label="0h")
ax.scatter([], [], s=12, color="black", label="24h (PBS)")

ax.set(
    xticks=gene_x, xticklabels=genes,
    xlim=(-0.6, len(genes) - 0.4),
    ylabel="log10(cpm)", title=cell_type,
)
ax.tick_params(axis="x", length=0)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis='x', rotation=90)
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left", ncol=1, fontsize=6)
ax.margins(y=0.12)
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, "1_Monocyte_gene_expression.svg"))
plt.show()

### Baseline log2FC and the interferon state

In [ ]:
from utils.processing_utils import get_raw_log2fc_to_condition_median
control_norm_gene_count_df_main = pd.read_parquet(os.path.join("../../outputs/main_screen/output_tables", "control_norm_gene_count_df_by_donor.parquet"))
control_raw_gene_count_df_main = pd.read_parquet(os.path.join("../../outputs/main_screen/output_tables", "control_raw_gene_count_df_by_donor.parquet"))

control_norm_gene_count_df_main.columns = pd.MultiIndex.from_tuples(
    (col[0], "PBS", col[1]) for col in control_norm_gene_count_df_main.columns
)
control_raw_gene_count_df_main.columns = pd.MultiIndex.from_tuples(
    (col[0], "PBS", col[1]) for col in control_raw_gene_count_df_main.columns
)
control_norm_gene_count_df_main.columns.names = ['cell_type', 'cytokine', 'donor']
control_raw_gene_count_df_main.columns.names = ['cell_type', 'cytokine', 'donor']

main_change_to_median_by_donor = get_raw_log2fc_to_condition_median(
    count_df = control_norm_gene_count_df_main,
    count_df_raw = control_raw_gene_count_df_main
)
main_change_to_median_by_donor.columns = pd.MultiIndex.from_tuples(
    [(c[0], c[1], c[2].replace("Donor", "D")) for c in main_change_to_median_by_donor.columns]
)

In [ ]:
review_change_to_median_by_donor = get_raw_log2fc_to_condition_median(
    count_df = count_df_norm.copy(),
    count_df_raw = count_df_raw.copy()
)

In [ ]:
# filter
review_change_to_median_by_donor = review_change_to_median_by_donor[review_change_to_median_by_donor.index.isin(protein_coding_genes)]
review_change_to_median_by_donor = review_change_to_median_by_donor.loc[
    ~review_change_to_median_by_donor.index.isin(sex_gene_list)
]
main_change_to_median_by_donor = main_change_to_median_by_donor[main_change_to_median_by_donor.index.isin(protein_coding_genes)]
main_change_to_median_by_donor = main_change_to_median_by_donor[
    ~main_change_to_median_by_donor.index.isin(sex_gene_list)
]

In [ ]:
import pyCytED
from pyCytED.gene_set import load_cytosig_signature, run_decoupler

def generate_cytosig_barplot(data, title, filename, mean_norm=True, background_data=None, ylim=None, figsize=None, signature='IFN1'):
    cyto_acts, cyto_padj = run_decoupler(
        data,
        cytosig_signature
    )
    present_donors = [d for d in DONOR_ORDER_IFN_GROUPED if d in cyto_acts.index]
    cyto_acts = cyto_acts[signature].loc[present_donors]
    if mean_norm:
        cyto_acts = cyto_acts - cyto_acts.mean()

    if not background_data is None:
        cyto_acts_background, _ =  run_decoupler(
            background_data,
            cytosig_signature
        )
        cyto_acts_background = cyto_acts_background[signature].loc[cyto_acts.index]
        if mean_norm:
            raise Exception("Not sure how mean norming interacts with background_data.")

    bar_colors = [DONOR_COLOR_DIRECT[donor] for donor in cyto_acts.index]
    if figsize is None:
        figsize = (5, 4)
    plt.figure(figsize=cm_to_inches(figsize))
    ax = plt.gca()
    x_pos = np.arange(len(cyto_acts))
    ax.bar(x_pos, cyto_acts.values, edgecolor='black', color=bar_colors, zorder=1)
    ax.set_xticks(x_pos, cyto_acts.index)
    if not background_data is None:
        ax.scatter(x_pos, cyto_acts_background.values, s=8, color='black', zorder=3, label='no AB')
        for x, start, end in zip(x_pos, cyto_acts_background.values, cyto_acts.values):
            if abs(end - start) >= 3:
                ax.annotate('', xy=(x, end), xytext=(x, start), zorder=4,
                            arrowprops=dict(arrowstyle='->', color='black', lw=0.6, shrinkA=0, shrinkB=0, mutation_scale=5))
        ax.legend(fontsize=7, loc='upper right', frameon=False)

    plt.xlabel("Donor")
    plt.ylabel(f"CytoSig {signature} activity")
    plt.tick_params(axis='x', length=0)
    plt.title(title, fontsize=7)
    if ylim is None:
        plt.ylim(-40, 40)
    else:
        plt.ylim(ylim[0], ylim[1])
    apply_greek_labels_heatmap(plt.gca())
    
    plt.tight_layout()
    plt.savefig(os.path.join(current_plot_folder, f"{filename}.svg"), dpi=300)
    plt.show()
    if background_data is None:
        return cyto_acts
    return cyto_acts, cyto_acts_background

cytosig_signature = load_cytosig_signature()

In [ ]:
data = main_change_to_median_by_donor['CD4 T cell', 'PBS'].T
title = 'first screen\nbaseline log2FC, CD4 T cell (24h)'
filename = '1_cytosig_24h_main'
cyto_acts_main_24h = generate_cytosig_barplot(data, title, filename)

data = review_change_to_median_by_donor['CD4 T cell', 'PBS'].T
title = 'second screen\nbaseline log2FC, CD4 T cell (24h)'
filename = '1_cytosig_24h_review'
cyto_acts_review_24h = generate_cytosig_barplot(data, title, filename)

data = review_change_to_median_by_donor['CD4 T cell', 'TimePoint-0'].T
title = 'second screen\nbaseline log2FC, CD4 T cell (0h)'
filename = '1_cytosig_0h_review'
cyto_acts_review_0h = generate_cytosig_barplot(data, title, filename)

data = raw_log2fc_by_donor_t0['CD4 T cell', 'PBS'].T
title = 'log2FC 24h - 0h, CD4 T cell'
filename = '1_cytosig_24h to 0h'
cyto_acts_review_24h_vs_0h = generate_cytosig_barplot(data, title, filename, mean_norm=False)

In [ ]:
cell = "CD4 T cell"

state_0h = review_change_to_median_by_donor[cell, "TimePoint-0"]
state_24h = state_0h + raw_log2fc_by_donor_t0[cell, "PBS"]

data = pd.concat({"0h": state_0h, "24h": state_24h}, axis=1).dropna().T

sample_index = data.index.copy()
data.index = data.index.map(lambda x: f"{x[0]}__{x[1]}")

activities, _ = run_decoupler(data, cytosig_signature)
activities = activities.reindex(data.index)
activities.index = sample_index

scores = activities["IFN1"].unstack(level=0)
df = scores.reindex(DONOR_ORDER_IFN_GROUPED).dropna(subset=["0h", "24h"])
fig, ax = plt.subplots(figsize=cm_to_inches((6, 3)))
ax.axhline(0, color="0.8", lw=0.5, ls="--")

for i, (donor, row) in enumerate(df.iterrows()):
    a, b = row[["0h", "24h"]]
    ax.plot([i, i], [a, b], color="black", lw=0.6, zorder=1)
    if abs(b - a) >= 3:
        ax.annotate("", xy=(i, b), xytext=(i, a),
                    arrowprops=dict(arrowstyle="->", color="black", lw=0.6,
                                    shrinkA=3, shrinkB=3, mutation_scale=5))
    ax.scatter([i, i], [a, b], s=14,
               facecolors=["white", DONOR_COLOR_DIRECT[donor]],
               edgecolors="black", linewidths=0.6, zorder=3)

for i in range(1, len(df)):
    if DONOR_TO_GROUP[df.index[i]] != DONOR_TO_GROUP[df.index[i - 1]]:
        ax.axvline(i - 0.5, color="0.9", lw=0.5)

for label, color in [("0h", "white"), ("24h", "0.5")]:
    ax.scatter([], [], s=14, facecolors=color, edgecolors="black",
               linewidths=0.6, label=label)

ax.set(xticks=np.arange(len(df)), xticklabels=df.index, xlim=(-0.6, len(df) - 0.4), ylabel="CytoSig IFN1 activity")
ax.set_title(cell, fontsize=7, pad=0)
ax.legend(frameon=False, ncol=2, loc="lower center",bbox_to_anchor=(0.6, 0.7), fontsize=6)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="x", length=0)
ax.margins(y=0.12)
plt.savefig(os.path.join(current_plot_folder, "1_evolution_24h.svg"))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=cm_to_inches((4.75, 3.5)))
plt.scatter(cyto_acts_main_24h, cyto_acts_review_24h, color='white')
r = np.corrcoef(cyto_acts_main_24h.values, cyto_acts_review_24h.values)[0, 1]
for i in range(len(cyto_acts_review_24h.index)):
    donor = cyto_acts_review_24h.index[i]
    plt.text(
        x = cyto_acts_main_24h.iloc[i],
        y = cyto_acts_review_24h.iloc[i],
        s = donor,
        color = DONOR_COLOR_DIRECT[donor]
        )
plt.text(-40, 25, f"r={r:.2f}", fontsize=7)
plt.xlabel(introduce_greek_letters("CytoSig IFN1 activity\n(first screen)"))
plt.ylabel(introduce_greek_letters("CytoSig IFN1 activity\n(second screen)"))
plt.xlim([-50, 50])
plt.ylim([-50, 50])
plt.title("CD4 T cell")
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, '1_donor_state_reproduction.svg'), dpi=400)
plt.show()

### The effects of antibody treatments on the state at t=24h

In [ ]:
all_cyto_acts_ab = {}
for cell_type in ['CD4 T cell', 'CD14 Mono']:
    for AB in ['AB-IFN-gamma', 'AB-IL-6', 'AB-TNF-alpha', 'AB-IL-10', 'IL-1Ra']:
        data = raw_log2fc_by_donor_t0[cell_type, AB].T
        background_data = raw_log2fc_by_donor_t0[cell_type, 'PBS'].T
        title = f'{cell_type}, {format_treatment(AB)}'
        filename = f'1_cytosig_24_0_{AB}_{cell_type}'
        if len(data) == 12:
            figsize = (4.75,3.5)
        else:
            figsize = (3.1,3.5)
        cyto_acts_review_0h, cyto_acts_background = generate_cytosig_barplot(
            data, title, filename, background_data=background_data,
            mean_norm=False, ylim = (-5, 40), figsize=figsize
            )
        all_cyto_acts_ab[(cell_type, AB)] = cyto_acts_review_0h - cyto_acts_background
all_cyto_acts_ab = pd.DataFrame(all_cyto_acts_ab)

In [ ]:
setup_plotting_environment()
plt.figure(figsize=cm_to_inches((4,3)))
for AB in AB_CONDITIONS:
    vals = all_cyto_acts_ab.xs(AB, level=1, axis=1)
    plt.scatter(vals['CD4 T cell'], vals['CD14 Mono'], label=format_treatment(AB), s=5)
x = all_cyto_acts_ab['CD4 T cell'].values.flatten()
y = all_cyto_acts_ab['CD14 Mono'].values.flatten()
plt.xlabel(rf"$\Delta$CytoSig IFN1 (CD4 T)")
plt.ylabel(rf"$\Delta$CytoSig IFN1 (CD14 Mono)")
mask = np.isfinite(x) & np.isfinite(y)
x = x[mask]
y = y[mask]
r = np.corrcoef(x,y)[0,1]
plt.text(-18, 0, f'r={r:.2f}')
plt.legend(ncol=2)
plt.show()

In [ ]:
all_cyto_acts_ab = {}
for cell_type in ['CD14 Mono']:
    for AB_pair in [('AB-IFN-gamma', 'TNFA', (-30, 5)), ('AB-TNF-alpha', 'TNFA', (-30, 5)), ('IL-1Ra', 'IL1B', (-30, 5))]:
        AB = AB_pair[0]
        signature = AB_pair[1]
        ylim = AB_pair[2]
        data = raw_log2fc_by_donor_t0[cell_type, AB].T
        background_data = raw_log2fc_by_donor_t0[cell_type, 'PBS'].T
        title = f'{cell_type}, {format_treatment(AB)}'
        filename = f'1_cytosig_24_0_sig_{signature}_{AB}_{cell_type}'
        if len(data) == 12:
            figsize = (4.75,3.5)
        else:
            figsize = (3.1,3.5)
        cyto_acts_review_0h, cyto_acts_background = generate_cytosig_barplot(
            data, title, filename, background_data=background_data,
            mean_norm=False, ylim = ylim, figsize=figsize, signature=signature
            )
        all_cyto_acts_ab[(cell_type, AB)] = cyto_acts_review_0h - cyto_acts_background
all_cyto_acts_ab = pd.DataFrame(all_cyto_acts_ab)

### Donor 2 - IL32-like state

In [ ]:
all_corrs = {}
for donor in DONOR_ORDER:
    df_donor = review_change_to_median_by_donor.loc[:, (slice(None), 'TimePoint-0', donor)].droplevel([1, 2], axis=1)
    df_edger = edger_df_log2FC_main.loc[:, (slice(None), 'IL-32-beta')].droplevel(1, axis=1)
    corr = df_edger.corrwith(df_donor, method="pearson")[ALL_CELL_TYPES]
    all_corrs[donor] = corr
all_corrs = pd.DataFrame(all_corrs)

plt.figure(figsize=cm_to_inches((3.8, 4.7)))
for donor_name in DONOR_ORDER:
    plt.scatter(
        all_corrs.loc[ALL_CELL_TYPES[::-1], donor_name],
        np.arange(len(ALL_CELL_TYPES)),
        alpha=1,
        s=3,
        color="black" if donor_name != "D2" else DONOR_COLOR_SCHEME_D2["D2"],
        label="D2" if donor_name == "D2" else None,
        edgecolors="none",
        rasterized=True,
        zorder = 12 if donor_name == "D2" else 1
        )
    
plt.yticks(np.arange(len(ALL_CELL_TYPES)), ALL_CELL_TYPES[::-1], fontsize=6)
plt.tick_params(axis='y', length=0)
plt.xlabel(introduce_greek_letters('r(baseline log2FC, consensus log2FC)'))
plt.xticks([-0.5,0,0.5,1])
plt.title(introduce_greek_letters("IL-32-beta\nbaseline at t=0"), fontsize=7)
# plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, "donor_il32_corr_by_cell_type_t0.svg"), dpi=400)
plt.show()

In [ ]:
counts = count_df_norm_agg['Mono/cDC']['TimePoint-0']
expression_filter = counts.max(axis=1) >= 10
expression_filter = expression_filter[expression_filter].index
counts = counts.loc[expression_filter]

proteins_of_interest = {
    'inflammatory\ncytokines': ['IL1A', 'IL1B', 'IL12', 'IL18', 'TNF', 'IL6'],
    'neutrophil\nchemokines': ['CCL2', 'CCL3', 'CCL4', 'CCL20', 'CXCL1', 'CXCL2', 'CXCL3', 'CXCL5', 'CXCL6', 'CXCL8'],
    'alarmins': ['S100A8', 'S100A9', 'S100A12'],
    "inflammatory\nmachinery": ["NLRP3", "TREM1", "PTGS2"],
    }
genes_by_category = {
    category: [g for g in genes if g in expression_filter] for category, genes in proteins_of_interest.items()
}
genes_by_category = {k: v for k, v in genes_by_category.items() if v}

proteins_of_interest_flat = [g for genes in genes_by_category.values() for g in genes]
z_scores_sub = robust_z_score(counts).loc[proteins_of_interest_flat]
plot_df = (z_scores_sub.rename_axis("gene_symbols").reset_index().melt(
    id_vars="gene_symbols", var_name="donor", value_name="value")
)

fig, ax = plt.subplots(figsize=cm_to_inches((6, 3)))

sns.stripplot(
    data=plot_df[plot_df["donor"] != "D2"],
    x="gene_symbols", y="value", order=proteins_of_interest_flat,
    hue="donor", palette=DONOR_COLOR_SCHEME_D2, legend=False, jitter=0.1, size=3, ax=ax, zorder=1,
)

sns.stripplot(
    data=plot_df[plot_df["donor"] == "D2"],
    x="gene_symbols", y="value", order=proteins_of_interest_flat,
    color="red", jitter=0.1, size=3.5, ax=ax, zorder=10,
)

# titles
start = 0
for category, genes in genes_by_category.items():
    end = start + len(genes) - 1
    center = (start + end) / 2

    ax.text(center, 1.04, category, transform=ax.get_xaxis_transform(),
        ha="center", va="bottom", fontsize=6, clip_on=False,)
    if start > 0:
        ax.axvline(start - 0.5, color="0.8", linewidth=0.6)
    start = end + 1

# # stats
# paired_df = pd.DataFrame({ "D2": z_scores_sub["D2"], "highest_other": z_scores_sub.drop(columns="D2").max(axis=1), }).dropna()
# paired_df["margin"] = paired_df["D2"] - paired_df["highest_other"]
# result = stats.wilcoxon(paired_df["margin"], alternative="greater", zero_method="wilcox")
# ax.text(10, 8, f"p={result.pvalue:.1E}")

ax.tick_params(axis="x", rotation=90, length=0)
ax.set_xlabel("")
ax.set_ylabel("robust z-score\nin myeloid cells")

plt.subplots_adjust(top=0.82)
plt.savefig(os.path.join(current_plot_folder, f'Donor2_t0_state.svg'))
plt.show()

In [ ]:
import decoupler as dc

progeny = dc.op.progeny(organism="human")
net = progeny.sort_values("padj").groupby("source").head(500)[["source", "target", "weight"]]
data = review_change_to_median_by_donor["CD14 Mono", "TimePoint-0"].T.fillna(0)
progeny_acts, progeny_pvals = dc.mt.ulm(data=data, net=net, tmin=5)
progeny_acts[["NFkB", "TNFa", "JAK-STAT"]]

In [ ]:
import decoupler as dc

progeny = dc.op.progeny(organism="human")
progeny_net = progeny.sort_values("padj").groupby("source").head(500)[["source", "target", "weight"]]
pathways = ["NFkB", "TNFa", "Hypoxia", "p53", "TGFb"]
timepoints = {"TimePoint-0": "0 h", "PBS": "24 h (PBS)"}

progeny_acts = []
for condition, label in timepoints.items():
    data = review_change_to_median_by_donor["CD14 Mono", condition].T.fillna(0)   # donors x genes
    acts, _ = dc.mt.ulm(data=data, net=progeny_net, tmin=5)
    acts = acts[pathways].rename_axis("donor").reset_index().melt(id_vars="donor", var_name="pathway", value_name="activity")
    acts["time"] = label
    progeny_acts.append(acts)
progeny_acts = pd.concat(progeny_acts)

fig, axes = plt.subplots(1, len(timepoints), figsize=cm_to_inches((7, 3.5)), sharey=True)
for ax, label in zip(axes, timepoints.values()):
    sub = progeny_acts[progeny_acts["time"] == label]
    sns.stripplot(data=sub[sub["donor"] != "D2"], x="pathway", y="activity", order=pathways,
                  color="black", jitter=0.15, size=2.5, ax=ax, zorder=1)
    sns.stripplot(data=sub[sub["donor"] == "D2"], x="pathway", y="activity", order=pathways,
                  color=DONOR_COLOR_SCHEME_D2["D2"], jitter=0, size=3.5, ax=ax, zorder=10)
    ax.axhline(0, color="0.7", lw=0.5, zorder=0)
    ax.set_title(label, fontsize=7)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=90, length=0)
axes[0].set_ylabel("PROGENy activity\n(CD14 Mono)")
axes[1].set_ylabel("")
axes[1].text(0.8, 0.95, "D2", color=DONOR_COLOR_SCHEME_D2["D2"], transform=axes[1].transAxes, fontsize=6, va="top")
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, "Donor2_progeny_0h_24h.svg"), bbox_inches="tight")
plt.show()

# 2 - Supernatant Proteomics

In [ ]:
current_plot_folder = os.path.join(plot_folder, '2_proteomics')
os.makedirs(current_plot_folder, exist_ok=True)

In [ ]:
# Censor treatments
TREATMENT_CENSOR_PATTERNS = {
    "C3": ["C3a"],
    "IFNB1": ["IFN-beta"],
    "IFNG": ["IFN-gamma"],
    "IL10": ["IL-10"],
    "IL12p35": ["IL-12"],
    "IL12p40": ["IL-12"],
    "IL12p70": ["IL-12"],
    "IL15": ["IL-15"],
    "IL15;IL15RA": ["IL-15"],
    "IL18": ["IL-18"],
    "IL1RN": ["IL-1Ra"],
    "IL1A": ["IL-1Ra"],
    "IL1B": ["IL-1Ra"],
    "IL6": ["IL-6"],
    "TGFB1": ["TGF-beta1"],
    "TGFB1 (LAP)": ["TGF-beta1"],
    "TNF": ["TNF-alpha"],
    "IL32": ['IL-32-beta']
}

ALL_TREATMENTS = list(count_df_norm.columns.get_level_values(1).unique())
PROTEIN_TO_CENSORED_TREATMENTS = {}
for protein, patterns in TREATMENT_CENSOR_PATTERNS.items():
    excluded = []
    for pattern in patterns:
        treats = [treat for treat in ALL_TREATMENTS if pattern in treat]
        excluded.extend(treats)
    PROTEIN_TO_CENSORED_TREATMENTS[protein] = excluded

### Processing the input data

In [ ]:
def read_nomic_excel(path, sheet_name=0, adjust_conc=1):
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)

    # find the header row
    header_mask = raw.eq("plate_barcode")
    if not header_mask.any().any():
        raise ValueError("Could not find sample header row containing 'plate_barcode'.")
    header_row = header_mask.stack()[lambda x: x].index[0][0]

    # find the analyte start column
    protein_name_hits = raw.eq("Protein Name:")
    analyte_label_col = protein_name_hits.stack()[lambda x: x].index[0][1]
    analyte_start_col = analyte_label_col + 1

    # ----------- Extract sample metadata and values -----------
    # Sample metadata columns are everything before the analyte label column
    sample_cols = raw.iloc[header_row, :analyte_label_col].tolist()
    if not 'Unnamed: 11' in sample_cols:
        raise ValueError("Expected 'Unnamed: 11' column in sample metadata columns.")
    sample_cols = [
        "title" if str(x).startswith("Unnamed") else x
        for x in sample_cols
    ]
    sample_metadata = raw.iloc[header_row + 1:, :analyte_label_col].copy()
    sample_metadata.columns = sample_cols
    sample_metadata['donor'] = 'D' + sample_metadata['donor'].astype(str)
    sample_metadata.index = pd.MultiIndex.from_arrays(
        [sample_metadata['treatment'].astype(str), sample_metadata['donor'].astype(str)],
        names=['treatment', 'donor']
        )

    # ----------- Extract analyte metadata -----------
    # Analyte metadata rows are above or at the header row, in analyte_label_col
    analyte_meta_rows = {}
    for r in range(header_row + 1):
        label = raw.iat[r, analyte_label_col]
        if pd.notna(label):
            clean_label = (
                str(label)
                .strip()
                .rstrip(":")
                .replace(" ", "_")
                .replace("/", "_per_")
                .replace("(", "")
                .replace(")", "")
                .replace("-", "_")
                .replace("_pg_per_ml", "")
                .lower()
            )
            analyte_meta_rows[clean_label] = r

    if "protein_name" not in analyte_meta_rows:
        raise ValueError("Could not find 'Protein Name:' row.")

    protein_names = raw.iloc[analyte_meta_rows["protein_name"], analyte_start_col:].dropna()
    uniprot_names = raw.iloc[analyte_meta_rows['uniprot_id'], analyte_start_col:].dropna()

    n_analytes = len(protein_names)
    analyte_cols = list(range(analyte_start_col, analyte_start_col + n_analytes))

    analyte_metadata = pd.DataFrame({
        "excel_col": analyte_cols,
        "protein": protein_names.values,
        "uniprot_names": uniprot_names.values
    })

    # are there duplicate protein names?
    if analyte_metadata["protein"].duplicated().any():
        raise ValueError("Duplicate protein names found in analyte metadata.")

    for field, row_i in analyte_meta_rows.items():
        vals = raw.iloc[row_i, analyte_cols].values
        analyte_metadata[field] = vals

    # map names
    uniprot_df = pd.read_csv(os.path.join(proteomics_folder, "HUMAN_9606_idmapping.dat.gz"), sep="\t", header=None, names=["uniprot_id", "db", "db_id"])
    uniprot_dict = uniprot_df[uniprot_df['db'] == 'Gene_Name'].set_index('uniprot_id')['db_id'].to_dict()
    analyte_metadata["gene_name"] = analyte_metadata["uniprot_id"].apply(
        lambda x: '; '.join(sorted([
            uniprot_dict[s.strip()] if s.strip() in uniprot_dict else "missing"
            for s in str(x).split(";")
        ]))
    )
    missing = analyte_metadata[analyte_metadata['gene_name'].str.contains('missing')]
    for uniprot_id in missing['uniprot_id']:
        missing_protein = analyte_metadata[analyte_metadata['uniprot_id'] == uniprot_id]
        print(f"Missing gene name for Uniprot ID: {uniprot_id}, Protein: {missing_protein['protein'].values[0]}")
    print(f"Matching protein and gene names: {len(analyte_metadata[analyte_metadata['gene_name'] == analyte_metadata['protein']])} / {len(analyte_metadata)}")

    # ----------- Extract analyte values -----------
    # Raw sample x protein values
    values_raw = raw.iloc[
        header_row + 1: header_row + 1 + len(sample_metadata),
        analyte_cols
    ].copy()

    values_raw.columns = analyte_metadata["protein"].tolist()
    values_raw.index = sample_metadata.index
    values_raw = values_raw.dropna(how='all', axis=0)
    values_raw = values_raw.replace("Low bead count", np.nan)

    # adjust concentrations
    values_raw = values_raw / adjust_conc
    analyte_metadata['upper_limit_of_detection'] = analyte_metadata['upper_limit_of_detection'] / adjust_conc
    analyte_metadata['lower_limit_of_detection'] = analyte_metadata['lower_limit_of_detection'] / adjust_conc

    sample_metadata = sample_metadata.dropna(subset='title')
    assert sample_metadata.shape[0] == values_raw.shape[0], "Mismatch in number of samples between metadata and values."

    return sample_metadata, analyte_metadata, values_raw
    
input_path = "nomic_2026-06-26_imputed pg ml.xlsx"

# adjust conc = 1000 sets this to ng/ml
proteomics_sample_metadata, proteomics_analyte_metadata, proteomics_values_raw = read_nomic_excel(os.path.join(proteomics_folder, 'original', input_path), adjust_conc=1000.0)
# imputed concentrations are already adjusted for a 1:2 dilution in the sample prep; ulod and llod are not
proteomics_analyte_metadata['upper_limit_of_detection'] = 2*proteomics_analyte_metadata['upper_limit_of_detection']
proteomics_analyte_metadata['lower_limit_of_detection'] = 2*proteomics_analyte_metadata['lower_limit_of_detection']

In [ ]:
proteomics_analyte_metadata = proteomics_analyte_metadata.drop(['protein_name', 'uniprot_names'], axis=1)

# add IFNA
index = proteomics_analyte_metadata[proteomics_analyte_metadata['protein'] == 'IFNA1'].iloc[0].name
proteomics_analyte_metadata.loc[index, 'gene_name'] = 'IFNA1'
proteomics_analyte_metadata.loc[index, 'uniprot_id'] = 'P0DY56'

# add C4B
index = proteomics_analyte_metadata[proteomics_analyte_metadata['protein'] == 'C4B'].iloc[0].name
proteomics_analyte_metadata.loc[index, 'gene_name'] = 'C4B'

# add IL27 variant
row = proteomics_analyte_metadata[proteomics_analyte_metadata['protein'] == 'IL27'].iloc[0]
row['protein'] = 'EBI3;IL27'
row['uniprot_id'] = 'Q8NEV9;Q14213'
row['gene_name'] = 'EBI3;IL27'
row = row.to_frame().T
proteomics_analyte_metadata = pd.concat([proteomics_analyte_metadata, row], axis=0, ignore_index=True).reset_index(drop=True)
proteomics_values_raw['EBI3;IL27'] = proteomics_values_raw['IL27']

proteomics_analyte_metadata['uniprot_id'] = proteomics_analyte_metadata['uniprot_id'].apply(
    lambda x: ";".join(sorted([s.strip() for s in x.split(';')]))
)
proteomics_analyte_metadata['cytokine'] = proteomics_analyte_metadata['uniprot_id'].apply(lambda x: uniprot_id_to_cytokine.get(x, 'none'))

In [ ]:
proteomics_analyte_metadata.to_parquet(os.path.join(output_folder, 'proteomics_analyte_metadata.parquet'))
proteomics_values_raw.to_parquet(os.path.join(output_folder, 'proteomics_values_raw.parquet'))
proteomics_sample_metadata.to_parquet(os.path.join(output_folder, 'proteomics_sample_metadata.parquet'))

## 2.1 - Protein secretion annotation

This uses uniprot and hpa to prompt an LLM to classify proteins. Results are then further compared to databases, and finally manually cleaned up.
The final annotation is provided in this repo, so LLM call cells can simply be commented out.

### Process uniprot

In [ ]:
desired_accessions = {
    accession.strip()
    for value in proteomics_analyte_metadata["uniprot_id"].dropna().astype(str)
    for accession in value.split(";")
    if accession.strip()
}

desired_accessions_cytokines = {
    accession.strip()
    for value in cytokine_info["uniprot"].dropna().astype(str)
    for accession in value.split(";")
    if accession.strip()
}

desired_accessions = sorted(desired_accessions.union(desired_accessions_cytokines))

In [ ]:
from io import StringIO
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen

FIELD_TO_COLUMN = {
    "accession": "Entry",
    "gene_names": "Gene Names",
    "protein_name": "Protein names",
    "cc_subcellular_location": "Subcellular location [CC]",
    "keyword": "Keywords",
    "ft_signal": "Signal peptide",
    "ft_transmem": "Transmembrane",
}
FORMATTED_COLUMNS = list(FIELD_TO_COLUMN.values())
RAW_COLUMNS = ["Entry", "raw_text"]

def _matching_columns(columns: pd.Index, aliases: list[str]) -> list[str]:
    """Find an original or pandas-suffixed duplicate of each alias."""
    matches = []
    for column in columns:
        text = str(column)
        if any(text == alias or re.fullmatch(re.escape(alias) + r"\.\d+", text) for alias in aliases):
            matches.append(column)
    return matches

def _first_nonempty(values: pd.Series):
    for value in values:
        if pd.notna(value) and str(value).strip():
            return value
    return pd.NA

def _coalesce_columns(frame, columns):
    if not columns:
        return pd.Series(pd.NA, index=frame.index, dtype="string")

    result = frame[columns[0]].copy()
    for column in columns[1:]:
        result = result.fillna(frame[column])
    return result

def _canonicalise_formatted(frame: pd.DataFrame) -> pd.DataFrame:
    """Repair old/new UniProt TSV schemas into one Entry-indexed table."""
    aliases = {
        "Entry": ["Entry", "accession", "Accessions"],
        "Gene Names": ["Gene Names", "gene_names"],
        "Protein names": ["Protein names", "Protein Name", "protein_name"],
        "Subcellular location [CC]": ["Subcellular location [CC]", "cc_subcellular_location"],
        "Keywords": ["Keywords", "keyword"],
        "Signal peptide": ["Signal peptide", "ft_signal"],
        "Transmembrane": ["Transmembrane", "ft_transmem"],
    }
    repaired = pd.DataFrame(index=frame.index)
    for target in FORMATTED_COLUMNS:
        source_columns = _matching_columns(frame.columns, aliases[target])
        repaired[target] = _coalesce_columns(frame, source_columns)
    repaired["Entry"] = repaired["Entry"].astype("string").str.strip()
    repaired = repaired.dropna(subset=["Entry"])
    repaired = repaired.loc[repaired["Entry"].ne("")]
    repaired = repaired.groupby("Entry", sort=False, as_index=False).agg(_first_nonempty)
    return repaired.set_index("Entry")[FORMATTED_COLUMNS[1:]]


def _canonicalise_raw(frame: pd.DataFrame) -> pd.DataFrame:
    repaired = pd.DataFrame(index=frame.index)
    entry_columns = _matching_columns(frame.columns, ["Entry", "accession"])
    text_columns = _matching_columns(frame.columns, ["raw_text"])
    repaired["Entry"] = frame[entry_columns].bfill(axis=1).iloc[:, 0] if entry_columns else pd.NA
    repaired["raw_text"] = frame[text_columns].bfill(axis=1).iloc[:, 0] if text_columns else pd.NA
    repaired["Entry"] = repaired["Entry"].astype("string").str.strip()
    repaired = repaired.dropna(subset=["Entry"])
    repaired = repaired.loc[repaired["Entry"].ne("")]
    repaired = repaired.groupby("Entry", sort=False, as_index=False).agg(_first_nonempty)
    return repaired.set_index("Entry")[["raw_text"]]

def _read_formatted_cache(path: Path) -> pd.DataFrame:
    return _canonicalise_formatted(pd.read_csv(path, sep="\t", dtype="string")) if path.exists() else pd.DataFrame(columns=FORMATTED_COLUMNS[1:]).rename_axis("Entry")

def _read_raw_cache(path: Path) -> pd.DataFrame:
    return _canonicalise_raw(pd.read_csv(path, sep="\t", dtype="string")) if path.exists() else pd.DataFrame(columns=["raw_text"]).rename_axis("Entry")

def _fetch_formatted(accessions: list[str]) -> pd.DataFrame:
    batches = []
    for start in range(0, len(accessions), 100):
        batch = accessions[start : start + 100]
        url = "https://rest.uniprot.org/uniprotkb/accessions?" + urlencode(
            {"accessions": ",".join(batch), "fields": ",".join(FIELD_TO_COLUMN), "format": "tsv"}
        )
        with urlopen(url, timeout=60) as response:
            batches.append(pd.read_csv(StringIO(response.read().decode("utf-8")), sep="\t", dtype="string"))
    if not batches:
        return pd.DataFrame(columns=FORMATTED_COLUMNS[1:]).rename_axis("Entry")
    return _canonicalise_formatted(pd.concat(batches, ignore_index=True))

def _fetch_raw(accession: str) -> str:
    with urlopen(f"https://rest.uniprot.org/uniprotkb/{accession}.txt", timeout=60) as response:
        return response.read().decode("utf-8")

def update_uniprot_caches(desired_accessions, formatted_path: Path, raw_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    desired = sorted({str(accession).strip() for accession in desired_accessions if pd.notna(accession) and str(accession).strip()})

    formatted_present = _read_formatted_cache(formatted_path)
    formatted_new = _fetch_formatted(sorted(set(desired) - set(formatted_present.index)))
    uniprot = _canonicalise_formatted(pd.concat([formatted_present.reset_index(), formatted_new.reset_index()], ignore_index=True))
    formatted_path.parent.mkdir(parents=True, exist_ok=True)
    uniprot.reset_index().to_csv(formatted_path, sep="\t", index=False)

    raw_present = _read_raw_cache(raw_path)
    raw_records = []
    for accession in sorted(set(desired) - set(raw_present.index)):
        try:
            raw_records.append({"Entry": accession, "raw_text": _fetch_raw(accession)})
        except Exception as exc:
            # Preserve a visible failure in the cache, but do not abort
            raw_records.append({"Entry": accession, "raw_text": f"ERROR: {exc}"})
    raw_new = _canonicalise_raw(pd.DataFrame.from_records(raw_records)) if raw_records else pd.DataFrame(columns=["raw_text"]).rename_axis("Entry")
    uniprot_raw = _canonicalise_raw(pd.concat([raw_present.reset_index(), raw_new.reset_index()], ignore_index=True))
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    uniprot_raw.reset_index().to_csv(raw_path, sep="\t", index=False)

    assert uniprot.index.is_unique, "Formatted UniProt Entry index must be unique"
    assert uniprot_raw.index.is_unique, "Raw UniProt Entry index must be unique"
    assert list(uniprot.columns) == FORMATTED_COLUMNS[1:], "Formatted UniProt columns changed unexpectedly"
    return uniprot, uniprot_raw

# ---------- Formatted UniProt / raw UniProt ----------
UNIPROT_PATH = Path("../../data/protein_atlas/uniprot.csv")
UNIPROT_RAW_PATH = Path("../../data/protein_atlas/uniprot_raw.tsv")

uniprot, uniprot_raw = update_uniprot_caches(desired_accessions, UNIPROT_PATH, UNIPROT_RAW_PATH)
uniprot.head(3)

In [ ]:
uniprot['Primary Gene Name'] = uniprot['Gene Names'].str.split(" ").str[0]

In [ ]:
# https://www.proteinatlas.org/about/download

import zipfile
protein_atlas_path = "../../data/protein_atlas/proteinatlas.tsv.zip"
with zipfile.ZipFile(protein_atlas_path) as archive:
    with archive.open("proteinatlas.tsv") as handle:
        hpa = pd.read_csv(
            handle,
            sep="\t",
            usecols=[
                "Gene",
                "Uniprot",
                "Protein class",
                "Secretome location",
                "Secretome function",
                "Subcellular location",
            ],
        )
hpa = hpa.dropna(subset='Gene')
hpa = hpa.drop_duplicates(subset='Gene', keep='first') # only a handful

# fix a couple oversights
hpa = hpa.set_index("Gene")
hpa.loc['IFNA1', 'Uniprot'] = 'P0DY56'
hpa.loc['C1QB', 'Uniprot'] = 'P02746'
hpa.loc['IFNL4', 'Uniprot'] = 'K9M1U5'
hpa.loc['CCL4L1'] = hpa.loc['CCL4L2']
hpa.loc['CCL3L3'] = hpa.loc['CCL3L1']
hpa = hpa.reset_index()

missing = uniprot.index.difference(hpa['Uniprot'])
for miss in missing:
    gene_names = uniprot.loc[miss, 'Gene Names'].split(' ')
    hpa_matches = hpa[hpa['Gene'].isin(gene_names)]
    if len(hpa_matches) > 1:
        raise Exception("More than one hit!")
    if len(hpa_matches) == 0:
        print("No match found for: ", miss)
    index = hpa_matches.index[0]
    hpa.loc[index, 'Uniprot'] = miss
    print(f"Filled in {miss} for {hpa_matches['Gene'].iloc[0]}")

hpa['Uniprot'] = hpa['Uniprot'].fillna("missing")
hpa['Uniprot'] = hpa['Uniprot'].apply(lambda x: [s.strip() for s in x.split(',')])
hpa = hpa.explode("Uniprot")

In [ ]:
def _join_unique_nonempty(values):
    parts = []
    for value in values:
        if pd.isna(value):
            continue
        for part in str(value).split(";"):
            part = part.strip()
            if part and part not in parts:
                parts.append(part)
    return "; ".join(parts) if parts else pd.NA

hpa_filter = hpa[hpa['Uniprot'].isin(uniprot.index)].copy()
hpa_filter = (
    hpa_filter.groupby("Uniprot", sort=False, dropna=False)
    .agg(
        {
            "Gene": _join_unique_nonempty,
            "Protein class": _join_unique_nonempty,
            "Secretome location": _join_unique_nonempty,
            "Secretome function": _join_unique_nonempty,
            "Subcellular location": _join_unique_nonempty,
        }
    )
)
if not hpa_filter.index.is_unique:
    raise ValueError("HPA aggregation failed: UniProt index is still non-unique")

In [ ]:
uniprot_total = uniprot.copy()
uniprot_total['raw_text'] = uniprot_raw['raw_text']
uniprot_total.columns = 'uniprot_' + uniprot_total.columns
hpa_filter.columns = 'hpa_' + hpa_filter.columns
uniprot_total = pd.concat([uniprot_total, hpa_filter], axis=1)

### Merge into metadata

In [ ]:
out_rows = []
present_combs = []
for index, row in proteomics_analyte_metadata.iterrows():
    accessions = split_and_strip(row["uniprot_id"], val=';')
    comb = ';'.join(accessions)
    present_combs.append(comb)
    u = uniprot_total.reindex(accessions)
    if len(u) < 1:
        print("Warning: " + index)
    out = {
        'protein': row['protein'],
        'origin': 'proteomics',
        'uniprot_id': comb
    }
    for column in [
        "uniprot_Primary Gene Name",
        "uniprot_Protein names",
        "uniprot_Subcellular location [CC]",
        "uniprot_Keywords",
        "uniprot_Signal peptide",
        "uniprot_Transmembrane",
        "hpa_Protein class",
        "hpa_Secretome location",
        "hpa_Secretome function",
        "hpa_Subcellular location",
        "uniprot_raw_text"
        ]:
        out[column] = " || ".join(
            u[column].dropna().astype(str).unique()
        )
    out_rows.append(out)
    
for index, row in cytokine_info.iterrows():
    accessions = split_and_strip(row["uniprot"], val=';')
    comb = ';'.join(accessions)
    if comb in present_combs:
        continue
    u = uniprot_total.reindex(accessions)
    if len(u) < 1:
        print("Warning: " + index)
    out = {
        'protein': index,
        'origin': 'other_cytokines',
        'uniprot_id': comb
    }
    for column in [
        "uniprot_Primary Gene Name",
        "uniprot_Protein names",
        "uniprot_Subcellular location [CC]",
        "uniprot_Keywords",
        "uniprot_Signal peptide",
        "uniprot_Transmembrane",
        "hpa_Protein class",
        "hpa_Secretome location",
        "hpa_Secretome function",
        "hpa_Subcellular location",
        "uniprot_raw_text"
        ]:
        out[column] = " || ".join(
            u[column].dropna().astype(str).unique()
        )
    out_rows.append(out)
    
protein_metadata = pd.DataFrame(out_rows)

### LLM classification and evaluation

In [ ]:
ENV_PATH = os.path.join(os.getcwd(), '.env')
def load_env() -> None:
    with open(ENV_PATH) as f:
        for line in f:
            if not line.strip() or line.lstrip().startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
load_env()

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

ExtracellularOrigin = Literal[
    "classically_secreted",
    "surface_derived",
    "nonclassical_regulated_release",
    "surface_bound",
    "expected_intracellular",
    "gene_isoform_origin_ambiguous",
    "unclear",
]
CanonicalBiologicalRole = Literal[
    "secreted_ligand",
    "surface_ligand",
    "receptor_or_decoy",
    "enzyme",
    "matrix_or_carrier_protein",
    "other_non_signaling",
    "unclear",
]
OriginFlag = Literal[
    "secondary_isoform_uncertainty",
    "weak_predicted_signal_peptide_only",
    "extracellular_vs_intracellular_evidence_conflict",
    "propeptide_or_latency_associated",
    "zymogen_or_processed_enzyme",
    "storage_or_granule_release",
    "matrix_retained",
    "carrier_or_lipoprotein_associated",
    "lysosomal",
]

class CoarseBiologyClassification(BaseModel):
    protein: str = Field(
        description="The relevant protein (not the gene; copy verbatim).")
    gene_names: list[str] = Field(
        description="Associated human gene symbol or symbols.")
    extracellular_origin: ExtracellularOrigin = Field(
        description=("The most plausible extracellular-compartment behavior of the protein product represented by gene-level expression."))
    canonical_biological_role: CanonicalBiologicalRole = Field(
        description=("Canonical biological role of the protein."))
    origin_flags: list[OriginFlag] = Field(
        default_factory=list,
        description=("A small nonredundant set of mechanism or uncertainty flags. Return an empty list when none apply."),)
    confidence: Literal["high", "medium", "low"] = Field(
        description=("Confidence in the extracellular_origin assignment based only on the supplied biological evidence."))
    short_reason: str = Field(
        description=("A concise explanation of the classification and its most important uncertainty, when relevant."))

SYSTEM_PROMPT = """Classify each the given gene(s) and its/their protein product(s) according to its/their canonical extracellular biology.
Use the supplied evidence as far as possible.

EXTRACELLULAR ORIGIN
Choose exactly one extracellular_origin.

1. classically_secreted
Use when the relevant protein product enters the conventional ER/Golgi secretory pathway and is canonically released as an extracellular soluble protein.
Examples include conventionally secreted cytokines, chemokines, growth factors, enzymes, plasma proteins, extracellular structural proteins, and carrier
proteins. Normal maturation, signal-peptide cleavage, propeptide processing, glycosylation, oligomerization, regulated storage, degranulation, zymogen activation, or
post-secretion trimming does not change a protein from classically_secreted to another origin.

2. surface_derived
Use for a canonically cell-surface-associated gene when the supplied evidence establishes a released extracellular form. Qualifying mechanisms include:
- documented ectodomain shedding
- release of a GPI-anchored or membrane-associated protein
- a protein that is retained on the cell surface by presentation by another protein
- biologically established membrane-particle or extracellular-vesicle release.
For this schema, receptor/chaperone-mediated surface presentation of a soluble ligand counts as surface_derived.
Mere detection in an extracellular-vesicle proteomics dataset, plasma dataset, conditioned-medium dataset, or other extracellular sample is insufficient.

3. nonclassical_regulated_release
Use for a canonically intracellular or leaderless protein with established regulated extracellular release or a well-established functional extracellular role.
Examples include leaderless cytokines, alarmins, DAMPs, and other intracellular mediators whose extracellular release is recognized biology. 
Do not assign this origin merely because a protein:
- is found extracellularly in a proteomics dataset
- is present in extracellular vesicles
- can leak from dying cells
- is stored in a secretory granule.
A signal-peptide-containing protein released by regulated degranulation remains classically_secreted. Add storage_or_granule_release when appropriate.

4. surface_bound
Use when the canonical protein is membrane-bound, GPI-anchored, or otherwise cell-surface-associated, but the supplied evidence does not establish a released
soluble, shed, or biologically established membrane-particle form. A supernatant measurement does not by itself establish shedding or active release.
Do not infer release merely from:
- transmembrane topology
- an extracellular domain
- surface localization
- generic extracellular-vesicle detection

5. expected_intracellular
Use when there is no convincing intended extracellular, shed, or regulated nonclassical-release biology. This includes intracellular housekeeping, transcriptional,
structural, metabolic, and signaling proteins for which supernatant detection would most plausibly reflect:
- nonspecific leakage
- cell death
- debris
- incidental vesicle association
Also use this for proteins confined to intracellular organelles or intracellular membranes, including lysosomal, endosomal, mitochondrial, ER, and Golgi
proteins, unless the supplied evidence establishes extracellular release. Lysosomal localization alone does not establish nonclassical release.

6. gene_isoform_origin_ambiguous
Use only when the gene/protein has two or more well-supported, biologically substantial isoforms with fundamentally different compartment behavior.
A representative case is one major isoform is conventionally secreted and another comparably important isoform is entirely intracellular.
The following categories do *not* count as gene_isoform_origin_ambiguous:
- precursor versus mature protein
- membrane-bound versus shed protein
- soluble receptor versus membrane receptor
- proteolytic processing
Do not use this category merely because minor, rare, predicted, or poorly supported alternative isoforms exist. When one isoform is clearly canonical or dominant,
assign the origin of that isoform and add secondary_isoform_uncertainty only when the alternative isoform is still biologically relevant.

7. unclear
Use when the supplied evidence is insufficient or genuinely contradictory and none of the other origins can be assigned reliably.

GENERAL ORIGIN RULES
- Prefer a concrete extracellular_origin whenever the dominant biology is clear
- Do not infer intended extracellular biology from plasma, conditioned-medium, or extracellular-vesicle detection alone
- Do not use gene_isoform_origin_ambiguous as a generic uncertainty category
- Put uncertainty about the precise assay-detected form into origin_flags when the broad origin remains clear
- Do not let normal processing or maturation obscure a clear secretory origin.

MEMBRANE AND SOLUBLE FORMS
- Use surface_derived when a released form of a canonically surface-associated gene is explicitly supported
- Use surface_bound when surface topology is supported but release is not
- A surface ligand with a known shed form remains surface_derived for origin, while its canonical biological role may remain surface_ligand
- An independently secreted soluble isoform of a predominantly surface-associated gene is surface_derived under this coarse scheme
- Do not use gene_isoform_origin_ambiguous merely because membrane and released forms coexist.
Use it only when major isoforms imply fundamentally different gene-level localization expectations, such as
secreted versus entirely intracellular.

CANONICAL BIOLOGICAL ROLE
Choose exactly one canonical_biological_role.
secreted_ligand: The canonical signaling-active form is soluble and binds a receptor. Use for secreted cytokines, chemokines, growth factors, and comparable
soluble ligands.

surface_ligand: The canonical signaling-active form is membrane- or cell-surface-anchored, even if a released form also exists.
Do not use surface_ligand merely because a soluble ligand can bind extracellular matrix or a cell surface.

receptor_or_decoy: The protein is canonically a receptor, soluble receptor, shed receptor ectodomain, receptor-binding trap, or decoy receptor.
Solubility does not change a receptor into a secreted_ligand.

enzyme: The protein is canonically a catalytically active enzyme or enzyme precursor, including zymogens and proteolytically processed enzymes.

matrix_or_carrier_protein: The protein is canonically an extracellular-matrix protein, structural extracellular protein, soluble transport protein, carrier
protein, or lipoprotein-associated protein. Do not use this role for transmembrane transporters merely because they transport molecules.

other_non_signaling: Use for an extracellular or surface protein that does not fit the ligand, receptor, enzyme, matrix, carrier, or intracellular-marker
categories and is not primarily a signaling molecule.

unclear: Use when the supplied evidence is insufficient to assign a canonical biological role.

ORIGIN FLAGS
Return only flags that are directly supported and useful. Do not return duplicates.

secondary_isoform_uncertainty: Use when an alternative isoform with different localization or secretion behavior is biologically relevant, but one isoform is
still sufficiently canonical or dominant to permit a concrete extracellular_origin assignment. Do not use this flag when gene_isoform_origin_ambiguous is assigned.

weak_predicted_signal_peptide_only: Use when support for classical secretion rests primarily on a predicted signal peptide without corroborating secretion,
extracellular localization, or curated biological evidence.

extracellular_vs_intracellular_evidence_conflict: Use when substantial supplied evidence supports intended extracellular biology but also supports prominent
intracellular localization that creates a real classification concern. Do not use it merely because secreted proteins pass through ER/Golgi or are transiently stored intracellularly.

propeptide_or_latency_associated: Use when the protein is synthesized as a proprotein, latent complex, or propeptide-associated product and this is relevant to
interpretation. Normal signal-peptide removal alone is insufficient.

zymogen_or_processed_enzyme: Use for an enzyme produced as a zymogen or requiring biologically important proteolytic processing for activity.

storage_or_granule_release: Use only when supplied evidence explicitly supports storage in secretory granules, platelet granules, secretory lysosomes, or
another regulated degranulation compartment. Do not infer this merely because a protein is a cytokine or chemokine. This flag does not determine extracellular_origin.
A conventionally secreted granule-stored protein remains classically_secreted.

matrix_retained: Use when a conventionally secreted protein is normally retained in, bound to, or tightly associated with extracellular matrix or a cell-surface
matrix.

carrier_or_lipoprotein_associated: Use when extracellular occurrence or transport substantially depends on association with a carrier protein, carrier complex,
or lipoprotein particle.

lysosomal: Use when supplied evidence supports lysosomal localization. The lysosomal flag does not imply regulated extracellular release.

CONFIDENCE
Confidence refers only to extracellular_origin.
high: Use when curated topology, localization, and established biology clearly support one origin.
A surface_bound call may be high confidence when membrane topology and surface localization are well established.
A surface_derived call may be high confidence only when supplied evidence explicitly supports shedding, an independently secreted soluble isoform, or
biologically established membrane-particle release. Generic extracellular- vesicle detection is insufficient.

medium: Use when one origin is most plausible but is materially affected by unresolved isoform biology, molecular form, conflicting localization evidence, or
reliance on limited evidence.

low: Use when evidence is sparse, weak, or substantially contradictory. gene_isoform_origin_ambiguous may have high confidence when the existence and
importance of the fundamentally different isoforms are themselves well established.
"""

In [ ]:
UNIPROT_FIELDS = [
    ("UniProt Protein Names", "uniprot_Protein names"),
    ("UniProt Subcellular location [CC]", "uniprot_Subcellular location [CC]"),
    ("UniProt Keywords", "uniprot_Keywords"),
    ("UniProt Signal peptide", "uniprot_Signal peptide"),
    ("UniProt Transmembrane", "uniprot_Transmembrane"),
    ("HPA Gene", "hpa_Gene"),
    ("HPA Protein class", "hpa_Protein class"),
    ("HPA Secretome location", "hpa_Secretome location"),
    ("HPA Secretome function", "hpa_Secretome function"),
    ("HPA Subcellular location", "hpa_Subcellular location"),
]

def value_text(row: pd.Series, column: str) -> str:
    value = row.get(column, "")
    return "" if pd.isna(value) else str(value)

def split_accessions(accession_text: object) -> list[str]:
    if ',' in accession_text:
        sep = ','
    else:
        sep = ';'
    return [part.strip() for part in str(accession_text).split(sep) if part.strip()]

KEEP_CC_TOPICS = {
    "SUBCELLULAR LOCATION",
    "FUNCTION",
    "PTM",
    "ALTERNATIVE PRODUCTS",
    "TISSUE SPECIFICITY",
    "INDUCTION",
    "MISCELLANEOUS",
    "ACTIVITY REGULATION",
    "SUBUNIT",
    "CAUTION",
}

KEEP_FT_KEYS = {
    "SIGNAL",
    "TRANSMEM",
    "TOPO_DOM",
    "INTRAMEM",
    "PROPEP",
    "CHAIN",
    "PEPTIDE",
    "LIPID",
    "INIT_MET",
}

def extract_relevant_uniprot_evidence(raw_text: object) -> str:
    """Extract UniProt flat-file evidence useful for secretion/form buckets.

    Filtering is by UniProt section type only, not by biological keywords. This
    keeps retention deterministic and avoids hiding evidence just because a term
    was not anticipated. Output is normalized for LLM readability rather than
    preserving raw UniProt line prefixes.
    """
    if not isinstance(raw_text, str) or not raw_text.strip():
        return ""

    def clean_cc_block(topic: str, block: list[str]) -> str:
        cleaned_lines = []
        for j, block_line in enumerate(block):
            if j == 0:
                text = block_line[len("CC   -!- ") :]
                if ":" in text:
                    text = text.split(":", 1)[1]
            else:
                text = block_line[len("CC       ") :]
            cleaned_lines.append(text.strip())
        text = " ".join(part for part in cleaned_lines if part)
        return f"[CC: {topic}]\n{text}"

    def clean_ft_block(feature_key: str, block: list[str]) -> str:
        cleaned_lines = []
        for j, block_line in enumerate(block):
            if j == 0:
                text = block_line[len("FT   ") :].strip()
            else:
                text = block_line[len("FT                   ") :].strip()
            cleaned_lines.append(text)
        text = "; ".join(part for part in cleaned_lines if part)
        return f"[FT: {feature_key}]\n{text}"

    lines = raw_text.splitlines()
    kept_blocks: list[str] = []
    i = 0
    while i < len(lines):
        line = lines[i]

        if line.startswith("CC   -!- "):
            topic = line[len("CC   -!- ") :].split(":", 1)[0]
            block = [line]
            i += 1
            while i < len(lines) and lines[i].startswith("CC       "):
                block.append(lines[i])
                i += 1
            if topic in KEEP_CC_TOPICS:
                kept_blocks.append(clean_cc_block(topic, block))
            continue

        if line.startswith("FT   "):
            feature_key = line[5:13].strip()
            block = [line]
            i += 1
            while i < len(lines) and lines[i].startswith("FT                   "):
                block.append(lines[i])
                i += 1
            if feature_key in KEEP_FT_KEYS:
                kept_blocks.append(clean_ft_block(feature_key, block))
            continue

        i += 1

    return "\n\n".join(kept_blocks)

def make_message(panel_row: pd.Series) -> str:
    protein = panel_row['protein']
    gene = panel_row['uniprot_Primary Gene Name']
    accessions = split_accessions(panel_row["uniprot_id"])
    message = f"Classify the following gene:{gene} and protein:{protein}. Multi-component proteins are specified by genes separated by '||'. Evidence:\n\n"

    accession_blocks = []
    for accession in accessions:
        if accession not in uniprot_total.index:
            raise KeyError(f"Accession {accession} missing from uniprot_total for {protein}")
        row = uniprot_total.loc[accession]
        lines = [f"Accession: {accession}"]
        lines.extend(f"{label}: {value_text(row, column)}" for label, column in UNIPROT_FIELDS)
        lines.append("Raw UniProt flat-file text:")
        lines.append(extract_relevant_uniprot_evidence(value_text(row, "uniprot_raw_text")))
        accession_blocks.append("\n".join(lines))

    return message + "\n\n".join(accession_blocks)
    
panel_row = protein_metadata[protein_metadata['protein'] == 'IL12p70'].iloc[0]
make_message(panel_row=panel_row)

In [ ]:
ALL_PROTEINS = []
all_messages = []
for _, data in protein_metadata.iterrows():
    message = make_message(data)
    ALL_PROTEINS.append(data['protein'])
    all_messages.append(message)

In [ ]:
from pyCytED.llm_utils.llm import AsyncLLMWrapper
from pyCytED.llm_utils.llm_tasks import VeryBasicTask
from pyCytED.parsing import fix_df_after_loading

# get the llm
model_name = "openai/gpt-5.6-terra"
llm = AsyncLLMWrapper(
    model_name=model_name,
    rpm=1000,
)

# prepare the task
task = VeryBasicTask(
    in_messages=all_messages,
    prompt=SYSTEM_PROMPT,
    output_format=CoarseBiologyClassification,
    llm=llm,
    tmp_dir=os.path.join(output_folder, 'ProteinClassification'),
    name="ProteinClassification",
    rows_per_query=1
)

# run the task
kwargs = {
    "thinking": {"type": "medium"}
}
out_file = await task.run(
    batch_size=100,
    minimum_wait_time=30,
    multi_item_processing=False,
    **kwargs
)
out_full_text = fix_df_after_loading(pd.read_csv(out_file))
out_full_text['uniprot_ids'] = out_full_text['protein'].map(protein_metadata.set_index('protein')['uniprot_id'])
if not os.path.exists(out_file.replace('_outputs', '_outputs_with_meta')):
    out_full_text.to_excel(out_file.replace('_outputs.csv', '_outputs_with_meta.xlsx'), index=False)

In [ ]:
# This was manually reviewed and checked
protein_classification = pd.read_excel('../../annotations/ProteinClassification_outputs_with_meta_improved.xlsx')
protein_classification = protein_classification.set_index("protein_orig")
# sort on IDs
protein_classification['uniprot_ids'] = protein_classification['uniprot_ids'].astype(str).apply(
    lambda x: ';'.join(sorted(x.split(';'))))
# the IL-27 heterodimer is called 'EBI3;IL27' in the proteomics panel
protein_classification = protein_classification.rename(index={'IL-27': 'EBI3;IL27'})
protein_classification.loc['EBI3;IL27', 'protein'] = 'EBI3;IL27'

protein_classification['lysosomal'] = protein_classification['origin_flags'].str.contains('lysosomal')
mask = protein_classification['lysosomal'] & (protein_classification['extracellular_origin'] == 'expected_intracellular')
protein_classification.loc[mask, 'extracellular_origin'] = 'intracellular (lysosomal)'

### check the classification (hpa)

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

TARGET_DIR = Path("../../data/protein_classification/")
HPA_FILE = TARGET_DIR / "hpa_v21_predicted_secreted_proteins.tsv"

# does this use cached data?
if not os.path.exists(HPA_FILE):
    HPA_URL = "https://v21.proteinatlas.org/api/search_download.php?search=protein_class%3APredicted+secreted+proteins&columns=eg%2Cg%2Cgd&compress=no&format=tsv"
    urlretrieve(HPA_URL, HPA_FILE)
hpa_sec = pd.read_csv(HPA_FILE, sep="\t").rename(columns={"Ensembl": "ensembl_id", "Gene": "gene_symbol", "Gene description": "gene_description"})

In [ ]:
# had some issues with drifting standards
uniprot_pattern = r'^[A-Z0-9]+(;[A-Z0-9]+)*$'
for name, ids in [("cytokine_info", cytokine_info['uniprot']),
                  ("proteomics", proteomics_analyte_metadata['uniprot_id']),
                  ("classification", protein_classification['uniprot_ids'])]:
    bad = ids.dropna()[~ids.dropna().astype(str).str.match(uniprot_pattern)]
    assert bad.empty, f"{name}: non-standard UniProt IDs {bad.tolist()}"

In [ ]:
# merge
from ast import literal_eval

classification_genes = protein_classification.assign(gene_symbol=protein_classification["gene_names"].map(literal_eval)).explode("gene_symbol")
classification_genes["gene_symbol"] = classification_genes["gene_symbol"].str.strip().str.upper()
classification_genes = classification_genes.merge(hpa_sec, on="gene_symbol", how="left", validate="many_to_one", indicator=True)
classification_genes['_merge'] = classification_genes['_merge'].astype(str)

In [ ]:
value_counts = classification_genes.groupby('extracellular_origin')['_merge'].value_counts()
value_counts

### check the classification (SheddomeDB 2023)

Database: https://bal.lab.nycu.edu.tw/sheddomedb/
Publication: https://pubmed.ncbi.nlm.nih.gov/37458416/

In [ ]:
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path

# 1. Download SheddomeDB

sheddome_url = "https://bal.lab.nycu.edu.tw/sheddomedb/app/browse"
sheddome_base_url = "https://bal.lab.nycu.edu.tw/sheddomedb/app/protein_search"
sheddome_dir = Path("../../data/protein_classification/sheddomedb")
sheddome_dir.mkdir(parents=True, exist_ok=True)
sheddome_html = sheddome_dir / "sheddomedb_browse_snapshot.html"
sheddome_human_csv = sheddome_dir / "sheddomedb_human.csv"

# Set this to True deliberately if you want to replace the frozen snapshot.
force_refresh_flag = False
if force_refresh_flag or not sheddome_html.exists():
    response = requests.get(
        sheddome_url,
        headers={
            "User-Agent": (
                "Mozilla/5.0 (compatible; academic-research; "
                "SheddomeDB-comparison/1.0)"
            )
        },
        timeout=60,
    )
    response.raise_for_status()
    sheddome_html.write_text(response.text, encoding="utf-8")

html = sheddome_html.read_text(encoding="utf-8")
retrieved_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")

tables = pd.read_html(StringIO(html), match="UniProtKB ID")
if len(tables) != 1:
    raise RuntimeError(f"Expected exactly one SheddomeDB browse table; found {len(tables)}.")
sheddomedb = tables[0].copy()

# The website currently returns a two-level header.
if isinstance(sheddomedb.columns, pd.MultiIndex):
    sheddomedb.columns = sheddomedb.columns.get_level_values(-1)

sheddomedb = sheddomedb.rename(
    columns={
        "UniProtKB ID": "sheddomedb_uniprot",
        "Entry Name": "sheddomedb_entry",
        "Organism": "sheddomedb_organism",
        "Gene Name": "sheddomedb_gene",
    }
)

required_columns = {
    "sheddomedb_uniprot",
    "sheddomedb_entry",
    "sheddomedb_organism",
    "sheddomedb_gene",
}
missing_columns = required_columns - set(sheddomedb.columns)
if missing_columns:
    raise RuntimeError("The SheddomeDB page structure may have changed. " f"Missing columns: {sorted(missing_columns)}")


def canonical_uniprot(value):
    """Upper-case accession and remove an isoform suffix such as '-2'."""
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().upper()
    value = re.sub(r"-\d+$", "", value)
    return value or pd.NA


sheddomedb_human = sheddomedb.loc[sheddomedb["sheddomedb_organism"].astype(str).str.contains(r"Homo sapiens", case=False, na=False)].copy()
sheddomedb_human["sheddomedb_uniprot"] = sheddomedb_human["sheddomedb_uniprot"].map(canonical_uniprot)
sheddomedb_human["sheddomedb_url"] = sheddome_base_url + "?entry=" + sheddomedb_human["sheddomedb_uniprot"].astype(str)
sheddomedb_human["sheddomedb_snapshot_utc"] = retrieved_utc

# There should be one browse-table row per accession.
duplicate_accessions = sheddomedb_human.loc[sheddomedb_human["sheddomedb_uniprot"].duplicated(keep=False),"sheddomedb_uniprot",].dropna().unique()
if len(duplicate_accessions):
    raise RuntimeError("Unexpected duplicate human SheddomeDB accessions: " + ", ".join(duplicate_accessions[:10]))

sheddomedb_human.to_csv(sheddome_human_csv, index=False)

In [ ]:
sheddome_ids = set(sheddomedb_human["sheddomedb_uniprot"].dropna().str.upper())
classification_genes["sheddomedb_hit"] = classification_genes["uniprot_ids"].apply(lambda x: any(i.upper() in sheddome_ids for i in x.split(';')))

classification_genes["hpa_secreted"] = classification_genes["_merge"].eq("both")
classification_genes["uniprot_location"] = classification_genes["uniprot_ids"].apply(lambda x: ';'.join([str(uniprot.loc[i, "Subcellular location [CC]"]) for i in x.split(';')]))
classification_genes['signal_peptide'] = classification_genes["uniprot_ids"].apply(lambda x: ';'.join([str(uniprot.loc[i, "Signal peptide"]) for i in x.split(';')]))
classification_genes["uniprot_secreted"] = classification_genes["uniprot_location"].str.contains(r"(?i)\bSecreted\b")
classification_genes['uniprot_cell_surface'] = classification_genes["uniprot_location"].str.lower().str.contains('cell membrane')
classification_genes['transmembrane'] = classification_genes["uniprot_ids"].apply(lambda x: ';'.join([str(uniprot.loc[i, "Transmembrane"]) for i in x.split(';')]))
classification_genes["multi_component"] = classification_genes["uniprot_ids"].str.contains(";")

In [ ]:
# gather more evidence where it matters
from concurrent.futures import ThreadPoolExecutor
from io import StringIO

if not os.path.exists(sheddome_dir / "sheddome_detailed_surface_evidence.xlsx"):
    surface_ids = list(classification_genes.loc[
        classification_genes["extracellular_origin"].isin(["surface_bound", "surface_derived"]), "uniprot_ids"
    ].values)
    surface_ids = set([j for i in surface_ids for j in i.split(';')])
    review_ids = sorted(surface_ids & sheddome_ids)

    def get_shedding_evidence(acc):
        r=requests.get(f"https://bal.lab.nycu.edu.tw/sheddomedb/app/protein_search?entry={acc}",timeout=60); r.raise_for_status()
        records=[]
        for t in pd.read_html(StringIO(r.text)):
            if t.shape[1]>=3 and t.iloc[:,1].astype(str).str.contains("Short Summary of Papers",case=False).any():
                records.append(dict(zip(t.iloc[:,1].astype(str).str.strip(),t.iloc[:,2].astype(str).str.strip())))
        join=lambda field:" | ".join(dict.fromkeys(d.get(field, "") for d in records if d.get(field, "") not in ("", "nan")))
        return {"uniprot_id":acc, "substrate":join("Substrate Name"), "sheddase":join("Sheddase"), "summary":join("Short Summary of Papers"), "pmid":join("PMID")}

    with ThreadPoolExecutor(max_workers=6) as pool:
        sheddomedb_evidence=pd.DataFrame(pool.map(get_shedding_evidence,review_ids))
    sheddomedb_evidence.to_excel(sheddome_dir / "sheddome_detailed_surface_evidence.xlsx", index=False)
else:
    sheddomedb_evidence = pd.read_excel(sheddome_dir / "sheddome_detailed_surface_evidence.xlsx")
sheddomedb_evidence = sheddomedb_evidence.set_index('uniprot_id')

### evaluate evidence from both

In [ ]:
# expected intracellular but has some secretion evidence
subset = classification_genes[
    (classification_genes['extracellular_origin'].isin(['expected_intracellular', 'intracellular (lysosomal)'])) &
    (classification_genes['hpa_secreted'] | classification_genes['uniprot_secreted'] | classification_genes['sheddomedb_hit'])
    ]

subset[['protein', 'extracellular_origin', 'hpa_secreted', 'uniprot_secreted', 'sheddomedb_hit', 'signal_peptide']]

# SORD -> expected_intracellular (false positive)

In [ ]:
# expected intracellular but has some signal peptide
subset = classification_genes[
    (classification_genes['extracellular_origin'].isin(['expected_intracellular', 'intracellular (lysosomal)'])) &
    (classification_genes['signal_peptide'].apply(lambda x: not(all([i == '<NA>' for i in x.split(';')]))))
    ]

subset[['protein', 'extracellular_origin', 'hpa_secreted', 'uniprot_secreted', 'sheddomedb_hit', 'signal_peptide']]
subset['protein'].values

# these all traffic to intracellular compartments

In [ ]:
# expected intracellular but has a transmembrane domain
subset = classification_genes[
    classification_genes['extracellular_origin'].isin(['expected_intracellular', 'intracellular (lysosomal)']) &
    (classification_genes['transmembrane'].apply(lambda x: not(all([i == '<NA>' for i in x.split(';')]))))
    ]

subset[['protein', 'extracellular_origin', 'hpa_secreted', 'uniprot_secreted', 'sheddomedb_hit', 'signal_peptide']]
subset['protein'].values

# these are all fine, basically, quite clearly intracellular

In [ ]:
# classically_secreted with some conflicting evidence
subset = classification_genes[
    classification_genes['extracellular_origin'].isin(['classically_secreted']) &
    (~classification_genes['hpa_secreted'] | ~classification_genes['uniprot_secreted'] | classification_genes['sheddomedb_hit'])
    ]

subset[['protein', 'hpa_secreted', 'uniprot_secreted', 'sheddomedb_hit', 'signal_peptide']]

# These look okay. Some notes:
# AGR2 / ERAP1 - can be ER-retained
# ACHE - isoform ambiguity to some degree
# IFNL4 - ususally inefficient / conditional
# TFPI - isoform ambiguity to some degree

In [ ]:
# classically_secreted with transmembrane
subset = classification_genes[
    classification_genes['extracellular_origin'].isin(['classically_secreted']) &
    (classification_genes['transmembrane'].apply(lambda x: not(all([i == '<NA>' for i in x.split(';')]))))
    ]

subset[['protein', 'hpa_secreted', 'uniprot_secreted', 'sheddomedb_hit', 'signal_peptide', 'transmembrane']]
# ACP3 has a membrane-bound isoform
# complement protein transmembrane refers to MAC insertion, not constitutive transmembrane
# IG signal peptide missing is an annotation artifact with respect to the rearranged antibody chain

In [ ]:
# classically_secreted but no signal peptide
subset = classification_genes[
    classification_genes['extracellular_origin'].isin(['classically_secreted']) &
    (classification_genes['signal_peptide'].apply(lambda x: all([i == '<NA>' for i in x.split(';')])))
    ]

subset[['protein', 'hpa_secreted', 'uniprot_secreted', 'sheddomedb_hit', 'signal_peptide', 'transmembrane', 'signal_peptide']]
# VEGFA is complicated, but classically secreted
# FGF9 is mechanistically atypical but classically secreted
# APRIL is cleaved intracellularly, fine as classically secreted
# IG signal peptide missing is an annotation artifact with respect to the rearranged antibody chain

In [ ]:
# nonclassical relese
subset = classification_genes[
    classification_genes['extracellular_origin'].isin(['nonclassical_regulated_release']) &
    (~classification_genes['hpa_secreted'] | ~classification_genes['uniprot_secreted'] | classification_genes['sheddomedb_hit'])
    ]

subset[['protein', 'hpa_secreted', 'uniprot_secreted', 'sheddomedb_hit', 'signal_peptide']]

# all of these are *probably* genuine nonclassically secreted proteins, though perhaps with moderate confidence (biology not always 100% clear) - calls are okay

In [ ]:
# nonclassical relese
subset = classification_genes[
    classification_genes['extracellular_origin'].isin(['nonclassical_regulated_release']) &
    (classification_genes['transmembrane'].apply(lambda x: not(all([i == '<NA>' for i in x.split(';')]))))
    ]

subset[['protein', 'hpa_secreted', 'uniprot_secreted', 'sheddomedb_hit', 'signal_peptide', 'transmembrane']]
# ERAP1 can be secreted (https://pubmed.ncbi.nlm.nih.gov/32009107/)

In [ ]:
# surface-bound / released but no transmembrane?
subset = classification_genes[
    classification_genes['extracellular_origin'].isin(['surface_bound', 'surface_derived']) &
    (
        (classification_genes['transmembrane'].apply(lambda x: all([i == '<NA>' for i in x.split(';')])))
    )
    ]

subset[['protein', 'hpa_secreted', 'uniprot_secreted', 'sheddomedb_hit', 'signal_peptide', 'transmembrane']]
subset['protein'].values

# GPI-anchored:
# BST1, CA4, CD14, CD48, CD59, CEACAM5, CEACAM6, CNTFR, CNTN1, CNTN2, CRIPTO, FCGR3B, FOLR1, FOLR2, GAS1, GPC3, GPC5, HJV, MSLN, OPCML, PLAUR, RGMA, RGMB, RTN4R, SEMA7A, THY1, ULBP1, ULBP2, ULBP3
# AGRN, ITLN1, IL15 are cell surface retained by other proteins
# ROBO4 and SPINT1 is a misannotation in uniprot
# NPHS2 could be called as expected_intracellular, let's keep it here

In [ ]:
# surface-bound but secreted?
subset = classification_genes[
    classification_genes['extracellular_origin'].isin(['surface_bound']) &
    (classification_genes['hpa_secreted'] | classification_genes['uniprot_secreted'])
    ]

subset[['protein', 'uniprot_ids', 'extracellular_origin', 'hpa_secreted', 'uniprot_secreted', 'sheddomedb_hit', 'signal_peptide']].drop_duplicates()
# LT-alpha1-beta2 and LT-alpha2-beta are both surface bound - the calls are for the LTA subdomain
# ANTXR2 - extracellular product is only computationally predicted
# IL17RB - no convincing evidence of actual release
# TMEM119 - extracellular detection but no convincing secretion or shedding

In [ ]:
# surface-derived but not secreted
subset = classification_genes[
    classification_genes['extracellular_origin'].isin(['surface_derived']) &
    (~classification_genes['hpa_secreted'] & ~classification_genes['uniprot_secreted'] & ~classification_genes['sheddomedb_hit'])
    ]

subset[['protein', 'uniprot_ids', 'extracellular_origin', 'hpa_secreted', 'uniprot_secreted', 'sheddomedb_hit', 'signal_peptide']].drop_duplicates()

# DLL1 - clear ectodomain shedding evidence
# PDPN - some evidence but also EV-associated
# CD80 - soluble receptor biology (alternative splicing)
# IFNAR1 - soluble receptor biology (alternative splicing) | maybe? could also be surface_bound
# IFNLR1 - soluble receptor biology (alternative splicing)

In [ ]:
subset = classification_genes[
    (classification_genes['extracellular_origin'].isin(['surface_bound']) & (classification_genes['sheddomedb_hit'])) |
    (classification_genes['extracellular_origin'].isin(['surface_derived']) & (~classification_genes['sheddomedb_hit']))
    ]

subset[['protein', 'uniprot_ids', 'extracellular_origin', 'hpa_secreted', 'uniprot_secreted', 'sheddomedb_hit', 'signal_peptide']].drop_duplicates()
# sort of a complicated, long list - how to adjudicate?
# -> gather more evidence

### Use LLM adjudication of surface_bound vs surface_derived distinction

In [ ]:
# add important evidence to surface proteins:
# uniprot - structured uniprot fields
# uniprot_raw - full raw uniprot text
# sheddomedb_evidence - content of sheddomedb for entries where that exists

surface_proteins = classification_genes[
    classification_genes["extracellular_origin"].isin(["surface_bound", "surface_derived"])
    ].drop_duplicates("protein").copy()

def join_sheddome_evidence(uniprot_ids, field):
    values = [str(sheddomedb_evidence[field].get(i, "")).strip() for i in uniprot_ids.split(";")]
    return " ; ".join(dict.fromkeys(v for v in values if v not in ("", "nan")))

for field in ["substrate", "sheddase", "summary", "pmid"]:
    surface_proteins[f"sheddome_{field}"] = surface_proteins["uniprot_ids"].apply(
        lambda x: join_sheddome_evidence(x, field)
    )

In [ ]:
def search_aliases(row):
    values = [row["gene_symbol"], *re.split(r"[;,]", row["protein"])]
    for acc in row["uniprot_ids"].split(";"):
        if acc not in uniprot.index: continue
        if pd.notna(uniprot.at[acc, "Gene Names"]): values.extend(str(uniprot.at[acc, "Gene Names"]).split())
        if pd.notna(uniprot.at[acc, "Protein names"]): values.append(str(uniprot.at[acc, "Protein names"]).split(" (")[0])
    aliases = []
    for value in values:
        value = re.sub(r'[\[\]():"]', " ", str(value)).strip()
        technical = re.fullmatch(r"(?:UNQ|PRO|PSEC|GP|ICD)\d+(?:/\w+)?", value, re.I)
        if 2 < len(value) <= 55 and not technical and value.lower() not in {x.lower() for x in aliases}: aliases.append(value)
    return aliases[:8]

surface_proteins["search_aliases"] = surface_proteins.apply(search_aliases, axis=1)

In [ ]:
pmc_search_dir = Path("../../data/protein_classification/pmc_search")
pmc_search_dir.mkdir(parents=True, exist_ok=True)
pmc_cache = pmc_search_dir / "europe_pmc_cache.json"
max_papers = 4

cache = json.loads(pmc_cache.read_text()) if pmc_cache.exists() else {}

def clean_search_term(value):
    return re.sub(r'[\[\]():"]', " ", str(value)).strip()

def relation_query(aliases, kind):
    templates = (
        ['{x} shedding', 'shedding of {x}', '{x} ectodomain', 'ectodomain of {x}', 'release of {x}', 'released {x}']
        if kind == "mechanism" else
        ['soluble {x}', '{x} soluble form', '{x} soluble isoform']
    )
    return "(" + " OR ".join(f'TITLE_ABS:"{template.format(x=alias)}"' for alias in aliases for template in templates) + ")"

def format_pubmed_record(record):
    pmid = str(record.get("pmid") or "")
    abstract = record.get("abstractText", "")
    if not re.fullmatch(r"\d{7,9}", pmid) or not abstract:
        return None
    return {
        "pmid": pmid,
        "title": record.get("title", ""),
        "year": record.get("pubYear", ""),
        "types": record.get("pubTypeList", {}).get("pubType", []),
        "abstract": abstract[:2500],
    }

def get_epmc_results(query, page_size):
    response = requests.get(
        "https://www.ebi.ac.uk/europepmc/webservices/rest/search",
        params={
            "query": query,
            "format": "json",
            "resultType": "core",
            "pageSize": page_size,
        },
        timeout=60,
    )
    response.raise_for_status()
    return response.json().get("resultList", {}).get("result", [])

def format_records(records, source, exclude=(), drop_reviews=False):
    papers, seen = [], set(exclude)
    for record in records:
        paper = format_pubmed_record(record)
        types = record.get("pubTypeList", {}).get("pubType", [])
        if paper is None or paper["pmid"] in seen or (drop_reviews and any("review" in str(t).lower() for t in types)): continue
        papers.append({**paper, "source": source}); seen.add(paper["pmid"])
        if len(papers) == max_papers: break
    return papers

def search_europe_pmc(row):
    aliases = row["search_aliases"]
    queries = {"mechanism": relation_query(aliases, "mechanism"), "soluble": relation_query(aliases, "soluble")}
    seed_pmids = list(dict.fromkeys(re.findall(r"\b\d{7,9}\b", row["sheddome_pmid"])))[:12]
    cached = cache.get(row["protein"])
    if cached and cached.get("queries") == queries and cached.get("seed_pmids") == seed_pmids: return cached

    seed_records = get_epmc_results(" OR ".join(f"EXT_ID:{p}" for p in seed_pmids), len(seed_pmids)) if seed_pmids else []
    mechanism_records = get_epmc_results(queries["mechanism"], max_papers * 3)
    soluble_records = get_epmc_results(queries["soluble"], max_papers * 3)

    seed_papers = format_records(seed_records, "sheddome")
    mechanism_papers = format_records(mechanism_records, "mechanism", {p["pmid"] for p in seed_papers}, True)
    used = {p["pmid"] for p in seed_papers + mechanism_papers}
    soluble_papers = format_records(soluble_records, "soluble", used, True)

    result = {"aliases": aliases, "queries": queries, "seed_pmids": seed_pmids, "seed_papers": seed_papers, "mechanism_papers": mechanism_papers, "soluble_papers": soluble_papers}
    cache[row["protein"]] = result
    return result

with ThreadPoolExecutor(max_workers=4) as pool:
    epmc_results = list(pool.map(search_europe_pmc, surface_proteins.to_dict("records")))

for field in ["seed_papers", "mechanism_papers", "soluble_papers"]:
    surface_proteins[f"epmc_{field}"] = [result[field] for result in epmc_results]

pmc_cache.write_text(json.dumps(cache, indent=2))

In [ ]:
class SurfaceReleaseCall(BaseModel):
    protein: str
    extracellular_origin: Literal["surface_bound", "surface_derived", "other"]
    supporting_pmids: list[str]
    reason: str

SURFACE_CLASSIFICATION_PROMPT = """
We are classifying proteins detected by an extracellular antibody-based assay. Every protein in this task has already been identified as normally associated with the cell surface. 
The question is whether extracellular assay signal could plausibly come from an endogenous extracellular version of that surface protein.

Use these definitions:

surface_derived:
The supplied evidence shows that the endogenous subject protein produces a substantial extracellular product from a surface-associated form.
This can occur through ectodomain shedding, release of a GPI-anchored protein, established release of the surface protein on a shed membrane particle,
or a biologically established soluble splice isoform. Examples of the intended biology include soluble TNF, RANKL and CD40L.

surface_bound:
The supplied evidence does not establish such an extracellular product. Use this as the default when the evidence is absent, indirect, ambiguous or contradictory.

other:
The protein is not surface-associated at all - its initial classification should be revisited,

Important exclusions:
- A recombinant protein, Fc fusion, or experimentally engineered soluble construct is not evidence of endogenous release.
- Detection in plasma, extracellular vesicles or conditioned medium is not sufficient unless the evidence identifies the origin of the extracellular protein.
- The subject protein regulating or causing release of another protein is not evidence that the subject is released.
- Release of an intracellular domain, a small activation peptide or an unspecified cleavage product is not sufficient.
- A paper returned by the mechanism search may still concern another molecule, drug release or an unrelated use of the word release.
- A paper returned by the soluble search may concern an engineered reagent rather than a natural soluble protein.
- A SheddomeDB entry is a lead, not automatically proof.
- Do not use biological facts that are absent from the supplied material.

For surface_derived, the reason must state what extracellular product is produced and how it arises. Cite at least one supporting PMID from SUPPLIED PMIDS.
For surface_bound, use an empty supporting_pmids list and explain briefly why the supplied evidence is insufficient.
Return the protein identifier exactly as supplied. Keep the reason to one or two sentences.
"""

def clean_evidence_text(x):
    return re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", str(x))).strip()

def make_surface_message(row):
    papers = row["epmc_seed_papers"] + row["epmc_mechanism_papers"] + row["epmc_soluble_papers"]
    supplied_pmids = list(dict.fromkeys(p["pmid"] for p in papers))
    source_names = {"sheddome": "SheddomeDB-linked paper", "mechanism": "mechanism search", "soluble": "soluble-product search"}
    literature = "\n\n".join(
        f"SOURCE: {source_names[p['source']]}\nPMID: {p['pmid']}\nYEAR: {p['year']}\nTYPE: {', '.join(p['types'])}\nTITLE: {clean_evidence_text(p['title'])}\nABSTRACT: {clean_evidence_text(p['abstract'])}"
        for p in papers
    )
    return (
        f"PROTEIN: {row['protein']}\n"
        f"GENE: {row['gene_symbol']}\n"
        f"UNIPROT: {row['uniprot_ids']}\n\n"
        f"SHEDDOME DATABASE LEAD\n"
        f"Substrate: {row['sheddome_substrate'] or 'none'}\n"
        f"Reported sheddase: {row['sheddome_sheddase'] or 'none'}\n"
        f"Summary: {row['sheddome_summary'] or 'none'}\n\n"
        f"SUPPLIED PMIDS: {', '.join(supplied_pmids) or 'none'}\n\n"
        f"LITERATURE\n{literature or 'No literature abstract was retrieved.'}"
    )

surface_proteins["evidence_papers"] = surface_proteins.apply(
    lambda row: row["epmc_seed_papers"] + row["epmc_mechanism_papers"] + row["epmc_soluble_papers"], axis=1
)
surface_proteins["supplied_pmids"] = surface_proteins["evidence_papers"].apply(
    lambda papers: list(dict.fromkeys(p["pmid"] for p in papers))
)
surface_proteins["evidence_packet"] = surface_proteins.apply(make_surface_message, axis=1)
all_messages = surface_proteins["evidence_packet"].tolist()

In [ ]:
import logging
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

from pyCytED.llm_utils.llm import AsyncLLMWrapper
from pyCytED.llm_utils.llm_tasks import VeryBasicTask
from pyCytED.parsing import fix_df_after_loading

# get the llm
model_name = "openai/gpt-5.6-terra"
llm = AsyncLLMWrapper(
    model_name=model_name,
    rpm=1000,
)

# prepare the task
task = VeryBasicTask(
    in_messages=all_messages,
    prompt=SURFACE_CLASSIFICATION_PROMPT,
    output_format=SurfaceReleaseCall,
    llm=llm,
    tmp_dir=os.path.join(output_folder, 'SurfaceClassification'),
    name="SurfaceClassification",
    rows_per_query=1
)

# run the task
kwargs = {
    "thinking": {"type": "medium"}
}
out_file = await task.run(
    batch_size=100,
    minimum_wait_time=1,
    multi_item_processing=False,
    **kwargs
)
out_full_text = fix_df_after_loading(pd.read_csv(out_file))

In [ ]:
out_full_text = out_full_text.set_index('protein').add_suffix('_revisited').reset_index()

In [ ]:
merged = pd.merge(surface_proteins, out_full_text, on='protein')

In [ ]:
merged[merged['extracellular_origin'] != merged['extracellular_origin_revisited']][['protein', 'extracellular_origin', 'extracellular_origin_revisited', 'reason_revisited']]

### Tons of disagreement here.
### Whether or not there are released surface forms is really rather subtle biology, and lots of the distinctions seem to be relatively nuanced...
### This is all interesting *in principle* but we're dropping this distinction for the main paper since that is really not what the publications is about.

In [ ]:
relabel = {
    'surface_bound': 'surface_associated',
    'surface_derived': 'surface_associated'
}
classification_genes['extracellular_origin'] = classification_genes['extracellular_origin'].apply(lambda x: relabel.get(x, x))
protein_classification['extracellular_origin'] = protein_classification['extracellular_origin'].apply(lambda x: relabel.get(x, x))

## 2.2 - Basic proteomics check

In [ ]:
pd.set_option('future.no_silent_downcasting', True)

donors = SHARED_DONORS
conditions = [
    (("PBS", "IL-15"), "IL15"),
    (("PBS", "TNF-alpha"), "TNF"),
    (("PBS", "IFN-beta"), "IFNB1"),
    (("PBS", "IL-32-beta"), "IL32"),
    (("PBS", "IL-18"), "IL18"),
    (("PBS", "IL-6"), "IL6"),
    (("PBS", "C3a"), "C3"),
    (("PBS", "IL-1Ra"), "IL1RN")
]

fig, axes = plt.subplots(nrows=1, ncols=len(conditions), figsize=cm_to_inches((20, 3)))
for i, condition in enumerate(conditions):
    ax = axes[i]
    treatments, label = condition
    sub = proteomics_values_raw.loc[(treatments, donors), :].replace('Low bead count', np.nan).astype(float)[label].to_frame().reset_index()
    sub[label] = sub[label].astype(float)
    sns.swarmplot(data=sub, x="treatment", y=label, ax=ax, color='black', s=3)
    ax.set_title(label)
    if i == 0:
        ax.set_ylabel('ng/ml')
    else:
        ax.set_ylabel('')
    ax.set_xlabel('')
    
    ulod = proteomics_analyte_metadata.loc[proteomics_analyte_metadata['protein'] == label, 'upper_limit_of_detection'].values[0]
    llod = proteomics_analyte_metadata.loc[proteomics_analyte_metadata['protein'] == label, 'lower_limit_of_detection'].values[0]
    max_val = sub[label].max()
    if ulod < 1.5 * max_val:
        ax.axhline(ulod, color='red', linestyle='--', label='ULOD')
    ax.axhline(llod, color='blue', linestyle='--', label='LLOD')
    if i == 0:
        ax.legend()
    ax.tick_params(length=0)
    apply_greek_labels(ax)

plt.subplots_adjust(wspace=0.4, hspace=0.3)
plt.savefig(os.path.join(current_plot_folder, "sanity_protein_ontarget.svg"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Do the antibodies properly inhibit detection of the protein?

from scipy.stats import ttest_rel
pd.set_option('future.no_silent_downcasting', True)

donors = SHARED_DONORS
conditions = [
    (("PBS", "IL-15", "IL-15_AB-IFN-gamma"), "IFNG"),
    (("PBS", "IL-15", "IL-15_AB-IL-6"), "IL6"),
    (("PBS", "IL-15", "IL-15_AB-TNF-alpha"), "TNF"),
    (("PBS", "IL-32-beta", "IL-32-beta_IL-1Ra"), "IL1B"),
    (("PBS", "IL-32-beta", "IL-32-beta_AB-IL-10"), "IL10"),
    (("PBS", "IL-32-beta", "IL-32-beta_AB-IL-6"), "IL6"),
]

fig, axes = plt.subplots(nrows=1, ncols=len(conditions), figsize=cm_to_inches((20, 3)))
for i, condition in enumerate(conditions):
    ax = axes[i]
    treatments, label = condition
    sub = proteomics_values_raw.loc[(treatments, donors), :].astype(float)[label].to_frame().reset_index()
    sub[label] = sub[label].astype(float)
    sub['treatment_alt'] = sub['treatment'].apply(lambda x: format_treatment(x, connection='\n+\n'))
    sns.swarmplot(data=sub, x="treatment_alt", y=label, ax=ax, s=3, color='black')
    ax.set_title(label)
    if i == 0:
        ax.set_ylabel('ng/ml')
    else:
        ax.set_ylabel('')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)

    apply_greek_labels(ax)
    ulod = proteomics_analyte_metadata.loc[proteomics_analyte_metadata['protein'] == label, 'upper_limit_of_detection'].values[0]
    llod = proteomics_analyte_metadata.loc[proteomics_analyte_metadata['protein'] == label, 'lower_limit_of_detection'].values[0]
    max_val = sub[label].max()
    if ulod < 1.5 * max_val:
        ax.axhline(ulod, color='red', linestyle='--', label='ULOD')
    ax.axhline(llod, color='blue', linestyle='--', label='LLOD')
    if i == 4:
        ax.legend()
    ax.tick_params(length=0)
    apply_greek_labels(ax)

    paired = np.log2(sub.pivot(columns='treatment', index='donor', values=label))
    result = ttest_rel(
        paired[treatments[1]],
        paired[treatments[2]],
    )
    ax.plot([1, 2], [1.2*max_val, 1.2*max_val], color='black')
    ax.text(1.5, 1.275*max_val, f"p={result.pvalue:.3g}", ha='center')
    ax.set_ylim(-0.1*max_val, 1.4*max_val)

    
plt.subplots_adjust(wspace=0.3, hspace=0.3)
plt.savefig(os.path.join(current_plot_folder, "sanity_antibodies.svg"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Do we see a couple of expected protein-level responses?
from scipy.stats import ttest_rel
pd.set_option('future.no_silent_downcasting', True)

donors = SHARED_DONORS
conditions = [
    (("PBS", "IL-18"), "IFNG"),
    (("PBS", "TNF-alpha"), "CXCL5"),
    (("PBS", "IL-32-beta"), "IL10"),
]

fig, axes = plt.subplots(nrows=1, ncols=len(conditions), figsize=cm_to_inches((6, 1.5)), sharey=False)
for i, condition in enumerate(conditions):
    ax = axes[i]
    treatments, label = condition
    available_donors = proteomics_values_raw.loc[treatments[1]].index
    sub = proteomics_values_raw.loc[(treatments, available_donors), :].astype(float)[label].to_frame().reset_index()
    sub[label] = np.log10(sub[label].astype(float))
    sub['treatment_alt'] = sub['treatment'].apply(lambda x: format_treatment(x, connection='\n+\n'))
    sns.swarmplot(data=sub, x="treatment_alt", y=label, ax=ax, s=3, color='black')
    ax.set_title(label, y=1.07)
    if i == 0:
        ax.set_ylabel('log10\n(meas. (ng/ml))')
    else:
        ax.set_ylabel('')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)

    apply_greek_labels(ax)
    ulod = np.log10(proteomics_analyte_metadata.loc[proteomics_analyte_metadata['protein'] == label, 'upper_limit_of_detection'].values[0])
    llod = np.log10(proteomics_analyte_metadata.loc[proteomics_analyte_metadata['protein'] == label, 'lower_limit_of_detection'].values[0])
    max_val = sub[label].max()
    min_val = sub[label].min()
    if ulod < 0.2 + max_val:
        ax.axhline(ulod, color='tab:gray', linestyle='--', label='ULOD')
    if llod > min_val - 0.2:
        ax.axhline(llod, color='tab:gray', linestyle='--', label='LLOD')  
    if i == 4:
        ax.legend()
    ax.tick_params(length=0)
    apply_greek_labels(ax)

    paired = sub.pivot(columns='treatment', index='donor', values=label)
    result = ttest_rel(
        paired[treatments[0]],
        paired[treatments[1]],
    )
    #ax.plot([0, 1], [1.2*max_val, 1.2*max_val], color='black', lw=0.5)
    ax.text(0.5, max_val+0.35, f"p={result.pvalue:.2g}", ha='center')
    ax.set_xlim(-0.5,1.5)
    ax.set_ylim(min_val-0.2, max_val+0.2)
    
plt.subplots_adjust(wspace=0.3, hspace=0.3)
plt.savefig(os.path.join(current_plot_folder, "canonical_responses.svg"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from matplotlib.ticker import MaxNLocator
locator = MaxNLocator(nbins=1)

pd.set_option('future.no_silent_downcasting', True)
cytokine_conditions = ["IFNA1", "IFNA2", "IFNB1", "IFNE", "IFNG", "IFNL1", "IFNL2", "IFNL3", "IFNW1"] + \
    ["IL1A", "IL1B", "IL18", "TNF", "IL6", "IL12p70", "EBI3;IL27"]
sub = proteomics_values_raw.loc["PBS"].astype(float)[cytokine_conditions]
sub = sub.loc[DONOR_ORDER_IFN_GROUPED]
sub = sub.reset_index().melt(id_vars='donor')
sub['donor_type'] = sub['donor'].map(DONOR_TO_GROUP)

fig, axes = plt.subplots(nrows=1, ncols=len(cytokine_conditions), figsize=cm_to_inches((12, 4)), gridspec_kw={"wspace": 0.015})

p_values = []
p_label_positions = []
for i, protein in enumerate(cytokine_conditions):
    ax = axes[i]

    # vestigial, but keep for convenience
    effective_protein = protein
    
    subsub = sub[sub["variable"] == effective_protein].sort_values('value', ascending=False)
    values = subsub["value"].dropna()

    metadata = (proteomics_analyte_metadata.loc[lambda x: x["protein"].eq(effective_protein)].iloc[0])
    llod = float(metadata["lower_limit_of_detection"])
    ulod = float(metadata["upper_limit_of_detection"])
    cytokine = introduce_greek_letters(metadata['cytokine'])

    # ---- range calculation ----
    data_min, data_max = values.min(), values.max()
    near_llod = np.isfinite(llod) and llod > 0 and data_min <= 2 * llod
    near_ulod = np.isfinite(ulod) and data_max >= 0.5 * ulod
    all_near_llod = near_llod and data_max <= 2 * llod
    if all_near_llod:
        # Keep essentially undetected values visually near the bottom.
        raw_min = max(0, llod * 0.8)
        raw_max = max(data_max * 1.15, llod * 3)
    else:
        raw_min = min(data_min, llod) if near_llod else data_min
        raw_max = max(data_max, ulod) if near_ulod else data_max
        span = raw_max - raw_min
        if span == 0:
            span = max(abs(raw_max), 1) * 0.1
        raw_min = max(0, raw_min - 0.08 * span)
        raw_max = raw_max + 0.12 * span
    ticks = MaxNLocator(nbins=3).tick_values(raw_min, raw_max)
    ymin = max(0, ticks[0])
    ymax = ticks[-1]
    # --------------------------

    sns.stripplot(data=subsub, y="value", hue="donor_type", palette=DONOR_COLOR_SCHEME, jitter=0.18, size=3, legend=False, ax=ax, zorder=3)
    
    # Detection-limit indicators
    if near_llod:
        ax.axhline(llod, linestyle=":", linewidth=0.6, color="0.55", zorder=1)
    if near_ulod:
        ax.axhline(ulod, linestyle=":", linewidth=0.6, color="0.55", zorder=1)

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    ax.set_yticks([])

    # Range labels
    plot_ymin, plot_ymax = ymin, ymax
    plot_span = plot_ymax - plot_ymin

    # Extra space inside the spines, outside the dot region
    label_space = 0.25 * plot_span
    ax.set_ylim(plot_ymin - label_space, plot_ymax + label_space)
    ax.set_xlim(-0.48, 0.48)

    # Short horizontal ticks marking the actual plotted range
    ax.hlines(plot_ymax, -0.18, 0.18, color="black", linewidth=0.7, zorder=4)
    ax.hlines(plot_ymin, -0.18, 0.18, color="black", linewidth=0.7, zorder=4)

    def compact_number(x):
        if x == 0:
            return "0"
        if abs(x) < 1e-2 or abs(x) >= 1e4:
            return f"{x:.1e}".replace("e-0", "e-").replace("e+0", "e+")
        return f"{x:g}"
    ax.text(0, plot_ymax + 0.35 * label_space, compact_number(plot_ymax),
            ha="center", va="bottom", fontsize=6, zorder=5)
    ax.text(0, plot_ymin - 0.35 * label_space, compact_number(plot_ymin),
            ha="center", va="top", fontsize=6, zorder=5)

    # Cytokine label
    ax.text(0.5, -0.08, cytokine, transform=ax.transAxes, ha="center", va="top", rotation=90)

    # Continuous upper and lower frame
    ax.spines["top"].set_visible(True)
    ax.spines["bottom"].set_visible(True)
    ax.spines["top"].set_linewidth(0.7)
    ax.spines["bottom"].set_linewidth(0.7)

    # Outer left/right borders only
    ax.spines["left"].set_visible(i == 0)
    ax.spines["right"].set_visible(i == len(cytokine_conditions) - 1)

    # Subtle internal separators
    if i < len(cytokine_conditions) - 1:
        ax.axvline(0.48,color="0.82",linewidth=0.45,clip_on=False,zorder=0,)

    group_ifn = subsub.loc[subsub["donor_type"] == "interferon", "value"].dropna().values
    group_nonifn = subsub.loc[subsub["donor_type"] == "non-interferon", "value"].dropna().values
    n_at_llod = (np.r_[group_ifn, group_nonifn] <= llod * (1 + 1e-6)).sum()
    if len(group_ifn) < 2 or len(group_nonifn) < 2 or n_at_llod > 3:
        p = np.nan
    else:
        stat, p = stats.mannwhitneyu(group_ifn, group_nonifn, alternative="two-sided", method="auto")
    p_values.append(p)
    p_label_positions.append((ax, plot_ymax + 1.5 * label_space))

p_values = np.asarray(p_values, dtype=float)
valid = np.isfinite(p_values)
p_adjusted = np.full_like(p_values, np.nan)
p_adjusted[valid] = multipletests(p_values[valid], method="fdr_bh")[1]
for (ax, y), p_adj in zip(p_label_positions, p_adjusted):
    label = "n/a" if np.isnan(p_adj) else f"{p_adj:.2g}"
    ax.text(0, y, label, ha="center", va="center", fontsize=6)

fig.text(0.025, 0.99, 'p =')
fig.text(0.025, 0.5, "Supernatant proteomics (ng/ml)",rotation=90,va="center",ha="left",)
plt.subplots_adjust(left=0.06,right=0.995,top=0.95,bottom=0.28,wspace=0.015,)
plt.savefig(os.path.join(current_plot_folder, "supernatant_cytokines_pbs.svg"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
data = raw_log2fc_by_donor_t0["CD4 T cell"].loc[:, ["PBS", "AB-IFN-gamma"]].T
sample_index = data.index
data.index = data.index.map("__".join)
activities, _ = run_decoupler(data, cytosig_signature)
activities.index = sample_index
scores = activities["IFN1"].unstack("cytokine")

paired = pd.concat({
    "IFNG": proteomics_values_raw.loc["PBS", "IFNG"].astype(float),
    "suppression": scores["PBS"] - scores["AB-IFN-gamma"],
}, axis=1).dropna()
x = np.log10(paired["IFNG"])
r = x.corr(paired["suppression"])

fig, ax = plt.subplots(figsize=cm_to_inches((5, 4)))
ax.scatter(x, paired["suppression"], s=10, c=[DONOR_COLOR_DIRECT[d] for d in paired.index])
for donor in paired.index:
    ax.annotate(donor, (x[donor], paired.loc[donor, "suppression"]),
                xytext=(2, 2), textcoords="offset points", fontsize=6)
ax.set(xlabel=introduce_greek_letters("log10(PBS IFN-gamma, ng/ml)"), ylabel="IFN1 suppression", title=f"CD4 T cell: r={r:.2f}")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, "IFN-gamma_supression.svg"))
plt.show()

In [ ]:
# Do the ABs inhibit supernatant cytokines in the PBS condition?
conditions = [
    (("PBS", "AB-IFN-gamma"), "IFNG"),
    (("PBS", "AB-IL-6"), "IL6"),
    (("PBS", "AB-TNF-alpha"), "TNF"),
    (("PBS", "IL-1Ra"), "IL1B"),
]

fig, axes = plt.subplots(
    nrows=1, ncols=4,
    figsize=cm_to_inches((14, 2.5)),
    gridspec_kw={'width_ratios': [1.8, 1, 1, 1]} 
)
for i, condition in enumerate(conditions):
    ax = axes[i]
    treatments, label = condition
    sub = proteomics_values_raw.loc[(treatments, slice(None)), :].astype(float)[label].to_frame().reset_index()
    sub[label] = sub[label].astype(float)
    sub['treatment_alt'] = sub['treatment'].apply(lambda x: format_treatment(x, connection='\n+\n'))
    sub_pbs = sub[sub['treatment'] == "PBS"].set_index('donor')[label]
    sub_treat = sub[sub['treatment'] == treatments[1]].set_index('donor')[label]
    donors = [d for d in DONOR_ORDER_IFN_GROUPED if d in sub_treat.index]
    sub_pbs = sub_pbs.loc[donors]
    sub_treat = sub_treat.loc[donors]

    bar_colors = [DONOR_COLOR_DIRECT[d] for d in donors]
    ax.bar(sub_pbs.index, sub_pbs.values, color=bar_colors, edgecolor='black')
    ax.bar(sub_treat.index, sub_treat.values, color='tab:gray', edgecolor='black', alpha=0.5,
           label=format_treatment(treatments[1]))
    ax.set_title(label)
    if i == 0:
        ax.set_ylabel('supernatant proteomics\n(ng/ml)')
    else:
        ax.set_ylabel('')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=90, length=0)
    ax.legend(loc='upper right', fontsize=7)
    ax.set_xlabel('Donor')
 
plt.subplots_adjust(wspace=0.35, hspace=0.3)
plt.savefig(os.path.join(current_plot_folder, "antibody_treatments.svg"), dpi=300, bbox_inches='tight')
plt.show()

### Donor state

In [ ]:
import statsmodels.api as statsmodels_api
import statsmodels.formula.api as statsmodels_formula

# for when donor and treatment are confounded - they're not, at least not all that much
# def variance_partition(data, factor, value_name="log10_raw_conc", categorical=True, prefix=None):
#     """Split R^2 between treatment and a second factor; jointly explained R^2 is split equally."""
#     prefix = factor if prefix is None else prefix
#     nan_result = {f"{prefix}_treatment_fraction": np.nan, f"{prefix}_fraction": np.nan, f"{prefix}_unexplained_fraction": np.nan}

#     data = data[[value_name, "treatment", factor]].dropna()
#     n_parameters = data["treatment"].nunique() + (data[factor].nunique() - 1 if categorical else 1)
#     if len(data) <= n_parameters or data["treatment"].nunique() < 2 or data[factor].nunique() < 2 or data[value_name].nunique() < 2:
#         return nan_result

#     term = f'C(Q("{factor}"))' if categorical else f'Q("{factor}")'
#     r2_treatment = statsmodels_formula.ols(f'Q("{value_name}") ~ C(treatment)', data=data).fit().rsquared
#     r2_factor = statsmodels_formula.ols(f'Q("{value_name}") ~ {term}', data=data).fit().rsquared
#     r2_both = statsmodels_formula.ols(f'Q("{value_name}") ~ C(treatment) + {term}', data=data).fit().rsquared

#     return {
#         f"{prefix}_treatment_fraction": 0.5 * (r2_treatment + (r2_both - r2_factor)),
#         f"{prefix}_fraction": 0.5 * (r2_factor + (r2_both - r2_treatment)),
#         f"{prefix}_unexplained_fraction": 1 - r2_both,
#     }

def variance_fraction(data, factor, value_name="log10_raw_conc", categorical=True):
    """Fraction of variation in a protein explained by a single factor (R^2 of a one-factor linear model)."""
    data = data[[value_name, factor]].dropna()
    n_parameters = data[factor].nunique() if categorical else 2
    if len(data) <= n_parameters or data[factor].nunique() < 2 or data[value_name].nunique() < 2:
        return np.nan
    term = f'C(Q("{factor}"))' if categorical else f'Q("{factor}")'
    return statsmodels_formula.ols(f'Q("{value_name}") ~ {term}', data=data).fit().rsquared

In [ ]:
intracellular_proteins = list(protein_classification[
    protein_classification['extracellular_origin'] == 'expected_intracellular'
]['protein'].values)
secreted_proteins = list(protein_classification[
    protein_classification['extracellular_origin'] == 'classically_secreted'
]['protein'].values)

PROTEIN_SCORE_SETS = {
    "NEUTROPHIL": ["AZU1", "ELANE", "MMP8"],
    "PLATELET": ["GP6", "PF4", "SELP", "TREML1", "PDGFA"],
    "INTRACELLULAR": [p for p in intracellular_proteins if p in proteomics_values_raw.columns],
    "SECRETED": [p for p in secreted_proteins if p in proteomics_values_raw.columns]
}

In [ ]:
SHARED_CONDITIONS = proteomics_values_raw.loc[(slice(None), 'D1'), :].index.get_level_values('treatment').unique()

def log2_or_nan(x):
    x = pd.to_numeric(x, errors="coerce")
    return np.log2(x.where(x > 0))
log_values = proteomics_values_raw.apply(log2_or_nan)

# need some variability for this to be meaningful
log_range = log_values.max() - log_values.min()
log_values = log_values.loc[:, log_range > 2].copy()

z_scores = robust_z_score(log_values.T).T
z_scores.index.names = ["treatment", "donor"]
z_scores.columns.name = "protein"

neutrophil_score = z_scores[["AZU1", "ELANE", "MMP8"]].mean(axis=1).rename("neutrophil_score")
intracellular_proteins_present = [p for p in intracellular_proteins if p in z_scores.columns]
intracellular_score = z_scores[intracellular_proteins_present].mean(axis=1).rename("intracellular_score")

proteomics_scores = []
for pset_name, pset in PROTEIN_SCORE_SETS.items():
    pset_present = [p for p in pset if p in z_scores.columns]
    set_score = z_scores[pset_present].mean(axis=1).rename(pset_name)
    proteomics_scores.append(set_score)
proteomics_scores = pd.concat(proteomics_scores, axis=1).reset_index()
proteomics_scores_unfiltered = proteomics_scores.copy()
proteomics_scores = proteomics_scores[proteomics_scores['treatment'].isin(SHARED_CONDITIONS)].copy()

In [ ]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
setup_plotting_environment()

# Highlight only D4, D9, and D11
proteomics_scores["donor_highlight"] = proteomics_scores["donor"].where(proteomics_scores["donor"].isin(["D4", "D9", "D11"]),"Other")

combined_mean = proteomics_scores.groupby('donor')[list(PROTEIN_SCORE_SETS.keys())].mean()
combined_mean['IFN'] = cyto_acts_review_24h
r_by_score = {}
for key in PROTEIN_SCORE_SETS.keys():
    r = np.corrcoef([combined_mean[key], combined_mean['IFN']])[0, 1]
    r_by_score[key] = r

for key in PROTEIN_SCORE_SETS.keys():
    if key == 'SECRETED':
        continue
    proteomics_scores[f"{key}_EXCESS"] = proteomics_scores[key] - proteomics_scores['SECRETED']

palette = {
    "Other": "0.75",
    "D4": sns.color_palette()[0],
    "D9": sns.color_palette()[1],
    "D11": sns.color_palette()[2],
}
plt.figure(figsize=cm_to_inches((3.5, 2.5)))
plt.axhline(0, color="0.8", linewidth=0.7, zorder=0)
plt.axvline(0, color="0.8", linewidth=0.7, zorder=0)
sns.scatterplot(
    data=proteomics_scores,
    x="INTRACELLULAR_EXCESS",
    y="NEUTROPHIL_EXCESS",
    hue="donor_highlight",
    hue_order=["Other", "D4", "D9", "D11"],
    palette=palette,
    s=10,
    edgecolor="none",
    zorder=1,
)
plt.xlabel('intracellular excess')
plt.ylabel('neutrophil excess')
plt.legend(loc=[1, 0], title="")
plt.savefig(os.path.join(current_plot_folder, '2_neutrophil_vs_intracellular_score.svg'))
plt.show()

fig, axes = plt.subplots(1,3,figsize=cm_to_inches((12,3)), sharex=True)
axes = axes.flatten()

scores = ["NEUTROPHIL", "PLATELET", "INTRACELLULAR"]

# Compare each donor's score distribution with the pooled other donors.
tests = []
for score in scores:
    for donor in DONOR_ORDER_IFN_GROUPED:
        mask = proteomics_scores["donor"].eq(donor)
        x = proteomics_scores.loc[mask, score].dropna()
        y = proteomics_scores.loc[~mask, score].dropna()
        p = mannwhitneyu(x, y, alternative="greater").pvalue
        tests.append({"score": score, "donor": donor, "p": p})

tests = pd.DataFrame(tests)
tests["p_adj"] = multipletests(tests["p"], method="holm")[1]
for score_i, score in enumerate(scores):
    ax = axes[score_i]
    sns.stripplot(proteomics_scores, x='donor', y=score, order=DONOR_ORDER_IFN_GROUPED, ax=ax, s=2, color='black', edgecolor='none', jitter=0.25)
    if score_i==0:
        ax.set_ylabel("score")
    else:
        ax.set_ylabel("")

    significant = tests[tests["score"].eq(score) & tests["p_adj"].lt(0.01)]
    for row in significant.itertuples():
        ax.text(DONOR_ORDER_IFN_GROUPED.index(row.donor), 1.03, f"{row.p_adj:.1e}", transform=ax.get_xaxis_transform(),
                ha="center", va="bottom", rotation=0, fontsize=6, clip_on=False)

    title = f"Pearson r to IFN activity: {r_by_score[score]:.2f}"
    ax.set_title(title, pad=10, fontsize=6)
    ax.text(-0.05, 1.03, f"p =", transform=ax.transAxes, va="bottom")
    ax.set_xlabel("")
    ax.set_ylabel(f"{score} score".lower())
    ax.tick_params(axis='x', length=0, rotation=90)
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, '2_supernatant_score_distribution.svg'))
plt.show()

# 2.3 - Supernatant simulation

In [ ]:
proteomics_analyte_metadata = pd.read_parquet(os.path.join(output_folder, 'proteomics_analyte_metadata.parquet'))
proteomics_analyte_metadata = proteomics_analyte_metadata.set_index('protein')

proteomics_values_raw = pd.read_parquet(os.path.join(output_folder, 'proteomics_values_raw.parquet'))
proteomics_melt = proteomics_values_raw.reset_index().melt(id_vars=['treatment', 'donor'], var_name='protein', value_name='raw_conc')
proteomics_melt['raw_conc'] = proteomics_melt['raw_conc'].astype(float)

# add metadata
proteomics_melt['ulod'] = proteomics_melt['protein'].apply(lambda x: proteomics_analyte_metadata.loc[x, 'upper_limit_of_detection'])
proteomics_melt['llod'] = proteomics_melt['protein'].apply(lambda x: proteomics_analyte_metadata.loc[x, 'lower_limit_of_detection'])
proteomics_melt['gene'] = proteomics_melt['protein'].apply(lambda x: proteomics_analyte_metadata.loc[x, 'gene_name'])

In [ ]:
significant_results_genes = set(significant_results['gene'].unique())

# check which genes are available
target_rows = []
for index, row in proteomics_analyte_metadata.iterrows():
    protein = index
    genes = split_and_strip(row["gene_name"])
    missing_genes = [gene for gene in genes if gene not in count_df_norm_agg.index]
    usable_genes = [gene for gene in genes if gene in count_df_norm_agg.index]
    any_significant = any([gene in significant_results_genes for gene in genes])
    target_rows.append(
        {
            "protein": protein,
            "target_genes": ";".join(genes),
            "usable_genes": ";".join(usable_genes),
            "missing_genes_from_scrna": ";".join(missing_genes),
            "n_target_genes": len(genes),
            "n_usable_genes": len(usable_genes),
            "all_genes_available": len(genes) > 0 and len(missing_genes) == 0,
            "any_significant": any_significant
        }
    )

target_table = pd.DataFrame(target_rows)
target_table["status"] = np.select(
    [
        target_table["n_target_genes"].eq(0),
        target_table["n_usable_genes"].eq(0),
        ~target_table["all_genes_available"],
    ],
    ["no_target_gene", "no_scrna_gene", "partial_scrna_genes"],
    default="ok",
)

print(target_table["status"].value_counts(dropna=False))
target_table[target_table['status']=='no_scrna_gene']

In [ ]:
# get the relevant parts of count_df_norm
genes_needed = sorted({gene for genes_text in target_table["usable_genes"] for gene in genes_text.split(";") if gene})
count_df_norm_needed = count_df_norm_agg.loc[genes_needed]

# calculate cell fractions
cell_fraction_vector = calculate_cell_fractions(cell_count_agg, count_df_norm_needed.columns)
cell_fraction_vector_scaled = calculate_cell_fractions(cell_count_agg, count_df_norm_needed.columns, mean_UMIs_per_cell)

## Supernatant calculation

In [ ]:
setup_plotting_environment()

In [ ]:
from utils.proteomics_utils import log10_or_nan, calculate_supernatant, run_fit, determine_censored_range

def classify(row):
    sim_range = row['total_sim_range']
    conc_range = row['total_conc_range']
    max_conc = row['max_conc']
    if sim_range < 1 and conc_range < 1:
        return "low_range_both"
    if sim_range > 1 and conc_range < np.log10(3):
        return "high_sim_low_conc"
    if sim_range < np.log10(3) and conc_range > 1 and max_conc > 0:
        return 'low_sim_high_conc'
    if sim_range > 1 and conc_range > 1:
        return "high_range_both"
    return 'intermediate'

def run_overall_calculation(row, count_df_norm_needed, cell_fraction_vector, cell_states=None, state_sensitivity=None, timepoint0_weight=0, min_target_umis=20):
    protein = row['protein']
        
    # baseline calculation
    supernatant, contrib = calculate_supernatant(
        row,
        count_df_norm_needed,
        cell_fraction_vector,
        cell_states=cell_states,
        state_sensitivity=state_sensitivity,
        censored_treatments_all=PROTEIN_TO_CENSORED_TREATMENTS,
        timepoint0_weight=timepoint0_weight
    )
    
    # merge and calculate log10s
    supernatant = supernatant.merge(
        proteomics_melt,
        on=["protein", "treatment", "donor"],
        how="inner",
    )
    for column in ["raw_conc", "llod", "ulod", "sim_supernatant"]:
        supernatant[f"log10_{column}"] = log10_or_nan(supernatant[column])
    supernatant = supernatant.dropna(subset=["log10_raw_conc", "log10_llod", "log10_ulod", "log10_sim_supernatant"])

    # add evaluability info
    total_sim_range = supernatant['log10_sim_supernatant'].max() - supernatant['log10_sim_supernatant'].min()
    total_conc_range = supernatant['log10_raw_conc'].max() - supernatant['log10_raw_conc'].min()
    cytokine = proteomics_analyte_metadata.loc[protein, 'cytokine']
    is_cytokine = cytokine != 'none'

    # donor and contamination contributions
    variance_info = {"model_donor_fraction": variance_fraction(supernatant, "donor")}
    supernatant_scores = supernatant.merge(proteomics_scores[["treatment", "donor", "NEUTROPHIL", "PLATELET", "INTRACELLULAR"]], on=["treatment", "donor"], how="inner")
    score_info = {f"{score_name}_score_fraction": variance_fraction(supernatant_scores, score_name, categorical=False) for score_name in ["NEUTROPHIL", "PLATELET", "INTRACELLULAR"]}
    
    # proteomics censoring based on detection sensitivity
    supernatant, boundary_by_interval = determine_censored_range(supernatant)

    # RNA UMI-based censoring
    genes = [g for g in row['usable_genes'].split(';') if g]
    umis = count_df_raw_agg.loc[genes].T.groupby(level=['cytokine', 'donor']).sum().min(axis=1)
    supernatant['target_umis'] = umis.reindex(pd.MultiIndex.from_frame(supernatant[['treatment', 'donor']])).values
    supernatant['rna_censored'] = supernatant['target_umis'].fillna(0) < min_target_umis
    supernatant['mostly_censored'] = supernatant['mostly_censored'] | supernatant['rna_censored']

    supernatant['row_identity'] = supernatant['protein'] + "_" + supernatant['treatment'] + "_" + supernatant['donor']
    protein_evaluability_info = {
        "protein": protein,
        "total_sim_range": total_sim_range,
        "max_sim": supernatant['log10_sim_supernatant'].max(),
        "total_conc_range": total_conc_range,
        "max_conc": supernatant['log10_raw_conc'].max(),
        "boundary_by_interval": boundary_by_interval,
        "no_llod": len(supernatant[supernatant['at_llod']]),
        "no_ulod": len(supernatant[supernatant['at_ulod']]),
        "no_rna_censored": len(supernatant[supernatant['rna_censored']]),
        "n": len(supernatant),
        "n_uncensored": (~supernatant['mostly_censored']).sum(),
        "cytokine": cytokine,
        "is_cytokine": is_cytokine,
        **variance_info,
        **score_info
        }
    protein_evaluability_info['class'] = classify(protein_evaluability_info)
    
    # fit
    fit, x, y, identity, _, _ = run_fit(supernatant)
    if not pd.isna(fit['corr']):
        predicted = fit['slope'] * x + fit['intercept']
        residual = y - predicted
        fitted_points = pd.DataFrame({
            "x": x,
            "y": y,
            "predicted": predicted,
            "residual": residual,
            "identity": identity,
        })
    else:
        fitted_points = pd.DataFrame()

    fit = {**fit, **protein_evaluability_info}

    return supernatant, contrib, fit, fitted_points

def post_process_supernatant_lists(
        simulated_supernatant,
        cell_type_contributions,
        fits
    ):
    fits = pd.DataFrame(fits)
    fits = fits.merge(protein_classification, on='protein')
    fits['family_level_1'] = fits['uniprot_ids'].apply(lambda x: cytokine_info.set_index('uniprot')['family_level_1'].get(x, 'missing'))
    fits['family_level_2'] = fits['uniprot_ids'].apply(lambda x: cytokine_info.set_index('uniprot')['family_level_2'].get(x, 'missing'))

    simulated_supernatant = pd.concat(simulated_supernatant, ignore_index=True)
    initial_cols = ['protein', 'treatment', 'donor', 'log10_sim_supernatant', 'log10_raw_conc']
    other_cols = [col for col in simulated_supernatant.columns if not col in initial_cols]
    simulated_supernatant = simulated_supernatant[initial_cols+other_cols]
    simulated_supernatant['cytokine'] = simulated_supernatant['treatment'].str.split("_").str[0]

    cell_type_contributions = pd.concat(cell_type_contributions, ignore_index=True)
    cell_type_contributions = cell_type_contributions.merge(
        simulated_supernatant[["protein", "treatment", "donor", "sim_supernatant"]],
        on=["protein", "treatment", "donor"],
        how="left",
    )
    cell_type_contributions["contribution_fraction"] = (
        cell_type_contributions["cell_type_contribution"] / cell_type_contributions["sim_supernatant"]
    )
    cell_type_contributions = cell_type_contributions[
    ["protein","treatment","donor","cell_type","target_genes","aggregation","n_subunits","cell_type_contribution","sim_supernatant","contribution_fraction"]
    ]
    cell_type_contributions['contribution_fraction'] = cell_type_contributions['contribution_fraction'].fillna(0)

    return simulated_supernatant, cell_type_contributions, fits

# -------- Baseline calculation -----------
simulated_supernatant_unscaled = []
cell_type_contributions_unscaled = []
fits_unscaled = []
fitted_points_all_unscaled = []
for _, row in target_table.loc[target_table["status"].eq("ok")].iterrows():
    supernatant, contrib, fit, fitted_points = run_overall_calculation(
        row,
        count_df_norm_needed=count_df_norm_needed,
        cell_fraction_vector=cell_fraction_vector,
        )
    fits_unscaled.append(fit)
    simulated_supernatant_unscaled.append(supernatant)
    cell_type_contributions_unscaled.append(contrib)
    fitted_points_all_unscaled.append(fitted_points)

simulated_supernatant_unscaled, cell_type_contributions_unscaled, fits_unscaled = post_process_supernatant_lists(
    simulated_supernatant=simulated_supernatant_unscaled,
    cell_type_contributions=cell_type_contributions_unscaled,
    fits=fits_unscaled
)
fitted_points_all_unscaled = pd.concat(fitted_points_all_unscaled)

# --------- scale RNA -------------
simulated_supernatant = []
cell_type_contributions = []
fits = []
fitted_points_all = []
for _, row in target_table.loc[target_table["status"].eq("ok")].iterrows():
    supernatant, contrib, fit, fitted_points = run_overall_calculation(
        row,
        count_df_norm_needed=count_df_norm_needed,
        cell_fraction_vector=cell_fraction_vector_scaled
        )
    simulated_supernatant.append(supernatant)
    cell_type_contributions.append(contrib)
    fits.append(fit)
    fitted_points_all.append(fitted_points)
    
simulated_supernatant, cell_type_contributions, fits = post_process_supernatant_lists(
    simulated_supernatant=simulated_supernatant,
    cell_type_contributions=cell_type_contributions,
    fits=fits
)
fitted_points_all = pd.concat(fitted_points_all) 

In [ ]:
run_t0_flag = False

if run_t0_flag:
    # --------- t0 addition (doesn't help, leave vestigial code here) -------------
    timepoint0_weights = {"t0_weight_0_1": 0.1, "t0_weight_0_5": 0.5}
    simulated_supernatants_by_weight = {name: [] for name in timepoint0_weights}
    cell_type_contributions_by_weight = {name: [] for name in timepoint0_weights}
    fits_by_weight = {name: [] for name in timepoint0_weights}
    fitted_points_by_weight = {name: [] for name in timepoint0_weights}

    for _, row in target_table.loc[target_table["status"].eq("ok")].iterrows():
        for result_name, timepoint0_weight in timepoint0_weights.items():
            supernatant, contrib, fit, fitted_points = run_overall_calculation(
                row,
                count_df_norm_needed=count_df_norm_needed,
                cell_fraction_vector=cell_fraction_vector_scaled,
                timepoint0_weight=timepoint0_weight,
            )
            simulated_supernatants_by_weight[result_name].append(supernatant)
            cell_type_contributions_by_weight[result_name].append(contrib)
            fits_by_weight[result_name].append(fit)
            fitted_points_by_weight[result_name].append(fitted_points)

    for result_name in timepoint0_weights:
        simulated_supernatants_by_weight[result_name], cell_type_contributions_by_weight[result_name], fits_by_weight[result_name] = post_process_supernatant_lists(
            simulated_supernatant=simulated_supernatants_by_weight[result_name],
            cell_type_contributions=cell_type_contributions_by_weight[result_name],
            fits=fits_by_weight[result_name],
        )
        fitted_points_by_weight[result_name] = pd.concat(fitted_points_by_weight[result_name], ignore_index=True)

### IL-1 family release analysis

In [ ]:
import gseapy as gp

msigdb = gp.Msigdb()

# Download the needed human collections as {gene_set_name: [gene_symbols]}.
hallmark = msigdb.get_gmt(category="h.all", dbver="2026.1.Hs")
canonical = msigdb.get_gmt(category="c2.cp", dbver="2026.1.Hs")
go_bp = msigdb.get_gmt(category="c5.go.bp", dbver="2026.1.Hs")

release_genes = ['IL1B', 'IL1A'] + canonical["REACTOME_NON_CANONICAL_INFLAMMASOME_ACTIVATION"] + canonical["REACTOME_THE_NLRP3_INFLAMMASOME"] + canonical['REACTOME_PYROPTOSIS']
release_genes = sorted(set(release_genes))
len(release_genes)

In [ ]:
def build_release_df(count_df_norm_agg, simulated_supernatant, cell_type="Mono/cDC"):
    rna = count_df_norm_agg[cell_type].T

    index = ["treatment", "donor"]
    measured = simulated_supernatant.pivot_table(index=index, columns="protein", values="raw_conc", aggfunc="first").add_suffix("_protein")
    simulated = simulated_supernatant.pivot_table(index=index, columns="protein", values="sim_supernatant", aggfunc="first").add_suffix("_sim")
    protein = measured.join(simulated).reset_index().rename(columns={"treatment": "cytokine"})

    df = rna.merge(protein, on=["cytokine", "donor"], how="inner", validate="one_to_one")
    return df

release_df = build_release_df(count_df_norm_agg, simulated_supernatant, cell_type="Mono/cDC")

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

def calculate_gene_set_protein_auc(genes, protein, release_df, protein_threshold, protein_col=None):
    """Calculate ROC/AUC curves for multiple genes using release_df."""
    if protein_col is None:
        protein_col = f"{protein}_protein"

    data = release_df.dropna(subset=[protein_col]).copy()
    data["high_release"] = data[protein_col] >= protein_threshold

    roc_dfs = []
    auc_rows = []
    for gene in dict.fromkeys(genes):
        gene_data = data[[gene, "high_release"]].dropna()

        fpr, tpr, roc_thresholds = roc_curve(gene_data["high_release"], gene_data[gene])
        auc = roc_auc_score(gene_data["high_release"], gene_data[gene])
        
        roc_dfs.append(pd.DataFrame({"gene": gene, "fpr": fpr, "tpr": tpr, "threshold": roc_thresholds}))
        auc_rows.append({"gene": gene, "auc": auc, "n": len(gene_data)})

    roc_data = pd.concat(roc_dfs, ignore_index=True)
    auc_data = pd.DataFrame(auc_rows)

    plot_info = {"protein": protein, "protein_col": protein_col, "protein_threshold": protein_threshold}
    return roc_data, auc_data, plot_info


def plot_gene_set_protein_auc(roc_data, auc_data, plot_info, highlight_genes, highlight_colors=None):
    """Plot ROC curves for background and highlighted genes."""
    protein = plot_info["protein"]
    protein_threshold = plot_info['protein_threshold']

    if highlight_colors is None:
        highlight_colors = [f"C{i}" for i in range(len(highlight_genes))]
    highlight_colors = dict(zip(highlight_genes, highlight_colors))
    fig, ax = plt.subplots(figsize=cm_to_inches((7.5, 4)))

    # Highlighted genes
    for gene in highlight_genes:
        subset = roc_data.loc[roc_data["gene"] == gene]
        if subset.empty:
            continue
        auc = auc_data.loc[auc_data["gene"] == gene, "auc"].iloc[0]
        ax.plot(subset["fpr"], subset["tpr"], color=highlight_colors[gene], linewidth=1, alpha=1, label=f"{gene} (AUC={auc:.2f})", zorder=1)

    # Background genes
    background_genes = [gene for gene in roc_data["gene"].unique() if gene not in highlight_genes]
    background_label_added = False
    for gene in background_genes:
        subset = roc_data.loc[roc_data["gene"] == gene]
        ax.plot(subset["fpr"], subset["tpr"], color="0.75", linewidth=1, alpha=0.5, zorder=0,
        label="Other reactome\n(non-)canonical\ninflammasome or\npyroptosis genes" if not background_label_added else None)
        background_label_added = True

    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=0.8, color="black",)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel(f"Fraction low {protein}")
    ax.set_ylabel(f"Fraction high {protein}")
    ax.set_title(f"Classification of high {protein} conditions: {protein} ≥ {protein_threshold:g} ng/ml", fontsize=6)
    ax.legend(frameon=False, fontsize=6, loc=[1.1,-0.3])
    fig.tight_layout()

    return fig, ax

for gene in ['IL1A', 'IL1B']:
    roc_data, auc_data, auc_info = calculate_gene_set_protein_auc(
        genes=release_genes,
        protein=gene,
        release_df=release_df,
        protein_threshold=0.03
    )
    fig, ax = plot_gene_set_protein_auc(
        roc_data=roc_data,
        auc_data=auc_data,
        plot_info=auc_info,
        highlight_genes=[gene, "SERPINB1", "APP"],
        highlight_colors=["C0", "C1", "C2"],
    )
    plt.savefig(os.path.join(current_plot_folder, f'AUC_{gene}.svg'))
    plt.show()

In [ ]:
def make_gene_data(release_df, genes, baseline_col="IL1B", protein_col="IL1B_protein", treatment_col="cytokine", donor_col="donor"):
    genes = [gene for gene in genes if gene in release_df.columns]
    required_cols = [treatment_col, donor_col, baseline_col, protein_col, *genes]
    data = release_df[required_cols].dropna().reset_index(drop=True)
    return data, genes

def calc_partial_r2(data, gene, baseline_col, protein_col, treatment_col="cytokine"):
    """Fraction of residual variation (after treatment and baseline) explained by the gene; signed by its coefficient."""
    base = f'Q("{protein_col}") ~ C({treatment_col}) + Q("{baseline_col}")'
    reduced = statsmodels_formula.ols(base, data=data).fit()
    full = statsmodels_formula.ols(f'{base} + Q("{gene}")', data=data).fit()
    return np.sign(full.params[f'Q("{gene}")']) * (reduced.ssr - full.ssr) / reduced.ssr

def evaluate_genes_partial_r2(release_df, genes, baseline_col="IL1B", protein_col="IL1B_protein", treatment_col="cytokine", donor_col="donor", log10=True):
    data, genes = make_gene_data(release_df, genes, baseline_col=baseline_col, protein_col=protein_col, treatment_col=treatment_col, donor_col=donor_col)
    if log10:
        log_columns = [baseline_col, protein_col, *genes]
        data[log_columns] = np.log10(data[log_columns])

    results = []
    for gene in genes:
        partial_r2 = calc_partial_r2(data, gene, baseline_col=baseline_col, protein_col=protein_col, treatment_col=treatment_col)
        results.append({
            "gene": gene,
            "n": len(data),
            "partial_r2": partial_r2,
        })
    return pd.DataFrame(results).sort_values("partial_r2", ascending=False).reset_index(drop=True)

linear_fit_results = []
for gene in ['IL1A', 'IL1B']:
    results = evaluate_genes_partial_r2(
        release_df=release_df,
        genes=release_genes,
        baseline_col=f'{gene}_sim',
        protein_col=f'{gene}_protein',
        log10=True
        )
    results['target'] = gene
    linear_fit_results.append(results)
linear_fit_results = pd.concat(linear_fit_results)

In [ ]:
def optimal_gene_threshold(y, x):
    fpr, tpr, thresholds = roc_curve(y, x)
    j = tpr - fpr
    i = np.argmax(j)
    high_gene = x >= thresholds[i]
    return {"threshold": thresholds[i], "auc": roc_auc_score(y, x), "sensitivity": tpr[i], "specificity": 1 - fpr[i], 
        "n_low_gene": int((~high_gene).sum()), "n_high_gene": int(high_gene.sum()), "n_release_low_gene": int(y[~high_gene].sum()), "n_release_high_gene": int(y[high_gene].sum()),
        "low_gene_release_frequency": y[~high_gene].mean(),
        "high_gene_release_frequency": y[high_gene].mean(),}

def threshold_sensitivity_analysis(df, gene, protein_col, protein_thresholds):
    results = []
    for protein_threshold in protein_thresholds:
        sub = df[[gene, protein_col]].dropna()
        y = (sub[protein_col] >= protein_threshold).astype(int)
        if y.nunique() < 2:
            continue
        result = optimal_gene_threshold(y, sub[gene])
        results.append({"protein": protein_col, "protein_threshold": protein_threshold, **result})
    return pd.DataFrame(results)

# sensitivity analysis
all_sensitivity_results = []
for gene in release_genes:
    try:
        il1b_results = threshold_sensitivity_analysis(release_df, gene=gene,protein_col="IL1B_protein", protein_thresholds=[0.01, 0.03, 0.1])
        il1a_results = threshold_sensitivity_analysis(release_df, gene=gene,protein_col="IL1A_protein", protein_thresholds=[0.01, 0.03, 0.1])
        results = pd.concat([il1b_results, il1a_results])
        results['gene'] = gene
        all_sensitivity_results.append(results)
    except:
        continue
all_sensitivity_results = pd.concat(all_sensitivity_results)

In [ ]:
setup_plotting_environment()

target_gene = 'IL1B'
other_gene = 'IL1B' if target_gene == "IL1A" else 'IL1A'
protein = f"{target_gene}_protein"
primary_threshold = 0.03
top_n = 12

# append results
r2_results = linear_fit_results[linear_fit_results['target'] == target_gene].set_index("gene")["partial_r2"]
sensitivity_results_primary = all_sensitivity_results[
    (all_sensitivity_results["protein"] == protein)
    & np.isclose(all_sensitivity_results["protein_threshold"], primary_threshold)
].set_index("gene")["auc"]

# rank genes
performance = pd.concat([r2_results.rename("partial_r2"), sensitivity_results_primary.rename("auc")], axis=1)
performance["r2_rank"] = performance["partial_r2"].rank(pct=True)
performance["auc_rank"] = performance["auc"].rank(pct=True)
performance["overall_score"] = performance[["r2_rank", "auc_rank"]].mean(axis=1)
performance = performance.drop(other_gene, axis=0, errors='ignore')
performance = performance.sort_values("overall_score", ascending=False).head(top_n)
gene_order = list(performance.index)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=cm_to_inches((10, 4.5)), gridspec_kw={"width_ratios": [1.25, 1, 1, 1]})
y = np.arange(len(gene_order))

# single-gene AUC across IL1B thresholds
ax = axes[0]
auc_matrix = all_sensitivity_results[all_sensitivity_results["protein"].eq(protein) & all_sensitivity_results["gene"].isin(gene_order)]
auc_matrix = auc_matrix.pivot_table(index="gene", columns="protein_threshold", values="auc", aggfunc="mean").reindex(gene_order)
image = ax.imshow(auc_matrix, cmap="coolwarm", vmin=0, vmax=1,aspect="auto", interpolation="none")
for gene_i, gene in enumerate(auc_matrix.index):
    for threshold_i, threshold in enumerate(auc_matrix.columns):
        value = auc_matrix.loc[gene, threshold]
        if pd.notna(value):
            ax.text(threshold_i, gene_i, f"{value:.2f}", ha="center",va="center",fontsize=6,color="black")
            
ax.set_xticks(np.arange(len(auc_matrix.columns)),labels=[f"{threshold:g}" for threshold in auc_matrix.columns])
ax.set_yticks(y, labels=gene_order)
ax.set_xlabel(f"{target_gene} threshold (ng/ml)")
ax.tick_params(axis="both", length=0)
ax.set_title("AUC", fontsize=7)

# stability of the optimal RNA threshold
ax = axes[1]
threshold_matrix = all_sensitivity_results[
    all_sensitivity_results["protein"].eq(protein)
    & all_sensitivity_results["gene"].isin(gene_order)
].pivot_table(index="gene", columns="protein_threshold", values="threshold", aggfunc="mean").reindex(gene_order)

primary_col = threshold_matrix.columns[np.argmin(np.abs(threshold_matrix.columns - primary_threshold))]
relative_threshold = threshold_matrix.div(threshold_matrix[primary_col], axis=0)
ax.imshow(relative_threshold, cmap="coolwarm", vmin=0.5, vmax=1.5, aspect="auto", interpolation="none")
for gene_i, gene in enumerate(threshold_matrix.index):
    for threshold_i, threshold in enumerate(threshold_matrix.columns):
        value = threshold_matrix.loc[gene, threshold]
        if pd.notna(value):
            ax.text(threshold_i, gene_i, f"{value:.0f}", ha="center", va="center", fontsize=6, color="black")

ax.set_xticks(np.arange(len(threshold_matrix.columns)), labels=[f"{threshold:g}" for threshold in threshold_matrix.columns])
ax.set_yticks(y, labels=[])
ax.set_xlabel(f"{target_gene} threshold (ng/ml)")
ax.tick_params(axis="both", length=0)
ax.set_title("optimal cpm\nthreshold", fontsize=7)

# release frequency
ax = axes[2]
release_results = all_sensitivity_results[
    all_sensitivity_results["protein"].eq(protein)
    & np.isclose(all_sensitivity_results["protein_threshold"], primary_threshold)
].drop_duplicates("gene").set_index("gene").reindex(gene_order)

low_frequency = release_results["low_gene_release_frequency"]
high_frequency = release_results["high_gene_release_frequency"]

for gene_i in range(len(gene_order)):
    ax.plot([low_frequency.iloc[gene_i], high_frequency.iloc[gene_i]], [gene_i, gene_i], color="grey", linewidth=0.8, zorder=1)

ax.scatter(low_frequency, y, s=14, color="lightgrey", edgecolor="black", linewidth=0.4, label="low RNA", zorder=2)
ax.scatter(high_frequency, y, s=14, color="tab:red", edgecolor="black", linewidth=0.4, label="high RNA", zorder=2)

ax.set_yticks(y, labels=[])
ax.set_ylim(len(gene_order) - 0.5, -0.5)
ax.set_xlim(0, 1)
ax.set_xlabel(f"Fraction {target_gene}\n≥ {primary_threshold:g} ng/ml")
ax.tick_params(axis="both", length=0)
ax.legend(frameon=False, fontsize=6)
ax.set_title('release\ngating')

# partial R^2
ax = axes[3]
values = performance["partial_r2"]
ax.barh(y, values, color='tab:blue', edgecolor="black", linewidth=0.4)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(y, labels=[])
ax.set_ylim(len(gene_order) - 0.5, -0.5)
ax.set_xlabel(r"partial $R^2$ (signed)")
ax.tick_params(axis="both", length=0)
ax.set_title('beyond treatment\nand production')

plt.subplots_adjust(left=0.13, right=0.98, bottom=0.19, top=0.86, wspace=0.2)
plt.savefig(os.path.join(current_plot_folder, f"gene_performance_{target_gene}.svg"),bbox_inches="tight",)
plt.show()

In [ ]:
def calculate_release_gate(release_df, protein, state_gene, state_threshold, protein_threshold, protein_gene=None, cell_type="Mono/cDC"):
    """Classify protein release using candidate-gene thresholds."""

    protein_gene = protein if protein_gene is None else protein_gene
    protein_col = f"{protein}_protein"

    columns = ["cytokine", "donor", state_gene, protein_gene, protein_col]
    missing = [column for column in columns if column not in release_df.columns]
    if missing:
        raise KeyError(f"Missing columns from release_df: {missing}")

    data = release_df[columns].dropna().copy()
    data = data.loc[(data[[state_gene, protein_gene, protein_col]] > 0).all(axis=1)]
    if data.empty:
        raise ValueError(f"No usable observations remain for {protein}")

    data["state_score"] = np.log10(data[state_gene])
    data["target_gene_score"] = np.log10(data[protein_gene])
    data["plot_log10_conc"] = np.log10(data[protein_col])
    data["high_release"] = data[protein_col] >= protein_threshold

    state_order = ["low", "high"]
    data["state_class"] = pd.Categorical(np.where(data[state_gene] >= state_threshold, "high", "low"), categories=state_order, ordered=True)
    plot_info = {
        "protein": protein,
        "protein_gene": protein_gene,
        "state_gene": state_gene,
        "cell_type": cell_type,
        "protein_threshold": protein_threshold,
        "log10_release_cutoff": np.log10(protein_threshold),
        "log10_state_threshold": np.log10(state_threshold),
        "state_order": state_order,
        "colors": {"low": "lightgrey", "high": "tab:red"},
    }

    return data, plot_info


def plot_release_gate_middle(data, plot_info):
    """Plot the candidate-gene release gate and quadrant counts."""

    state_gene = plot_info["state_gene"]
    threshold = plot_info["log10_state_threshold"]
    release_cutoff = plot_info["log10_release_cutoff"]

    fig, ax = plt.subplots(figsize=cm_to_inches((3.6, 2.7)))
    ax.scatter(data["state_score"], data["plot_log10_conc"], s=5, color="black", edgecolor="none", alpha=0.85, rasterized=True)
    ax.axvline(threshold, color="tab:red", linestyle="--", linewidth=1)
    ax.axhline(release_cutoff, color="tab:red", linestyle="--", linewidth=1)
    ax.set_xlabel(f'log10({state_gene} RNA) in {plot_info["cell_type"]}')
    ax.set_ylabel(f'log10({plot_info["protein"]} ng/ml)')

    left = data["state_score"] < threshold
    high = data["plot_log10_conc"] > release_cutoff
    quadrants = [
        (left & high, 0.03, 0.97, "left", "top"),
        (~left & high, 0.97, 0.97, "right", "top"),
        (left & ~high, 0.03, 0.03, "left", "bottom"),
        (~left & ~high, 0.97, 0.03, "right", "bottom"),
    ]

    for mask, x, y, ha, va in quadrants:
        ax.text(x, y, str(mask.sum()), transform=ax.transAxes, color="tab:red", fontsize=6, fontweight="bold", ha=ha, va=va,
                bbox=dict(facecolor="white", edgecolor="none", pad=0, alpha=0.5))

    ax.tick_params(axis='both', length=0)
    fig.tight_layout()
    return fig, ax

for target_gene in ['IL1A', 'IL1B']:
    gate_data, plot_info = calculate_release_gate(
        release_df=release_df,
        protein=target_gene,
        state_gene="SERPINB1",
        state_threshold=90,
        protein_threshold=0.03,
    )
    fig, axes = plot_release_gate_middle(gate_data, plot_info)
    plt.savefig(os.path.join(current_plot_folder, f'visualize_release_gate_{target_gene}.svg'), dpi=400)
    plt.show()

### Supernatant model performance

In [ ]:
def apply_fit_filter(fits):
    fits_filter = fits.dropna(subset='corr').copy()
    fits_filter = fits_filter[fits_filter['class'] != 'low_range_both'].copy()
    fits_filter = fits_filter[fits_filter['fitted_points'] > 12].copy()
    fits_filter = fits_filter[fits_filter['effective_range'] > np.log10(5)]

    fits_filter['extracellular_origin'] = fits_filter['extracellular_origin'].str.replace("_", " ")
    replace = {
        "nonclassical regulated release": "nonclassical release"
    }
    fits_filter['extracellular_origin'] = fits_filter['extracellular_origin'].apply(lambda x: replace.get(x, x))

    # throw out IL27 (IL27/EBI3 is the relevant protein complex here)
    fits_filter = fits_filter[fits_filter['protein'] != 'IL27'].copy()

    print(f"Retained {len(fits_filter)}/{len(fits)} supernatant proteins.")
    return fits_filter

fits_filter_unscaled = apply_fit_filter(fits_unscaled)
fits_filter = apply_fit_filter(fits)

if run_t0_flag:
    fits_filter_by_weight = {}
    for key in fits_by_weight.keys():
        fits_filter_by_weight[key] = apply_fit_filter(fits_by_weight[key])

In [ ]:
all_mean_contributions = []
all_std_contributions = []
for index, row in fits_filter.iterrows():
    protein = row['protein']
    supernatant = simulated_supernatant[simulated_supernatant['protein'] == protein].copy().sort_values('log10_sim_supernatant')
    contributions = cell_type_contributions[cell_type_contributions['protein'] == protein].copy()

    high_supernatant = supernatant.iloc[int(len(supernatant) * 0.8):].copy()
    high_supernatant['treat_identity'] = high_supernatant['treatment'] + high_supernatant['donor']
    
    contributions['treat_identity'] = contributions['treatment'] + contributions['donor']
    original = len(contributions)
    contributions = contributions[contributions['treat_identity'].isin(high_supernatant['treat_identity'])].copy()

    mean_contributions = contributions.groupby('cell_type')['contribution_fraction'].mean()
    std_contributions = contributions.groupby('cell_type')['contribution_fraction'].std()
    mean_contributions.name = protein
    std_contributions.name = protein
    all_mean_contributions.append(mean_contributions)
    all_std_contributions.append(std_contributions)
all_mean_contributions = pd.concat(all_mean_contributions, axis=1).T
all_std_contributions = pd.concat(all_std_contributions, axis=1).T

In [ ]:
agg_colors = {
    "B cell": 'tab:blue',
    'T cell': 'tab:orange',
    'Mono/cDC': 'tab:purple',
    'NK cell': 'tab:brown'
}

high_r = fits_filter[
    (fits_filter['corr'] > 0.5) &
    (fits_filter['extracellular_origin'] == 'classically secreted')
    ].copy()
all_mean_contributions_sub = all_mean_contributions.loc[high_r['protein'].values]

plt.figure(figsize=cm_to_inches((3,4.4)))
sns.boxplot(data=all_mean_contributions_sub, flierprops={'markersize': 2, 'markerfacecolor': 'black'}, palette=agg_colors)
plt.xlabel('')
plt.ylabel('relative contribution')
plt.title('high secretion conditions\n(80th percentile)', fontsize=6)
plt.tick_params(axis='x', length=0, rotation=90)

plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, '2_per_cell_type_contributions.svg'))
plt.show()

In [ ]:
specific_plot_cytokines = ['CXCL9', 'IL6', 'IFNG', 'IL1B', 'TNF']
fig, axes = plt.subplots(1,5,figsize=cm_to_inches((13,1.5)), sharex=True, sharey = True)
for c_i, c in enumerate(specific_plot_cytokines):
    ax = axes[c_i]
    sub = all_mean_contributions.loc[c]
    std = all_std_contributions.loc[c]
    name = gene_to_cytokine[c]
    colors = agg_colors.values()
    ax.bar(sub.index, sub, color=colors, yerr=std, edgecolor='black')
    ax.set_ylim(0, 1.2)
    ax.set_title(introduce_greek_letters(name))
    if c_i == 0:
        ax.set_ylabel("relative\ncontribution")
    ax.tick_params(axis='x', length=0, rotation=45)

plt.subplots_adjust(wspace=0.3,  hspace=0.1)
plt.savefig(os.path.join(current_plot_folder, '2_contributions_specific.svg'))
plt.show()

In [ ]:
# The improvements on r are tiny, but this makes sense from first principles, so keep the weighted fractions
from scipy.stats import wilcoxon

fits_filter_sub = fits_filter_unscaled[
    (fits_filter_unscaled['extracellular_origin'] == 'classically secreted')
].copy().set_index('protein')
fits_filter_scaled_sub = fits_filter[
    (fits_filter['extracellular_origin'] == 'classically secreted')
].copy().set_index('protein')
shared_proteins = fits_filter_sub.index.intersection(fits_filter_scaled_sub.index)
fits_filter_sub = fits_filter_sub.loc[shared_proteins]
fits_filter_scaled_sub = fits_filter_scaled_sub.loc[shared_proteins]

corrs = pd.concat(
    {
        "Unscaled": fits_filter_sub["corr"],
        "RNA-scaled": fits_filter_scaled_sub["corr"],
    },
    names=["method", "protein"],
).rename("corr").reset_index()

plt.figure(figsize=cm_to_inches((4,3)))
sns.swarmplot(data=corrs, x="method", y="corr", s=3)

paired = corrs.pivot(index="protein", columns="method", values="corr").dropna()
test = wilcoxon(paired["Unscaled"], paired["RNA-scaled"])
results = pd.DataFrame([{
    "group1": "Unscaled", "group2": "RNA-scaled",
    "statistic": test.statistic, "pvalue": test.pvalue,
    "n1": len(paired), "n2": len(paired),
}])
annotate_pair_results(plt.gca(), results, order=['Unscaled', 'RNA-scaled'], loc='outside')

diff = paired["RNA-scaled"] - paired["Unscaled"]
n = (diff != 0).sum()
w_plus = diff[diff != 0].abs().rank()[diff[diff != 0] > 0].sum()
w_minus = n * (n + 1) / 2 - w_plus
effect_rb = (w_plus - w_minus) / (n * (n + 1) / 2)
print(f"median Δr = {diff.median():+.3f}, improved in {(diff > 0).mean():.0%} of proteins, "
      f"rank-biserial = {effect_rb:+.2f}, p = {test.pvalue:.2g}")

plt.show()

In [ ]:
if run_t0_flag:
    fig, axes = plt.subplots(1, 2, figsize=cm_to_inches((5, 3)), sharey=True)

    for key_i, key in enumerate(fits_filter_by_weight.keys()):
        ax = axes[key_i]
        fits_filter_sub = fits_filter[
            (fits_filter['extracellular_origin'] == 'classically secreted')
        ].copy().set_index('protein')

        fits_filter_weight = fits_filter_by_weight[key].copy()
        fits_filter_weight_sub = fits_filter_weight[
            (fits_filter_weight['extracellular_origin'] == 'classically secreted')
        ].copy().set_index('protein')
        shared_proteins = fits_filter_sub.index.intersection(fits_filter_weight_sub.index)
        fits_filter_sub = fits_filter_sub.loc[shared_proteins]
        fits_filter_weight_sub = fits_filter_weight_sub.loc[shared_proteins]

        ax.scatter(fits_filter_sub['corr'], fits_filter_weight_sub['corr'], s=5)
        ax.plot([0, 1], [0, 1], ls='--', color='black')

        if key_i == 0:
            ax.set_xlabel('corr (basic model)')
            ax.set_ylabel('corr (time0 add)')
        ax.set_title(f"weight: 0.{key.split('_')[-1]}")

    plt.tight_layout()
    plt.savefig(os.path.join(current_plot_folder, "2_t0_addition_model.png"), dpi=400)
    plt.show()

In [ ]:
setup_plotting_environment()
indicator_proteins = [
    "CXCL9",
    "MMP9",
    "FASLG",
    "RIGI",
    "IL32",
    "TNF",
    "IFNG",
    "CXCL5",
    "IL1B",
    "ACAT1",
    "IL15",
    "TSLP"
    ]

fig, axes = plt.subplots(2, 6, figsize=cm_to_inches((20, 7)))
axes = axes.flatten()

for protein_i, protein in enumerate(indicator_proteins):
    ax = axes[protein_i]
    df = simulated_supernatant[simulated_supernatant['protein'] == protein].copy().reset_index(drop=True)
    df = df.sort_values(by='log10_sim_supernatant')

    x = df['log10_sim_supernatant']
    y = df['log10_raw_conc']
    llod = df['log10_llod'].iloc[0]
    ulod = df['log10_ulod'].iloc[0]

    result, x_fit, y_fit, _, mask_uncensored, mask_for_fit = run_fit(df)
    corr = result['corr']
    slope = result['slope']
    intercept = result['intercept']
    x_min = result['x_min']
    x_max = result['x_max']

    if pd.isna(corr):
        ax.set_title(f"{protein}\nno fit possible")
        ax.scatter(x[mask_uncensored], y[mask_uncensored], s=3, color='tab:blue', rasterized=True)
        ax.scatter(x[~mask_uncensored], y[~mask_uncensored], s=3, color='tab:red', rasterized=True)
    else:
        x_filter = x[mask_uncensored]
        y_filter = y[mask_uncensored]
        ax.set_title(f"{protein}\nr={corr:.2f}, slope={slope:.2f}")
        ax.scatter(x_filter[mask_for_fit], y_filter[mask_for_fit], s=3, color='tab:blue', rasterized=True)
        ax.scatter(x_filter[~mask_for_fit], y_filter[~mask_for_fit], s=3, color='tab:red', rasterized=True)
        ax.scatter(x[~mask_uncensored], y[~mask_uncensored], s=3, color='tab:red', rasterized=True)

        x_range = np.arange(x_min, x_max, 0.1)
        y_fit = intercept + slope * x_range
        ax.plot(x_range, y_fit, color='black', ls='-')
        
    min_y = y.min()
    max_y = y.max()

    if min_y - 0.5 < llod:
        ax.axhline(llod, ls='--', color='black', zorder=0)
    if max_y + 0.5 > ulod:
        ax.axhline(ulod, ls='--', color='black', zorder=0)

    if protein == 'IFNG':
        ax.legend()
    
    if protein_i == 6:
        ax.set_xlabel('log10(simulated supernatant)')
    if protein_i == 6:
        ax.set_ylabel('log10(measured conc. (ng/ml))')

plt.subplots_adjust(hspace=0.6, wspace=0.4)
plt.savefig(os.path.join(current_plot_folder, 'fitting_examples.svg'), dpi=400)
plt.show()

In [ ]:
indicator_proteins = [
    "IL27",
    "EBI3;IL27"
    ]

fig, axes = plt.subplots(1, 2, figsize=cm_to_inches((6, 3)))
axes = axes.flatten()

for protein_i, protein in enumerate(indicator_proteins):
    ax = axes[protein_i]
    df = simulated_supernatant[simulated_supernatant['protein'] == protein].copy().reset_index(drop=True)
    df = df.sort_values(by='log10_sim_supernatant')

    x = df['log10_sim_supernatant']
    y = df['log10_raw_conc']
    llod = df['log10_llod'].iloc[0]
    ulod = df['log10_ulod'].iloc[0]

    result, x_fit, y_fit, _, mask_uncensored, mask_for_fit = run_fit(df)
    corr = result['corr']
    slope = result['slope']
    intercept = result['intercept']
    x_min = result['x_min']
    x_max = result['x_max']

    if pd.isna(corr):
        ax.set_title(f"{protein}\nno fit possible")
        ax.scatter(x[mask_uncensored], y[mask_uncensored], s=3, color='tab:blue')
        ax.scatter(x[~mask_uncensored], y[~mask_uncensored], s=3, color='tab:red')
    else:
        x_filter = x[mask_uncensored]
        y_filter = y[mask_uncensored]
        ax.set_title(f"{protein}\nr={corr:.2f}, slope={slope:.2f}")
        ax.scatter(x_filter[mask_for_fit], y_filter[mask_for_fit], s=3, color='tab:blue')
        ax.scatter(x_filter[~mask_for_fit], y_filter[~mask_for_fit], s=3, color='tab:red')
        ax.scatter(x[~mask_uncensored], y[~mask_uncensored], s=3, color='tab:red')

        x_range = np.arange(x_min, x_max, 0.1)
        y_fit = intercept + slope * x_range
        ax.plot(x_range, y_fit, color='black', ls='--', label='linear fit')

    if protein == 'IFNG':
        ax.legend()
    
    if protein_i == 6:
        ax.set_xlabel('log10(simulated supernatant)')
    if protein_i == 6:
        ax.set_ylabel('log10(measured conc. (ng/ml))')

plt.subplots_adjust(hspace=0.6, wspace=0.4)
plt.savefig(os.path.join(current_plot_folder, 'IL27_predict.svg'))
plt.show()

In [ ]:
treatment_colors = {
    'IL-32-beta': 'tab:red',
    'TNF-alpha': 'tab:blue',
    'IFN-beta': 'tab:green',
    'IL-18': 'tab:orange',
    'IL-6': 'tab:brown',
}

indicator_proteins = [
    "CXCL9",
    "IL6",
    "IFNG",
    "IL1B",
    "TNF",
    ]

fig, axes = plt.subplots(1, 5, figsize=cm_to_inches((13, 2.2)))
axes = axes.flatten()

for protein_i, protein in enumerate(indicator_proteins):
    ax = axes[protein_i]
    df = simulated_supernatant[simulated_supernatant['protein'] == protein].copy().reset_index(drop=True)
    df = df.sort_values(by='log10_sim_supernatant')

    x = df['log10_sim_supernatant']
    y = df['log10_raw_conc']
    llod = df['log10_llod'].iloc[0]
    ulod = df['log10_ulod'].iloc[0]

    result, _, _, _, mask_uncensored, mask_for_fit = run_fit(df)
    corr = result['corr']
    slope = result['slope']
    intercept = result['intercept']
    x_min = result['x_min']
    x_max = result['x_max']

    for treat, color in treatment_colors.items():
        mask = df['cytokine'] == treat
        ax.scatter(x[mask], y[mask], s=3, color=color, label=introduce_greek_letters(treat), zorder=1)
    mask = ~df['cytokine'].isin(treatment_colors.keys())
    ax.scatter(x[mask], y[mask], s=3, color="tab:grey", label='other', zorder=0)

    ax.set_title(
        rf"$\mathbf{{{protein}}}$" + f"\nr={corr:.2f}, slope={slope:.2f}",
        fontsize=6
    )

    x_range = np.arange(x_min, x_max, 0.1)
    y_fit = intercept + slope * x_range
    ax.plot(x_range, y_fit, color='black', ls='-')
    ax.axhline(ulod, ls='--', color='black', zorder=0)
    ax.axhline(llod, ls='--', color='black', zorder=0)

    if protein == 'TNF':
        ax.legend(loc=[1,0], title='perturbation')
    
    if protein_i == 2:
        ax.set_xlabel('log10(simulated supernatant)')
    if protein_i == 0:
        ax.set_ylabel('log10(meas. supernatant (ng/ml))')

    ax.tick_params(axis='both', length=2)

plt.subplots_adjust(hspace=0.4, wspace=0.33)
plt.savefig(os.path.join(current_plot_folder, 'main_text_examples.svg'))
plt.show()

In [ ]:
from matplotlib.ticker import StrMethodFormatter

# EXPLORE DONOR-LEVEL EFFECTS
setup_plotting_environment()
treatment_colors = {
    'IL-32-beta': 'tab:red',
    'TNF-alpha': 'tab:blue',
    'IL-15': 'tab:orange',
    'IFN-beta': 'tab:pink',
    'TGF-beta1': 'tab:brown',
    'C3a': 'black'
}
colors = plt.get_cmap('tab20').colors
donor_colors = {
    'D4': 'tab:blue',
    'D9': 'tab:green',
    'D11': 'tab:red',
    }
score_name = 'NEUTROPHIL'
colorby = 'TREATMENT'
indicator_proteins = ['MMP9', 'IL32', 'IL1B', 'S100A9']#fits_filter[fits_filter['extracellular_origin'] == 'nonclassical release']['protein'].values

fig, axes = plt.subplots(1, 4, figsize=cm_to_inches((16, 4)), sharex=False, sharey=False)
axes = axes.flatten()

for protein_i, protein in enumerate(indicator_proteins):
    ax = axes[protein_i]
    df = simulated_supernatant[simulated_supernatant['protein'] == protein].copy().reset_index(drop=True)
    df = df.sort_values(by='log10_sim_supernatant')

    x = df['log10_sim_supernatant']
    y = df['log10_raw_conc']
    llod = df['log10_llod'].iloc[0]
    ulod = df['log10_ulod'].iloc[0]

    result, _, _, _, mask_uncensored, mask_for_fit = run_fit(df)
    corr = result['corr']
    slope = result['slope']
    intercept = result['intercept']
    x_min = result['x_min']
    x_max = result['x_max']

    if colorby == "TREATMENT":
        for treat, color in treatment_colors.items():
            mask = df['cytokine'] == treat
            ax.scatter(x[mask], y[mask], s=3, color=color, label=introduce_greek_letters(treat), zorder=2)
        mask = ~df['cytokine'].isin(treatment_colors.keys())
        ax.scatter(x[mask], y[mask], s=3, color="tab:grey", label='other', zorder=1, rasterized=True)
    elif colorby == "DONOR":
        for donor, color in donor_colors.items():
            mask = df['donor'] == donor
            ax.scatter(x[mask], y[mask], s=3, color=color, label=donor, zorder=2)
        mask = ~df['donor'].isin(donor_colors.keys())
        ax.scatter(x[mask], y[mask], s=3, color="tab:grey", label='other', zorder=1, rasterized=True)

    elif colorby == "SCORE":
        scores = df.merge(proteomics_scores_unfiltered, on=['treatment', 'donor'])[score_name].values
        data = pd.DataFrame({'x': x, 'y': y, 'score': scores})
        sns.scatterplot(data, x='x', y='y', hue='score', palette='viridis', ax=ax, legend=False, hue_norm=(-4,5),
                        s=8, rasterized=True)

    ax.set_title(f"{protein}\nr={corr:.2f}, slope={slope:.2f}", fontsize=6)

    x_range = np.arange(x_min, x_max, 0.1)
    y_fit = intercept + slope * x_range
    ax.plot(x_range, y_fit, color='black', ls='-')
    
    min_y = y.min()
    max_y = y.max()

    if min_y - 0.5 < llod:
        ax.axhline(llod, ls='--', color='black', zorder=0)
    if max_y + 0.5 > ulod:
        ax.axhline(ulod, ls='--', color='black', zorder=0)

    if protein_i == 1:
        ax.set_xlabel('log10(simulated supernatant)')
    else:
        ax.set_xlabel('')
    if protein_i == 0:
        ax.set_ylabel('log10(meas. supernatant (ng/ml))')
    else:
        ax.set_ylabel('')
    if protein_i == 3:
        if colorby != 'SCORE':
            ax.legend(loc=[1,0])
    
    ax.tick_params(axis='both', length=2)
    ax.yaxis.set_major_formatter(StrMethodFormatter('{x:.1f}'))

if colorby == "SCORE":
    sm = plt.cm.ScalarMappable(
        norm=plt.Normalize(-4, 5),
        cmap='viridis'
    )
    sm.set_array([])

    # [left, bottom, width, height], in figure coordinates
    cax = fig.add_axes([0.92, 0.20, 0.012, 0.60])
    fig.colorbar(
        sm,
        cax=cax,
        label='neutrophil score'
    )

plt.subplots_adjust(hspace=0.7, wspace=0.3)
for f in ['png', 'svg']:
    plt.savefig(os.path.join(current_plot_folder, f'example_scatter.{f}'), dpi=400)
plt.show()

In [ ]:
immune = list(fits_filter[
    (fits_filter['family_level_1'] != 'missing') &
    (fits_filter['extracellular_origin'] == 'classically secreted') &
    (fits_filter['corr'] > 0.6)
    ]['protein'].unique())
non_immune = list(fits_filter[
    (fits_filter['family_level_1'] == 'missing') &
    (fits_filter['extracellular_origin'] == 'classically secreted') &
    (fits_filter['corr'] > 0.6)
    ]['protein'].unique())
fitted_points_all[["protein", "stim", "donor"]] = fitted_points_all["identity"].str.extract(r"^([^_]+)_(.+)_([^_]+)$")
fitted_points_all['ulod'] = np.log10(fitted_points_all['protein'].map(proteomics_analyte_metadata['upper_limit_of_detection']))
fitted_points_all['llod'] = np.log10(fitted_points_all['protein'].map(proteomics_analyte_metadata['lower_limit_of_detection']))

df = fitted_points_all[fitted_points_all['protein'].isin(immune + non_immune)].copy()

condition_names = ["IL-32", "IL-18", "IL-15", "TNF-alpha", "IFN-beta", "TGF-beta"]
for condition_name in condition_names:
    df[f'{condition_name}_condition'] = np.where(
        df['stim'].str.startswith(condition_name),
        condition_name,
        'other'
    )
    df['group'] = np.where(
        df['protein'].isin(immune),
        'cytokine',
        'non-cytokine'
    )
    plt.figure(figsize=cm_to_inches((3.5,3.5)))
    ax = sns.boxplot(
        data=df,
        x='group',
        y='residual',
        hue=f'{condition_name}_condition',
        order=['cytokine', 'non-cytokine'],
        hue_order=['other', condition_name],
        flierprops={'markersize': 2, 'markerfacecolor': 'black'}
    )
    
    ymax = df['residual'].max()
    yrange = df['residual'].max() - df['residual'].min()
    for i, group in enumerate(['cytokine', 'non-cytokine']):
        sub = df[df['group'] == group]

        # use one value per protein (avoids inflation of p values)
        per_protein = sub.dropna(subset=['residual']).groupby(['protein', f'{condition_name}_condition'])['residual'].median()
        per_protein = per_protein.unstack().dropna()
        delta = per_protein[condition_name] - per_protein['other']
        stat, p = stats.wilcoxon(delta)
        ax.text(i, ymax + 0.05 * yrange, f'p={p:.2g}\nΔres={delta.median():.2f}\nn={len(delta)}', ha='center', va='bottom')

    leg = ax.legend(title=None)
    for txt in leg.get_texts():
        txt.set_text(introduce_greek_letters(txt.get_text()))
    plt.xlabel("")
    plt.ylabel('residual in fit')
    plt.tight_layout()
    plt.savefig(os.path.join(current_plot_folder, f'residual_change_{condition_name}.svg'))
    plt.show()

### What's necessary for a fit to be reliable?

This is for diagnosis only.

In [ ]:
setup_plotting_environment()
fits_filter_reliable = fits_filter[
    (fits_filter['corr'] > 0.6) &
    (fits_filter['extracellular_origin'] == 'classically secreted') &
    (fits_filter['family_level_1'] != 'missing') &
    (fits_filter['slope'] > 0.6)
    ].copy()

In [ ]:
threshold_conc = np.log10(0.1)
fits_filter_reliable['threshold_sim'] = (threshold_conc - fits_filter_reliable['intercept']) / fits_filter_reliable['slope']

In [ ]:
df = simulated_supernatant[simulated_supernatant['protein'].isin(fits_filter_reliable['protein'])]
df['high_conc'] = (df['log10_raw_conc'] >= -1) & (df['log10_raw_conc'] > (df['log10_llod'] + np.log10(2)))
bins = np.arange(0.2,3,0.1)

plt.figure(figsize=cm_to_inches((6, 3)))
plt.hist(
    df[df['high_conc']]['log10_sim_supernatant'].values,
    color='tab:red',
    bins=bins,
    alpha=0.5,
    density=False,
    label=f'supernatant conc >= {threshold_conc:.1f}'
)
plt.hist(
    df[~df['high_conc']]['log10_sim_supernatant'].values,
    color='tab:blue',
    bins=bins,
    alpha=0.5,
    density=False,
    label=f'supernatant conc < {threshold_conc:.1f}'
)
plt.axvline(np.log10(3))
plt.legend(loc='upper right')
plt.xlabel('log10(simulated superantant)')
plt.ylabel('data points')
plt.show()

info = []
thresholds = [0, np.log10(3), np.log10(5), np.log10(10), np.log10(20), np.log10(50)]
for threshold in thresholds:
    df_high = df[df['log10_sim_supernatant'] >= threshold]
    df_low = df[df['log10_sim_supernatant'] < threshold]

    fraction_high = len(df_high[df_high['high_conc']]) / len(df_high)
    fraction_high_missed = len(df_low[df_low['high_conc']]) / (len(df_high[df_high['high_conc']]) + len(df_low[df_low['high_conc']]))
    info.append({
        'threshold': threshold,
        'fraction_high': fraction_high,
        'fraction_high_missed': fraction_high_missed
    })
info = pd.DataFrame(info)

plt.figure(figsize=cm_to_inches((4,3)))
plt.scatter(info['threshold'], info['fraction_high'], label='above threshold, supernatant conc >= 0.3', s=3)
plt.scatter(info['threshold'], info['fraction_high_missed'], label='supernatant conc >= 0.3 missed by threshold', s=3)
plt.ylim([0, 1])
plt.show()

### Aggregate prediction statistics

In [ ]:
order = ['classically secreted', 'nonclassical release', 'surface associated', 'intracellular (lysosomal)', 'expected intracellular']

plt.figure(figsize=cm_to_inches((9,4.5)))
plt.axhline(0.5, color='gray', alpha=0.2, lw=0.5)
plt.axhline(0, color='gray', alpha=0.2, lw=0.5)
ax = sns.swarmplot(data=fits_filter, x='extracellular_origin', y='corr', order=order, s=2.75,
    hue_norm=(0, 1),palette='viridis',hue='model_donor_fraction')
ax.legend_.remove()
sm = plt.cm.ScalarMappable(
    norm=plt.Normalize(0, 1),
    cmap='viridis'
)
plt.colorbar(sm, ax=ax, label=r'$R^2$ (donor)')

pairs = [
    ('classically secreted', 'surface associated'),
    ('classically secreted', 'nonclassical release'),
    ('classically secreted', 'intracellular (lysosomal)'),
    ('surface associated', 'expected intracellular'),
    ('intracellular (lysosomal)', 'expected intracellular'),
]

results = test_pairs(fits_filter, x="extracellular_origin", y="corr", pairs=pairs)
results["pvalue_adj"] = multipletests(results["pvalue"], method="holm")[1]
annotate_pair_results(ax, results, order=order, loc='outside', lw=0.75, h=0.03, fmt="{:.2g}", p_col="pvalue_adj")

ax = plt.gca()
labels = [label.get_text().replace(" ", "\n") for label in ax.get_xticklabels()]
ax.set_xticklabels(labels)
plt.tick_params(axis='x', rotation=0, length=0)

plt.xlabel("")
plt.ylim(-0.5, 1.)
plt.yticks([-0.5, 0, 0.5, 1.0])

plt.ylabel("Pearson r\n(simulated, measured)")
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, 'swarmplot_r_secretion_type.svg'))
plt.show()

In [ ]:
labels = ["S100A8", "S100A9", "LGALS3"]

fig, axes = plt.subplots(1, 1, figsize=cm_to_inches((3,1.75)), sharey=True)
for release_i, release_mode in enumerate(['nonclassical release']):
    ax = axes
    sub = fits_filter[fits_filter['extracellular_origin'] == release_mode]
    sns.scatterplot(sub, x='NEUTROPHIL_score_fraction', y='corr', s=5, ax=ax, color='black')
    r = np.corrcoef(sub['NEUTROPHIL_score_fraction'], sub['corr'])[0, 1]
    for label in labels:
        rows = sub[sub['protein'] == label]
        for _, row in rows.iterrows():
            ax.text(row['NEUTROPHIL_score_fraction'],row['corr'],label,fontsize=5.5,ha='left',va='bottom')
    if release_i == 0:
        ax.set_xlabel(r"$R^2$ (neutrophil score)")
        ax.set_ylabel("Pearson r\n(sim., meas.)")
    else:
        ax.set_xlabel("")
        ax.set_ylabel("")
    ax.set_title(release_mode, fontsize=6)
    ax.set_ylim(-0.25, 1)
    ax.set_xlim(-0.05, 1.05)
    ax.text(0.95, 0.8, f"r={r:.2f}", ha='right', transform=ax.transAxes)
plt.savefig(os.path.join(current_plot_folder, 'neutrophil_score_variance_fraction.svg'))
plt.show()

In [ ]:
score_labels = {'INTRACELLULAR_score_fraction': 'intracellular score', 'PLATELET_score_fraction': 'platelet score', 'NEUTROPHIL_score_fraction': 'neutrophil score', 'model_donor_fraction': 'donor'}

for score in ['INTRACELLULAR_score_fraction', 'PLATELET_score_fraction', 'NEUTROPHIL_score_fraction', 'model_donor_fraction']:
    fig, axes = plt.subplots(1, 2, figsize=cm_to_inches((6,3)), sharey=True)
    for release_i, release_mode in enumerate(['classically secreted', 'nonclassical release']):
        ax = axes[release_i]
        sub = fits_filter[fits_filter['extracellular_origin'] == release_mode]
        sns.scatterplot(sub, x=score, y='corr', s=5, ax=ax, color='black')
        r = np.corrcoef(sub[score], sub['corr'])[0, 1]
        if release_i == 0:
            ax.set_xlabel(rf"$R^2$ ({score_labels[score]})")
            ax.set_ylabel("Pearson r\n(sim., meas.)")
        else:
            ax.set_xlabel("")
            ax.set_ylabel("")
        ax.set_title(release_mode, fontsize=6)
        ax.set_ylim(-0.25, 1)
        ax.text(0.95, 0.8, f"r={r:.2f}", ha='right', transform=ax.transAxes)
    plt.tight_layout()
    for f in ['svg', 'png']:
        plt.savefig(os.path.join(current_plot_folder, f'scatter_{score.replace("/", "_")}.{f}'), dpi=400)
    plt.show()

In [ ]:
df = fits_filter[
    (fits_filter['corr'] > 0.6) &
    (fits_filter['extracellular_origin'].isin(['classically secreted'])) &
    (fits_filter['n_uncensored'] > 60)
    ].copy()

plt.figure(figsize=cm_to_inches((4,3)))
median = df['slope'].median()
plt.hist(df['slope'], bins=np.arange(0, 2.1, 0.1), edgecolor='black')
plt.axvline(median, color='red')
plt.text(2.1, 6, f"median={median:.2f}", ha='right', color='red')
labels = [label.get_text().replace(" ", "\n") for label in ax.get_xticklabels()]
ax.set_xticklabels(labels)
plt.tick_params(axis='x', rotation=0)

plt.xlabel("slope")
# plt.ylim(-0.6, 1.05)
plt.title("fits with r > 0.6, classical secretion", fontsize=6)
plt.ylabel("count")
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, 'cytokine_slopes.svg'))
plt.show()

In [ ]:
setup_plotting_environment()

In [ ]:
fits_cytokine = fits_filter[fits_filter['family_level_1'] != 'missing'].copy()
fits_cytokine['family_level_1'] = fits_cytokine['family_level_1'].str.replace('complement', 'other')

order = ['interleukin', 'interferon', 'chemokine', 'TNF family', 'growth factor', 'other']

plt.figure(figsize=cm_to_inches((6,3)))
ax = sns.swarmplot(data=fits_cytokine, x='family_level_1', y='corr', s=3, hue='extracellular_origin', order=order)
ax = plt.gca()
plt.tick_params(axis='x', rotation=90)

annotate_manual = ['IL32', 'TNF', 'FASLG', 'IFNG', 'IL1B', 'VEGFA', 'CSF3', 'BMP6']
for p in annotate_manual:
    row = fits_cytokine.loc[fits_cytokine['protein'] == p].iloc[0]
    x = order.index(row['family_level_1'])

    ax.text(
        x,
        row['corr'] + 0.05,
        p,
        fontdict={'size': 5.5}
    ) 

plt.xlabel("")
plt.ylim(-0.2, 1.05)
plt.ylabel("Pearson r\n(simulated, measured)")
plt.legend(loc=[1,0])
plt.tick_params(axis='x', length=0)
#plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, 'swarmplot_r_family.svg'))
plt.show()

In [ ]:
df = fits_cytokine[
    (fits_cytokine['extracellular_origin'].isin(['classically secreted', 'nonclassical release'])) &
    (fits_cytokine['n_uncensored'] > 40)
    ].copy()

plt.figure(figsize=cm_to_inches((4,3)))
plt.hist(df['corr'], bins=np.arange(0, 1.1, 0.1), edgecolor='black')

median = df['corr'].median()
plt.axvline(median, color='red')
plt.text(0.05, 0.7, f"median\n{median:.2f}", ha='left', color='red', transform=plt.gca().transAxes)

labels = [label.get_text().replace(" ", "\n") for label in ax.get_xticklabels()]
ax.set_xticklabels(labels)
plt.tick_params(axis='x', rotation=0)

plt.xlabel("corr")
plt.title("classical & non-classical secretion", fontsize=6)
plt.ylabel("count")
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, 'cytokine_corr.svg'))
plt.show()

In [ ]:
df = fits_cytokine[
    (fits_cytokine['corr'] > 0.5) &
    (fits_cytokine['extracellular_origin'].isin(['classically secreted', 'nonclassical release'])) &
    (fits_cytokine['n_uncensored'] > 40)
    ].copy()

plt.figure(figsize=cm_to_inches((4,3)))
plt.hist(df['slope'], bins=np.arange(0, 2.1, 0.2), edgecolor='black')

median = df['slope'].median()
plt.axvline(median, color='red')
plt.text(0.05, 0.7, f"median\n{median:.2f}", ha='left', color='red', transform=plt.gca().transAxes)

labels = [label.get_text().replace(" ", "\n") for label in ax.get_xticklabels()]   

plt.xlabel("slope")
plt.title("r > 0.5, classical & non-classical secretion", fontsize=6)
plt.ylabel("count")
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, 'cytokine_slopes.svg'))
plt.show()

# 3 - Secondary effect analysis

In [ ]:
current_plot_folder = os.path.join(plot_folder, "3_secondary_effects")
os.makedirs(current_plot_folder, exist_ok=True)

In [ ]:
from scipy.stats import gaussian_kde
def density_scatter(ax, x, y, s=2):
    x = np.asarray(x)
    y = np.asarray(y)

    xy = np.vstack([x, y])
    density = gaussian_kde(xy)(xy)
    density = np.minimum(density, np.quantile(density, 0.95))
    density_plot = np.log10(density)

    order = np.argsort(density_plot)
    scatter = ax.scatter(x[order], y[order], c=density_plot[order], s=s, edgecolor="none", rasterized=True)

    return scatter

def fit_slope(x, y):
    """OLS slope and intercept of y on x across genes present in both."""
    d = pd.concat({"x": x, "y": y}, axis=1).dropna()
    slope, intercept = np.polyfit(d["x"], d["y"], 1)
    return slope, intercept

def add_density_panel(ax, x, y, title, beta_label, title_font=8):
    density_scatter(ax, x, y)

    slope, intercept = fit_slope(x, y)
    r = np.corrcoef(x,y)[0,1]
    x_range = np.linspace(x.min(), x.max(), 200)

    ax.plot(x_range, slope * x_range + intercept, color="black", linewidth=0.75)
    beta_label = beta_label.replace("+", r"\!+\!")
    ax.text(
        0.05, 0.8,
        rf"$\beta_{{\mathrm{{{beta_label}}}}}$={slope:.2f}", fontsize=6,
        transform=ax.transAxes, bbox=dict(facecolor='white', alpha=0.5, edgecolor='none', pad=1),
    )
    ax.tick_params(axis='both', length=2)
    if title is not None:
        ax.set_title(title, fontsize=title_font)

In [ ]:
GENE_THRESHOLD = 30

padj = edger_df_padj_main.fillna(1) > 0.1
log2fc = edger_df_log2FC_main.fillna(0).copy()
log2fc[padj] = np.nan
log2fc = log2fc[log2fc.index.isin(protein_coding_genes)]

gene_signatures = []
for cell, treat in log2fc.columns:
    sub = log2fc[cell][treat].dropna()
    sub = sub[sub.abs() > 0.5].copy()
    if len(sub) < GENE_THRESHOLD:
        continue
    gene_signatures.append({
        "cell": cell,
        "treat": treat,
        "signature": list(sub.index)
    })
    
gene_signatures = pd.DataFrame(gene_signatures)
gene_signatures = gene_signatures.set_index(['cell', 'treat'])['signature']

In [ ]:
def get_comparison_x(cell_type, main_cytokine, comp_cytokine, comparison_donors):
    """Mean main-screen log2FC of comp_cytokine (matched donors) restricted to signature genes shared across conditions."""
    if (cell_type, comp_cytokine) not in gene_signatures.index:
        return None

    # use matching donors
    x = raw_log2fc_values_by_donor_main[cell_type][comp_cytokine][comparison_donors]
    x = x.loc[pd.isna(x).sum(axis=1) < 3].mean(axis=1) # at least two donors
    x = x[x.index.isin(gene_signatures[cell_type][comp_cytokine])]

    # use the same genes for all conditions so that slopes are comparable
    conditions = [main_cytokine]
    for antibody in AB_CONDITIONS:
        if f"{main_cytokine}_{antibody}" in ALL_TREATMENTS:
            conditions += [antibody, f"{main_cytokine}_{antibody}"]
    for condition in conditions:
        genes = raw_log2fc_values_by_donor[cell_type][condition][comparison_donors].dropna().index
        x = x[x.index.isin(genes)]

    x = x.dropna()
    return x if len(x) >= GENE_THRESHOLD else None

comparison_donors_by_cyto = {
    "IL-15": ['D5', 'D7', 'D9', 'D12'],
    "IL-32-beta": ['D5', 'D7', 'D9', 'D12'],
}

In [ ]:
pairs = [
    {'cell_type': "CD14 Mono", 'comp_cytokine': 'IFN-gamma', 'main_cytokine': 'IL-15', 'antibodies': ['AB-IL-6', 'AB-TNF-alpha', 'AB-IFN-gamma']},
    {'cell_type': "CD14 Mono", 'comp_cytokine': 'IL-6', 'main_cytokine': 'IL-15', 'antibodies': ['AB-IL-6', 'AB-TNF-alpha', 'AB-IFN-gamma']},
    {'cell_type': "CD4 T cell", 'comp_cytokine': 'IL-10', 'main_cytokine': 'IL-32-beta', 'antibodies': ['AB-IL-6', 'IL-1Ra', 'AB-IL-10']},
]

for pair in pairs:
    cell_type = pair['cell_type']
    comp_cytokine = pair['comp_cytokine']
    main_cytokine = pair['main_cytokine']
    antibodies = pair['antibodies']
    
    curr_ax = 0
    comparison_donors = comparison_donors_by_cyto[main_cytokine]
    y = raw_log2fc_values_by_donor[cell_type][main_cytokine][comparison_donors].dropna()

    fig, axes = plt.subplots(len(comparison_donors),7,sharex=True, sharey=True, figsize=cm_to_inches((14, 1 + 1.4 * len(comparison_donors))))
    axes = axes.flatten()

    x = get_comparison_x(cell_type, main_cytokine, comp_cytokine, comparison_donors)
    for comparison_i, comparison_donor in enumerate(comparison_donors):
        y_reg = y[comparison_donor]
        x_reg, y_reg = get_common_index(x.copy(), y_reg)

        add_density_panel(
            axes[curr_ax],
            x_reg,
            y_reg,
            title = format_treatment(main_cytokine) if comparison_i == 0 else None,
            beta_label = main_cytokine.replace("-", ""),
            title_font=6.5
        )
        curr_ax+=1

        for i in range(len(antibodies)):
            antibody = antibodies[i]
            y_ab = raw_log2fc_values_by_donor[cell_type][antibody][comparison_donor]
            y_treat = raw_log2fc_values_by_donor[cell_type][f"{main_cytokine}_{antibody}"][comparison_donor]
            x_ab, y_ab = get_common_index(x.copy(), y_ab)
            x_treat, y_treat = get_common_index(x.copy(), y_treat)

            add_density_panel(
                axes[curr_ax],
                x_ab,
                y_ab,
                title = format_treatment(antibody) if comparison_i == 0 else None,
                beta_label = "AB",
                title_font=6.5
            )
            curr_ax+=1

            add_density_panel(
                axes[curr_ax],
                x_treat,
                y_treat,
                title = format_treatment(f"{main_cytokine}_{antibody}")  if comparison_i == 0 else None,
                beta_label = main_cytokine.replace("-", "") + "+AB",
                title_font=6.5
            )
            curr_ax+=1

        axes[curr_ax-1].text(1.2,  0.5, comparison_donor, fontsize=6.5, transform=axes[curr_ax-1].transAxes )
        
    axes[7].set_ylabel('log2FC treatment')
    axes[-4].set_xlabel(f'log2FC {format_treatment(comp_cytokine)}')
    plt.subplots_adjust(hspace=0.1, wspace=0.1)
    plt.savefig(os.path.join(current_plot_folder, f"AB_impact_scatter_{cell_type}_{main_cytokine}_{comp_cytokine}.svg"), dpi=400)
    plt.show()

In [ ]:
pairs = [
    {'cell_type': "CD14 Mono", 'comp_cytokine': 'IFN-gamma', 'main_cytokine': 'IL-15', 'antibodies': ['AB-IL-6', 'AB-TNF-alpha', 'AB-IFN-gamma'], 'plot_donor': 'D5'},
    {'cell_type': "CD14 Mono", 'comp_cytokine': 'IL-6', 'main_cytokine': 'IL-15', 'antibodies': ['AB-IL-6', 'AB-TNF-alpha', 'AB-IFN-gamma'], 'plot_donor': 'D5'},
    {'cell_type': "CD4 T cell", 'comp_cytokine': 'IL-10', 'main_cytokine': 'IL-32-beta', 'antibodies': ['AB-IL-6', 'IL-1Ra', 'AB-IL-10'], 'plot_donor': 'D3'},
]

for pair in pairs:
    setup_plotting_environment()
    fig, axes = plt.subplots(1,4, sharex=True, sharey=True, figsize=cm_to_inches((6.5,1.6)))
    axes = axes.flatten()
    
    cell_type = pair['cell_type']
    comp_cytokine = pair['comp_cytokine']
    main_cytokine = pair['main_cytokine']
    antibodies = pair['antibodies']
    plot_donor = pair['plot_donor']
    comparison_donors = comparison_donors_by_cyto[main_cytokine]
    
    curr_ax = 0
    x = get_comparison_x(cell_type, main_cytokine, comp_cytokine, comparison_donors)

    y_reg = raw_log2fc_values_by_donor[cell_type][main_cytokine][plot_donor]
    x_reg, y_reg = get_common_index(x.copy(), y_reg)

    add_density_panel(
        axes[curr_ax],
        x_reg,
        y_reg,
        title = "none",
        beta_label = "",
        title_font=6.5 # main_cytokine.replace("-", ""),
    )
    curr_ax+=1

    for i in range(len(antibodies)):
        antibody = antibodies[i]
        y_ab = raw_log2fc_values_by_donor[cell_type][antibody][plot_donor]
        y_treat = raw_log2fc_values_by_donor[cell_type][f"{main_cytokine}_{antibody}"][plot_donor]
        x_ab, y_ab = get_common_index(x.copy(), y_ab)
        x_treat, y_treat = get_common_index(x.copy(), y_treat)

        add_density_panel(
            axes[curr_ax],
            x_treat,
            y_treat,
            title = format_treatment(antibody),  # if comparison_i == 0 else None,
            beta_label = "",
            title_font=6
        )
        curr_ax+=1
        
    axes[0].set_ylabel('log2FC treatment')
    axes[1].set_xlabel(f'log2FC {format_treatment(comp_cytokine)}')

    for ax in axes:
        ax.tick_params(axis='both', length=2.5)

    plt.subplots_adjust(hspace=0.1, wspace=0.1)
    plt.savefig(os.path.join(current_plot_folder, f"AB_impact_scatter_main_fig_{cell_type}_{main_cytokine}_{comp_cytokine}_{plot_donor}.svg"), dpi=400)
    plt.show()

In [ ]:
def get_contrast_df(cell_type, main_cytokine, comp_cytokine, comparison_donors):
    all_rs = []
    x = get_comparison_x(cell_type, main_cytokine, comp_cytokine, comparison_donors)
    if x is None:
        return None
    y = raw_log2fc_values_by_donor[cell_type][main_cytokine][comparison_donors].dropna()

    for donor in y.columns:
        x_reg, y_reg = get_common_index(x.copy(), y[donor])
        r = np.corrcoef(x_reg, y_reg)[0, 1]
        beta, _ = fit_slope(x_reg, y_reg)
        all_rs.append({
            'r': r,
            'donor': donor,
            'cytokine': main_cytokine,
            'comp_cytokine': comp_cytokine,
            'antibody': 'none',
            'beta': beta,
            'cell_type': cell_type,
        })

    present_antibodies = []
    for antibody in AB_CONDITIONS:
        treatment = f"{main_cytokine}_{antibody}"
        if not treatment in ALL_TREATMENTS:
            continue
        present_antibodies.append(antibody)
        y_ab = raw_log2fc_values_by_donor[cell_type][antibody][comparison_donors].dropna()
        y_treat = raw_log2fc_values_by_donor[cell_type][treatment][comparison_donors].dropna()
        for donor in y.columns:
            x_ab, y_ab_donor = get_common_index(x.copy(), y_ab[donor])
            r = np.corrcoef(x_ab, y_ab_donor)[0, 1]
            beta, _ = fit_slope(x_ab, y_ab_donor)
            all_rs.append({
                'r': r,
                'donor': donor,
                'cytokine': 'none',
                'comp_cytokine': comp_cytokine,
                'antibody': antibody,
                'beta': beta,
                'cell_type': cell_type,
            })

            x_treat, y_treat_donor = get_common_index(x.copy(), y_treat[donor])
            r = np.corrcoef(x_treat, y_treat_donor)[0, 1]
            beta, _ = fit_slope(x_treat, y_treat_donor)
            all_rs.append({
                'r': r,
                'donor': donor,
                'cytokine': main_cytokine,
                'comp_cytokine': comp_cytokine,
                'antibody': antibody,
                'beta': beta,
                'cell_type': cell_type,
            })

    all_rs = pd.DataFrame(all_rs)
    # process the slope
    slope_df = all_rs.pivot_table(index='donor', columns=['cytokine', 'antibody'], values='beta')
    # subtract cytokine w/o antibody
    contrast_df = slope_df[main_cytokine][present_antibodies].subtract(slope_df[main_cytokine]['none'], axis=0)
    
    # subtract antibody w/o cytokine
    contrast_df = contrast_df.subtract(slope_df['none'][present_antibodies], axis=0)
    contrast_df = contrast_df.reset_index().melt(id_vars='donor')
    contrast_df['cell_type'] = cell_type
    contrast_df['main_cytokine'] = main_cytokine
    contrast_df['comp_cytokine'] = comp_cytokine

    return contrast_df, slope_df, all_rs, present_antibodies

In [ ]:
pairs = [
    ('CD14 Mono', 'IL-15', 'IFN-gamma',),
    ('CD14 Mono', 'IL-15', 'TNF-alpha'),
    ('CD14 Mono', 'IL-15', 'IL-6'),
    ('CD4 T cell', 'IL-15', 'IL-6'),
    ('CD4 T cell', 'IL-32-beta', 'IL-10')
]

contrast_dfs = []
all_rs_df = []
for cell_type, main_cytokine, comp_cytokine in pairs:
    comparison_donors = comparison_donors_by_cyto[main_cytokine]
    try:
        contrast_df, slope_df, all_rs, present_antibodies = get_contrast_df(cell_type, main_cytokine, comp_cytokine, comparison_donors)
    except:
        continue
    contrast_dfs.append(contrast_df)
    all_rs_df.append(all_rs)

    order = ['none']
    for ab in present_antibodies:
        order.append(ab)
    slope_df = slope_df.loc[comparison_donors, main_cytokine][order]
    slope_df.columns = [format_treatment(c) for c in slope_df.columns]

    plt.figure(figsize=cm_to_inches((3.7,3)))

    max_val = slope_df.max(axis=None)
    min_val = slope_df.min(axis=None)
    max_val = max(abs(max_val), abs(min_val))
    max_val = max(max_val, 0.8)

    sns.heatmap(slope_df.T, square=False, cbar_kws={
        'label': r'$\beta$'+introduce_greek_letters(f" ({comp_cytokine})"), 'shrink': 1},
                cmap='coolwarm', vmin=-max_val, vmax=max_val)
    cbar = plt.gca().collections[0].colorbar
    cbar.ax.tick_params(length=0)
    plt.text(0.5, 1.1, introduce_greek_letters(f"{main_cytokine} in {cell_type}"), transform=plt.gca().transAxes, fontsize=6.5, ha='center' )
    plt.tick_params(axis='x', rotation=90)
    plt.tick_params(axis='both', length=0)
    plt.xlabel("")
    plt.tight_layout()
    plt.savefig(os.path.join(current_plot_folder, f'Betas_{cell_type}_{main_cytokine}_vs_{comp_cytokine}.svg'))
    plt.show()

contrast_dfs = pd.concat(contrast_dfs)
all_rs_df = pd.concat(all_rs_df)

In [ ]:
main_cytokines = ['IL-15', 'IL-32-beta']
comp_cytokines = ['IFN-gamma', 'IL-1-beta', 'IL-10', 'IL-6', 'TNF-alpha']

contrast_dfs = []
slope_dfs = []
for cell_type in ['B cell', 'CD4 T cell', 'NK CD56low', 'CD14 Mono']:
    for main_cytokine in main_cytokines:
        for comp_cytokine in comp_cytokines:
            try:
                contrast_df, slope_df, all_rs, present_antibodies = get_contrast_df(
                    cell_type, main_cytokine, comp_cytokine, comparison_donors_by_cyto[main_cytokine]
                    )
            except:
                continue
            contrast_dfs.append(contrast_df)
            slope_dfs.append(slope_df)
contrast_dfs = pd.concat(contrast_dfs)
slope_dfs = pd.concat(slope_dfs)

# interferon donors with antibody conditions: shown in plots, not used for statistics
contrast_dfs_ifn = []
for cell_type in ['B cell', 'CD4 T cell', 'NK CD56low', 'CD14 Mono']:
    for main_cytokine in main_cytokines:
        for comp_cytokine in comp_cytokines:
            result = get_contrast_df(cell_type, main_cytokine, comp_cytokine, ['D3', 'D4'])
            if result is None:
                continue
            contrast_dfs_ifn.append(result[0])
contrast_dfs_ifn = pd.concat(contrast_dfs_ifn)

In [ ]:
from matplotlib.ticker import MultipleLocator, MaxNLocator
def compact_pvalue(p):
    if p < 0.01:
        return "<.01"
    return f"{p:.2f}".lstrip("0")

comps_per_main = {
    "IL-15": ["IL-6", 'TNF-alpha', 'IFN-gamma'],
    "IL-32-beta": ["IL-6", 'IL-1-beta', 'IL-10'],
}
stats_test=True
for main in ['IL-15', 'IL-32-beta']:
    contrast_df_sub = contrast_dfs[
        (contrast_dfs['main_cytokine'] == main) &
        (contrast_dfs['comp_cytokine'].isin(comps_per_main[main]))
    ].copy()
    no_plots = len(contrast_df_sub[['cell_type', 'comp_cytokine']].value_counts())
    fig, axes = plt.subplots(1, no_plots, figsize=cm_to_inches((20, 2.5)), sharey=False)
    comp_i = 0

    for cell in ['B cell', 'CD4 T cell', 'NK CD56low', 'CD14 Mono']:
        sub_cell = contrast_df_sub[contrast_df_sub['cell_type'] == cell]
        comps = [comp for comp in comps_per_main[main] if comp in sub_cell['comp_cytokine'].values]
        for comp in comps:
            sub_cell_comp = sub_cell[
                (sub_cell['comp_cytokine'] == comp)
            ].copy()

            ax = axes[comp_i]
            sub_cell_comp['antibody'] = sub_cell_comp['antibody'].apply(format_treatment)
            plot_order = list(sub_cell_comp["antibody"].unique())
            
            sns.barplot(
                data=sub_cell_comp,
                x="antibody",
                y="value",
                errorbar="sd",
                capsize=0.15,
                err_kws={'linewidth': 1, 'color': 'black'},
                edgecolor="black",
                facecolor=CELL_TYPE_COLORS[cell],
                ax=ax,
                order=plot_order,
            )
            sns.stripplot(
                data=sub_cell_comp,
                x="antibody",
                y="value",
                dodge=True,
                jitter=0.2,
                size=2.5,
                color='black',
                edgecolor="black",
                linewidth=0.3,
                ax=ax,
                order=plot_order,
            )

            ifn = contrast_dfs_ifn[
                (contrast_dfs_ifn['main_cytokine'] == main) &
                (contrast_dfs_ifn['cell_type'] == cell) &
                (contrast_dfs_ifn['comp_cytokine'] == comp)
            ].copy()
            if len(ifn) > 0:
                ifn['antibody'] = ifn['antibody'].apply(format_treatment)
                sns.stripplot(data=ifn, x="antibody", y="value", order=plot_order, jitter=0.2, size=2.5,
                              color='tab:red', edgecolor="black", linewidth=0.3, ax=ax)

            if stats_test:
                test_results = (
                    sub_cell_comp
                    .groupby("antibody", sort=False)["value"]
                    .apply(lambda values: stats.ttest_1samp(values.dropna(), popmean=0,).pvalue)
                    .rename("pvalue")
                    .reset_index()
                    .rename(columns={"antibody": "group2"})
                )

                annotate_pvalues(
                    ax=ax,
                    results=test_results,
                    order=plot_order,
                    group_col="group2",
                    p_col="pvalue",
                    fmt=compact_pvalue,
                )

            if comp_i == 0:
                ax.set_ylabel(
                    rf'$\beta_{{\mathrm{{{format_treatment(main).replace("-", "")}\!+\!AB}}}}'
                    rf'-\beta_{{\mathrm{{{format_treatment(main).replace("-", "")}}}}}'
                    rf'-\beta_{{\mathrm{{AB}}}}$',
                    fontsize=6
                )
            else:
                ax.set_ylabel('')

            ax.set_title(f"comparison to\n{format_treatment(comp)}", y=1.1, fontsize=6)
            ax.set_xlabel('')
            ax.tick_params(axis='x', length=0, rotation=90)
            ax.tick_params(axis='y', length=0, pad=1)
            ax.yaxis.set_major_locator(MaxNLocator(nbins=4, steps=[1, 2, 5, 10]))
            comp_i += 1

    plt.suptitle(format_treatment(f"{main} treatment"), fontsize=7, y=1.1)
    plt.subplots_adjust(wspace=0.5, top=0.7)
    plt.savefig(os.path.join(current_plot_folder, f'ABimpact_{main}.svg'))
    plt.show()

In [ ]:
contrast_df_sub = contrast_dfs[
    (contrast_dfs['cell_type'] == 'CD14 Mono') &
    (contrast_dfs['main_cytokine'] == "IL-15") &
    (contrast_dfs['comp_cytokine'].isin(['IFN-gamma', 'TNF-alpha', 'IL-6']))
].copy()
contrast_df_sub = contrast_df_sub[
    contrast_df_sub.apply(lambda row: row['comp_cytokine'] in row['antibody'], axis=1)
]
main = "IL-15"

fig, ax = plt.subplots(figsize=cm_to_inches((1.5,1.8)))
setup_plotting_environment()
contrast_df_sub['antibody'] = contrast_df_sub['antibody'].apply(format_treatment)
plot_order = ['α-IFN-γ', 'α-TNF-α', 'α-IL-6']

sns.barplot(
    data=contrast_df_sub,
    x="antibody",
    y="value",
    errorbar="sd",
    capsize=0.15,
    linewidth=0.75,
    err_kws={'linewidth': 0.75, 'color': 'black'},
    edgecolor="black",
    ax=ax,
    order=plot_order,
)
sns.stripplot(
    data=contrast_df_sub,
    x="antibody",
    y="value",
    dodge=True,
    jitter=0.2,
    size=2.5,
    color='black',
    edgecolor="black",
    linewidth=0.3,
    ax=ax,
    order=plot_order,
)

# ifn = contrast_dfs_ifn[
#     (contrast_dfs_ifn['cell_type'] == 'CD14 Mono') &
#     (contrast_dfs_ifn['main_cytokine'] == main) &
#     (contrast_dfs_ifn['comp_cytokine'].isin(['IFN-gamma', 'TNF-alpha', 'IL-6']))
# ].copy()
# ifn = ifn[ifn.apply(lambda row: row['comp_cytokine'] in row['antibody'], axis=1)]
# ifn['antibody'] = ifn['antibody'].apply(format_treatment)
# sns.stripplot(data=ifn, x="antibody", y="value", order=plot_order, jitter=0.2, size=2.5,
#     color='tab:red', edgecolor="black", linewidth=0.3, ax=ax)

test_results = contrast_df_sub.groupby("antibody", sort=False)["value"].apply(lambda values: stats.ttest_1samp(values.dropna(), popmean=0,).pvalue)
test_results = test_results.rename("pvalue").reset_index().rename(columns={"antibody": "group2"})

annotate_pvalues(ax=ax, results=test_results, order=plot_order, group_col="group2", p_col="pvalue", fmt=compact_pvalue)
ax.set_ylabel(
    rf'$\beta_{{\mathrm{{{format_treatment(main).replace("-", "")}\!+\!AB}}}}'
    rf'-\beta_{{\mathrm{{{format_treatment(main).replace("-", "")}}}}}'
    rf'-\beta_{{\mathrm{{AB}}}}$'
    f'\n(CD14 Mono)',
    fontsize=6
)

# # fraction of the comparison-like response removed by the antibody (only where beta(IL-15) is positive in all donors)
# for i, (comp, antibody) in enumerate([('IFN-gamma', 'AB-IFN-gamma'), ('TNF-alpha', 'AB-TNF-alpha'), ('IL-6', 'AB-IL-6')]):
#     contrast_df, slope_df, _, _ = get_contrast_df('CD14 Mono', main, comp, comparison_donors_by_cyto[main])
#     beta_main = slope_df[main]['none']
#     delta = contrast_df[contrast_df['antibody'] == antibody].set_index('donor')['value']
#     label = f"{100 * (-delta / beta_main).mean():.0f}%" if (beta_main > 0).all() else "–"
#     ax.text(i, 0.02, label, transform=ax.get_xaxis_transform(), ha='center', va='bottom', fontsize=5.5)

for spine in ax.spines.values():
    spine.set_linewidth(0.75)
ax.set_xlabel('')
ax.tick_params(axis='x', length=0, rotation=45)
ax.tick_params(axis='y', length=2)
ax.yaxis.set_major_locator(MaxNLocator(nbins=4, steps=[1, 2, 5, 10]))
plt.savefig(os.path.join(current_plot_folder, 'antibody_impact_main.svg'))
plt.show()

In [ ]:
for cell, main, comp in pairs:
    comp_gene = cytokine_info.loc[comp, 'gene']
    comparison_donors = comparison_donors_by_cyto[main]

    present_antibodies = []
    for antibody in AB_CONDITIONS:
        treatment = f"{main}_{antibody}"
        if not treatment in ALL_TREATMENTS:
            continue
        present_antibodies.append(antibody)

    # simulated supernatant as in 2.3, without excluding antibody conditions (RNA only, no protein measurement)
    target_row = target_table.set_index('protein', drop=False).loc[comp_gene]
    sim, _ = calculate_supernatant(target_row, count_df_norm_needed, cell_fraction_vector_scaled)
    expression_df = sim.pivot_table(index='donor', columns='treatment', values='sim_supernatant')

    order = [main]
    for ab in present_antibodies:
        order.append(main + "_" + ab)
    expression_df = np.log2(expression_df.loc[comparison_donors, order])
    expression_df.columns = [format_treatment(c.replace(main + "_", "").replace(main, "none")) for c in expression_df.columns]
    expression_df = expression_df.sub(expression_df['none'], axis=0)

    plt.figure(figsize=cm_to_inches((3.7,2.65)))

    sns.heatmap(expression_df.T, cbar_kws={'label': f"log2FC\n{format_treatment(comp)} sim. super."},
    center=0, cmap='icefire') #annot=True, fmt=".1f", annot_kws={'size': 5.5})
    cbar = plt.gca().collections[0].colorbar
    cbar.ax.tick_params(length=0)

    plt.tick_params(axis='both', length=0)
    plt.tick_params(axis='x', rotation=90)
    plt.tick_params(axis='y', rotation=0)
    plt.xlabel("")
    plt.tight_layout()
    plt.savefig(os.path.join(current_plot_folder, f'Total_expression_{main}_vs_{comp}.svg'))
    plt.show()

In [ ]:
contrast = contrast_dfs[
    (contrast_dfs['antibody'] == 'AB-TNF-alpha') &
    (contrast_dfs['main_cytokine'] == 'IL-15') &
    (contrast_dfs['comp_cytokine'] == 'IFN-gamma') &
    (contrast_dfs['cell_type'] == 'CD14 Mono')
].set_index('donor')['value']
sub = simulated_supernatant[
    simulated_supernatant['treatment'].isin(['IL-15', 'IL-15_AB-TNF-alpha', 'PBS']) &
    (simulated_supernatant['protein'] == 'IFNG')
].copy()
main = 'IL-15'
comp_donors = comparison_donors_by_cyto['IL-15']

# unused
rna = sub.pivot_table(index='donor', columns='treatment', values='log10_sim_supernatant').loc[comp_donors]
rna['contrast_val'] = contrast
rna['log2fc_diff'] = np.log2((10**rna['IL-15_AB-TNF-alpha']) / (10**rna['IL-15']))

protein = sub.pivot_table(index='donor', columns='treatment', values='log10_raw_conc').loc[comp_donors]
protein['contrast_val'] = contrast
protein['log2fc_diff'] = np.log2((10**protein['IL-15_AB-TNF-alpha']) / (10**protein['IL-15']))

fig, ax = plt.subplots(figsize=cm_to_inches((5,4)))

ax.scatter(protein['IL-15'], protein['contrast_val'], color='tab:blue', s=10, label='no antibody')
ax.scatter(protein['IL-15_AB-TNF-alpha'], protein['contrast_val'], color='tab:red', s=10, label='α-TNF-α')

for _, row in protein.iterrows():
    x1 = row['IL-15_AB-TNF-alpha']
    x2 = row['IL-15']
    shift = 0.05
    if x1 > x2:
        x1 -= shift
        x2 += shift
    else:
        x1 += shift
        x2 -= shift
    ax.annotate(
        '',
        xy=(x1, row['contrast_val']),
        xytext=(x2, row['contrast_val']),
        arrowprops=dict(
            arrowstyle='->',
            linewidth=1,
            shrinkA=0,
            shrinkB=0,
        )
    )

ax.set_xlabel(introduce_greek_letters('log10(IFN-gamma (ng/ml))'))
ax.set_ylabel(
    rf'$\beta_{{\mathrm{{{format_treatment(main).replace("-", "")}+AB}}}}'
    rf'-\beta_{{\mathrm{{{format_treatment(main).replace("-", "")}}}}}'
    rf'-\beta_{{\mathrm{{AB}}}}$'
    f'\n(CD14 Mono)',
    fontsize=6
)
ax.legend(loc='upper left', fontsize=6)
ax.set_title(introduce_greek_letters('impact on IFN-gamma comparison\nunder IL-15 stimulation'), fontsize=6)
ax.set_ylim(-0.35, 0.1)
ax.set_xlim(-1.5, 1)
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, 'explanation_of_TNFA_IFNG_interaction.svg'))
plt.show()

In [ ]:
# SPECIAL CASE FOR INTERFERON VS NON-INTERFERON
cell_type = 'CD4 T cell'
comp_cytokine = 'IL-10'
main_cytokine = 'IL-32-beta'
antibody = 'AB-IL-10'

x = edger_df_log2FC_main[cell_type][comp_cytokine].dropna()
rs = []

y_all = raw_log2fc_values_by_donor[cell_type][main_cytokine].dropna()
for donor in y_all.columns:
    x_sub, y_sub = get_common_index(x, y_all[donor])
    r = np.corrcoef(x_sub, y_sub)[0, 1]
    rs.append({
        "donor": donor,
        "r": r,
        "stim": "IL-32-beta"
    })

y_all = raw_log2fc_values_by_donor[cell_type][f"{main_cytokine}_{antibody}"].dropna()
for donor in y_all.columns:
    x_sub, y_sub = get_common_index(x, y_all[donor])
    r = np.corrcoef(x_sub, y_sub)[0, 1]
    rs.append({
        "donor": donor,
        "r": r,
        "stim": "IL-32-beta_AB-IL-10"
    })
rs = pd.DataFrame(rs)
rs['stim'] = rs['stim'].apply(lambda x: format_treatment(x, connection='\n'))
rs['donor_type'] = rs['donor'].apply(lambda x: 'interferon' if x in DONOR_GROUPS['interferon'] else 'non-interferon')

plt.figure(figsize=cm_to_inches((4, 2.8)))
ax = sns.barplot(data=rs, x="stim", y="r", hue="donor_type", palette=DONOR_COLOR_SCHEME, errorbar="sd", capsize=0.15, edgecolor="black")

sns.stripplot(data=rs, x="stim", y="r", hue="donor_type", palette=DONOR_COLOR_SCHEME, dodge=True, jitter=0.08, size=4, edgecolor="black", linewidth=0.3, ax=ax)

# Remove duplicated legend entries from barplot + stripplot
handles, labels = ax.get_legend_handles_labels()
n_groups = rs["donor_type"].nunique()
ax.legend_.remove()
ax.tick_params(axis='x', length=0)
ax.set_xlabel("")
ax.set_ylabel("Pearson r\ndonor log2FC\nconsensus log2FC (IL-10)")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, "IL32_with_IL10AB.svg"))
plt.show()